In [ ]:
# ============================================================================
# CELL 1: State Management & Utilities
# ============================================================================
# CHANGES from v17.2:
# 1. NEW: parse_game_state_data() now extracts "tf" (text_flag) from JSON
# 2. UPDATED: read_game_state() returns 11 values (added text_flag as 11th)
# 3. All other functions UNCHANGED
# ============================================================================

from pathlib import Path
import json
import numpy as np
import time
from collections import deque

BASE_PATH = Path("C:/Users/HP/Documents/cogai/")
ACTION_FILE = BASE_PATH / "action.json"
STATE_FILE = BASE_PATH / "game_state.json"
TAUGHT_TRANSITIONS_FILE = BASE_PATH / "taught_transitions.json"
TAUGHT_BATTLE_TRANSITIONS_FILE = BASE_PATH / "taught_battle_transitions.json"
TAUGHT_BAG_TRANSITIONS_FILE = BASE_PATH / "taught_bag_transitions.json"
EXPLORATION_MEMORY_FILE = BASE_PATH / "exploration_memory.json"
MODEL_CHECKPOINT_FILE = BASE_PATH / "model_checkpoint.json"
TAUGHT_EXPLORATION_FILE = BASE_PATH / "taught_exploration_memory.json"
TAUGHT_NAV_TARGETS_FILE = BASE_PATH / "taught_nav_targets.json"
ROSTER_FILE = BASE_PATH / "roster.json"
MOVE_KNOWLEDGE_FILE = BASE_PATH / "move_knowledge.json"
ITEM_KNOWLEDGE_FILE = BASE_PATH / "item_knowledge.json"
EVENT_TIMELINE_FILE = BASE_PATH / "event_timeline.json"

# === NEW FILE PATHS (v17.2) ===
TYPE_CLUSTERS_FILE = BASE_PATH / "type_clusters.json"
AI_EVENT_TIMELINE_FILE = BASE_PATH / "ai_event_timeline.json"
TAUGHT_START_MENU_TRANSITIONS_FILE = BASE_PATH / "taught_start_menu_transitions.json"
TYPE_DATA_FILE = BASE_PATH / "type_data.json"

# === MARKOV SIMILARITY WEIGHTS ===
MARKOV_IMMEDIATE_WEIGHT = 0.5
MARKOV_SEQUENTIAL_WEIGHT = 0.3
MARKOV_PARTIAL_WEIGHT = 0.2
MARKOV_FAMILIARITY_THRESHOLD = 0.6

MARKOV_SEQ_FULL_WEIGHT = 1.0
MARKOV_SEQ_MEDIUM_WEIGHT = 0.6
MARKOV_SEQ_SHORT_WEIGHT = 0.3

MARKOV_POS_EXACT_BONUS = 0.35
MARKOV_POS_NEAR_BONUS = 0.25
MARKOV_POS_FAR_BONUS = 0.1
MARKOV_POS_MAX_DIST = 5

# === BATTLE MARKOV WEIGHTS ===
BATTLE_MARKOV_ACTION_SEQ_WEIGHT = 0.70
BATTLE_MARKOV_PALETTE_WEIGHT = 0.20
BATTLE_MARKOV_MENU_STATE_WEIGHT = 0.10
BATTLE_MARKOV_THRESHOLD_LOW = 0.35
BATTLE_MARKOV_THRESHOLD_HIGH = 0.45

# === BAG MARKOV WEIGHTS ===
BAG_MARKOV_MENU_STATE_WEIGHT = 0.40
BAG_MARKOV_ACTION_SEQ_WEIGHT = 0.35
BAG_MARKOV_PARTY_CONTEXT_WEIGHT = 0.25
BAG_MARKOV_THRESHOLD = 0.35

# === START MENU MARKOV WEIGHTS ===
START_MENU_MARKOV_MENU_STATE_WEIGHT = 0.45
START_MENU_MARKOV_ACTION_SEQ_WEIGHT = 0.35
START_MENU_MARKOV_CONTEXT_WEIGHT = 0.20
START_MENU_MARKOV_THRESHOLD = 0.35

# === STATE DIMENSIONS ===
EXPECTED_STATE_DIM = 6
PALETTE_DIM = 768
TILE_DIM = 600

# Per-chain learning state dimensions
OVERWORLD_STATE_DIM = 8 + TILE_DIM + PALETTE_DIM  # 1376 (derived + tiles + palette)
OVERWORLD_BATTLE_STATE_DIM = 8 + PALETTE_DIM       # 776  (derived + palette, no tiles)
BATTLE_CHAIN_DIM = 41    # compact battle features
PARTY_CHAIN_DIM = 50     # party slot stats + context
BAG_CHAIN_DIM = 18       # bag state + party HP context

# === DEFAULT BATTLE DATA ===
DEFAULT_BATTLE_DATA = {
    'battle_cursor': -1, 'move_cursor': -1, 'party_cursor': -1,
    'player_species': -1, 'enemy_species': -1,
    'player_hp': -1, 'player_max_hp': -1, 'enemy_hp': -1, 'enemy_max_hp': -1,
    'player_level': -1, 'enemy_level': -1,
    'player_status': 0, 'enemy_status': 0, 'battle_type': 0,
    'move0': -1, 'move1': -1, 'move2': -1, 'move3': -1,
    'pp0': -1, 'pp1': -1, 'pp2': -1, 'pp3': -1,
    'player_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
    'enemy_move0': -1, 'enemy_move1': -1, 'enemy_move2': -1, 'enemy_move3': -1,
    'enemy_pp0': -1, 'enemy_pp1': -1, 'enemy_pp2': -1, 'enemy_pp3': -1,
    'enemy_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
}

# === DEFAULT PARTY DATA ===
DEFAULT_PARTY_SLOT = {
    'level': 0, 'hp': 0, 'max_hp': 0,
    'atk': 0, 'def': 0, 'spd': 0, 'spatk': 0, 'spdef': 0,
    'status': 0
}
DEFAULT_PARTY_DATA = {'count': 0, 'slots': []}

# === DEFAULT MENU DATA ===
DEFAULT_MENU_DATA = {'mc': -1, 'mm': -1, 'pc': -1, 'sc': -1}

# === DEFAULT BAG DATA ===
DEFAULT_BAG_DATA = {'pocket': -1, 'cursor': -1, 'active': 0, 'items': []}

# === DEFAULT ITEM KNOWLEDGE ENTRY ===
DEFAULT_ITEM_KNOWLEDGE_ENTRY = {
    'uses': 0, 'category': 'unknown', 'confidence': 0.0,
    'avg_hp_restored': 0.0, 'total_hp_restored': 0,
    'status_cured': [], 'catch_attempts': 0,
    'catch_successes': 0, 'last_used_timestep': 0,
}


# ============================================================================
# NORMALIZATION & DERIVED FEATURES
# ============================================================================

def normalize_game_state(raw_state):
    if len(raw_state) < 6:
        return raw_state
    normalized = raw_state.copy()
    normalized[0] = raw_state[0] / 255.0
    normalized[1] = raw_state[1] / 255.0
    normalized[2] = np.clip(raw_state[2], 0, 255)
    normalized[3] = 1.0 if raw_state[3] > 0 else 0.0
    normalized[4] = 1.0 if raw_state[4] > 0 else 0.0
    normalized[5] = int(raw_state[5]) % 4
    return normalized


def compute_derived_features(current, prev):
    if prev is None:
        return np.zeros(8)
    vel_x = current[0] - prev[0]
    vel_y = current[1] - prev[1]
    map_changed = 1.0 if abs(current[2] - prev[2]) > 0.5 else 0.0
    battle_started = 1.0 if current[3] > prev[3] else 0.0
    battle_ended = 1.0 if current[3] < prev[3] else 0.0
    menu_opened = 1.0 if current[4] > prev[4] else 0.0
    menu_closed = 1.0 if current[4] < prev[4] else 0.0
    direction_changed = 1.0 if current[5] != prev[5] else 0.0
    return np.array([vel_x, vel_y, map_changed, battle_started, battle_ended,
                     menu_opened, menu_closed, direction_changed])


# ============================================================================
# CHAIN-SPECIFIC LEARNING STATE BUILDERS
# ============================================================================

def build_learning_state_overworld(derived, palette, tiles, in_battle):
    """
    Overworld chain learning state.
    This is the original build_learning_state — visual + spatial features.

    In battle: derived(8) + palette(768) = 776 dims
    In overworld: derived(8) + tiles(600) + palette(768) = 1376 dims
    """
    if in_battle > 0.5:
        state = np.concatenate([derived, palette])
    else:
        state = np.concatenate([derived, tiles, palette])
    noise = np.random.randn(len(state)) * 0.0001
    return state + noise


# Backward compatibility alias
build_learning_state = build_learning_state_overworld


def build_learning_state_battle(battle_data, party_data=None, turn_count=0):
    """
    Battle chain learning state — compact features from battle memory.

    All values normalized to roughly 0-1 range for perceptron compatibility.

    Layout (41 dims):
    [0]     player_species / 500       (normalized species ID)
    [1]     player_hp_ratio            (current_hp / max_hp)
    [2]     player_level / 100         (normalized level)
    [3]     player_status_flag         (0=none, 1=has status)
    [4:8]   player_move_ids / 500      (4 normalized move IDs)
    [8:12]  player_pp_ratios           (4 PP/max_pp estimates)
    [12:19] player_stat_stages / 12    (7 stats, neutral=6 → 0.5)
    [19]    enemy_species / 500
    [20]    enemy_hp_ratio
    [21]    enemy_level / 100
    [22]    enemy_status_flag
    [23:27] enemy_move_ids / 500
    [27:31] enemy_pp_ratios
    [31:38] enemy_stat_stages / 12
    [38]    is_trainer                 (0 or 1)
    [39]    turn_count / 20            (normalized, capped)
    [40]    damage_trend               (placeholder for future)
    """
    bd = battle_data
    state = np.zeros(BATTLE_CHAIN_DIM)

    # Player
    state[0] = max(0, bd.get('player_species', -1)) / 500.0
    php, pmhp = bd.get('player_hp', -1), bd.get('player_max_hp', -1)
    state[1] = php / pmhp if pmhp > 0 and php >= 0 else 0.0
    state[2] = max(0, bd.get('player_level', -1)) / 100.0
    state[3] = 1.0 if bd.get('player_status', 0) != 0 else 0.0

    # Player moves (normalized IDs)
    for i, mk in enumerate(['move0', 'move1', 'move2', 'move3']):
        state[4 + i] = max(0, bd.get(mk, -1)) / 500.0

    # Player PP ratios (estimate max PP as 40 if unknown)
    for i, pk in enumerate(['pp0', 'pp1', 'pp2', 'pp3']):
        pp = bd.get(pk, -1)
        state[8 + i] = pp / 40.0 if pp >= 0 else 0.0

    # Player stat stages (neutral=6, normalize by /12 so neutral=0.5)
    pss = bd.get('player_stat_stages', [-1]*7)
    for i in range(7):
        state[12 + i] = pss[i] / 12.0 if pss[i] >= 0 else 0.5

    # Enemy
    state[19] = max(0, bd.get('enemy_species', -1)) / 500.0
    ehp, emhp = bd.get('enemy_hp', -1), bd.get('enemy_max_hp', -1)
    state[20] = ehp / emhp if emhp > 0 and ehp >= 0 else 0.0
    state[21] = max(0, bd.get('enemy_level', -1)) / 100.0
    state[22] = 1.0 if bd.get('enemy_status', 0) != 0 else 0.0

    # Enemy moves
    for i, mk in enumerate(['enemy_move0', 'enemy_move1', 'enemy_move2', 'enemy_move3']):
        state[23 + i] = max(0, bd.get(mk, -1)) / 500.0

    # Enemy PP
    for i, pk in enumerate(['enemy_pp0', 'enemy_pp1', 'enemy_pp2', 'enemy_pp3']):
        pp = bd.get(pk, -1)
        state[27 + i] = pp / 40.0 if pp >= 0 else 0.0

    # Enemy stat stages
    ess = bd.get('enemy_stat_stages', [-1]*7)
    for i in range(7):
        state[31 + i] = ess[i] / 12.0 if ess[i] >= 0 else 0.5

    # Context
    state[38] = 1.0 if (bd.get('battle_type', 0) & 8) != 0 else 0.0
    state[39] = min(turn_count, 20) / 20.0
    state[40] = 0.0  # damage_trend placeholder

    # Small noise for perceptron diversity
    state += np.random.randn(BATTLE_CHAIN_DIM) * 0.0001

    return state


def build_learning_state_party(party_data, active_slot=-1):
    """
    Party chain learning state — per-slot stats + context.

    Layout (50 dims):
    Per slot (6 slots × 8 features = 48):
      [i*8 + 0] hp_ratio
      [i*8 + 1] level / 100
      [i*8 + 2] status_flag (0 or 1)
      [i*8 + 3] atk / 500
      [i*8 + 4] def / 500
      [i*8 + 5] spd / 500
      [i*8 + 6] spatk / 500
      [i*8 + 7] spdef / 500
    Context (2):
      [48] party_count / 6
      [49] active_slot / 6 (or 0.0 if unknown)
    """
    state = np.zeros(PARTY_CHAIN_DIM)
    slots = party_data.get('slots', [])

    for i in range(min(6, len(slots))):
        slot = slots[i]
        base = i * 8
        mhp = slot.get('max_hp', 0)
        state[base + 0] = slot.get('hp', 0) / mhp if mhp > 0 else 0.0
        state[base + 1] = slot.get('level', 0) / 100.0
        state[base + 2] = 1.0 if slot.get('status', 0) != 0 else 0.0
        state[base + 3] = slot.get('atk', 0) / 500.0
        state[base + 4] = slot.get('def', 0) / 500.0
        state[base + 5] = slot.get('spd', 0) / 500.0
        state[base + 6] = slot.get('spatk', 0) / 500.0
        state[base + 7] = slot.get('spdef', 0) / 500.0

    state[48] = party_data.get('count', 0) / 6.0
    state[49] = active_slot / 6.0 if active_slot >= 0 else 0.0

    state += np.random.randn(PARTY_CHAIN_DIM) * 0.0001
    return state


def build_learning_state_bag(bag_data, party_data, menu_data, in_battle=False, item_knowledge=None):
    """
    Bag chain learning state — bag navigation context + party needs.

    Layout (18 dims):
    [0]    pocket / 4               (normalized pocket index)
    [1]    cursor / 20              (normalized cursor position)
    [2]    item_at_cursor_id / 500  (normalized item ID, 0 if empty)
    [3]    n_items / 20             (items in current pocket)
    [4:10] party_hp_ratios          (6 slots, 0 if empty)
    [10:16] party_status_flags      (6 slots, 0 or 1)
    [16]   in_battle                (0 or 1)
    [17]   mc / 6                   (submenu cursor, normalized)
    """
    state = np.zeros(BAG_CHAIN_DIM)

    # Bag state
    pocket = bag_data.get('pocket', -1)
    cursor = bag_data.get('cursor', -1)
    items = bag_data.get('items', [])

    state[0] = max(0, pocket) / 4.0
    state[1] = max(0, cursor) / 20.0

    # Item at cursor
    if 0 <= cursor < len(items):
        state[2] = items[cursor].get('id', 0) / 500.0
    else:
        state[2] = 0.0

    state[3] = len(items) / 20.0

    # Party HP ratios
    slots = party_data.get('slots', [])
    for i in range(min(6, len(slots))):
        mhp = slots[i].get('max_hp', 0)
        state[4 + i] = slots[i].get('hp', 0) / mhp if mhp > 0 else 0.0

    # Party status flags
    for i in range(min(6, len(slots))):
        state[10 + i] = 1.0 if slots[i].get('status', 0) != 0 else 0.0

    # Context
    state[16] = 1.0 if in_battle else 0.0
    state[17] = max(0, menu_data.get('mc', -1)) / 6.0

    state += np.random.randn(BAG_CHAIN_DIM) * 0.0001
    return state


# ============================================================================
# ARRAY HELPERS
# ============================================================================

def _pad_or_trim(arr, target_dim):
    if arr.shape[0] < target_dim:
        return np.pad(arr, (0, target_dim - arr.shape[0]))
    elif arr.shape[0] > target_dim:
        return arr[:target_dim]
    return arr


# ============================================================================
# GROUND-TRUTH TYPE DATA LOADER (Track B — Optional)
# ============================================================================

def load_type_data(filepath=None):
    """
    Load ground-truth type data from Lua verification script output.

    Expected format of type_data.json:
    {
        "species_types": {
            "1": [12, 3],       // species_id → [type1, type2] (Bulbasaur = Grass/Poison)
            "4": [10, -1],      // -1 means single-type (Charmander = Fire)
            ...
        },
        "move_types": {
            "1": 0,             // move_id → type_id (Pound = Normal)
            "10": 0,            // Scratch = Normal
            "52": 10,           // Ember = Fire
            ...
        },
        "type_chart": {
            "10_12": 2.0,       // Fire vs Grass = 2x
            "10_11": 0.5,       // Fire vs Water = 0.5x
            "10_10": 0.5,       // Fire vs Fire = 0.5x
            ...
        },
        "type_names": {
            "0": "Normal",
            "1": "Fighting",
            "2": "Flying",
            "3": "Poison",
            "4": "Ground",
            "5": "Rock",
            "6": "Bug",
            "7": "Ghost",
            "8": "Steel",
            "9": "Mystery",
            "10": "Fire",
            "11": "Water",
            "12": "Grass",
            "13": "Electric",
            "14": "Psychic",
            "15": "Ice",
            "16": "Dragon",
            "17": "Dark"
        }
    }

    Returns: dict with parsed data, or None if file not found/invalid.
    """
    filepath = filepath or TYPE_DATA_FILE
    try:
        if not Path(filepath).exists():
            return None

        with open(filepath, 'r') as f:
            raw = json.load(f)

        data = {
            'species_types': {},
            'move_types': {},
            'type_chart': {},
            'type_names': {},
            'loaded': True,
        }

        # Species → [type1, type2]
        for sid, types in raw.get('species_types', {}).items():
            species_id = int(sid)
            if isinstance(types, list) and len(types) >= 1:
                t1 = types[0]
                t2 = types[1] if len(types) > 1 else -1
                data['species_types'][species_id] = [t1, t2]

        # Move → type
        for mid, mtype in raw.get('move_types', {}).items():
            data['move_types'][int(mid)] = int(mtype)

        # Type matchup chart: "atk_def" → multiplier
        for key, mult in raw.get('type_chart', {}).items():
            data['type_chart'][key] = float(mult)

        # Type names
        for tid, tname in raw.get('type_names', {}).items():
            data['type_names'][int(tid)] = tname

        print(f"  🧬 Type data loaded (Track B):")
        print(f"     Species: {len(data['species_types'])}")
        print(f"     Moves: {len(data['move_types'])}")
        print(f"     Chart entries: {len(data['type_chart'])}")
        print(f"     Types: {len(data['type_names'])}")

        return data

    except Exception as e:
        print(f"  ⚠️ Error loading type data: {e}")
        return None


def get_type_effectiveness(type_data, move_id, species_id):
    """
    Look up type effectiveness for a move against a species using
    ground-truth type data (Track B).

    Returns: float multiplier (0.0, 0.25, 0.5, 1.0, 2.0, 4.0) or None if unknown.
    """
    if type_data is None or not type_data.get('loaded'):
        return None

    move_type = type_data['move_types'].get(move_id)
    if move_type is None:
        return None

    species_types = type_data['species_types'].get(species_id)
    if species_types is None:
        return None

    multiplier = 1.0
    for def_type in species_types:
        if def_type < 0:
            continue
        key = f"{move_type}_{def_type}"
        chart_mult = type_data['type_chart'].get(key)
        if chart_mult is not None:
            multiplier *= chart_mult

    return multiplier


# ============================================================================
# PARSERS
# ============================================================================

def parse_battle_data(data):
    b = data.get('b')
    if b is None:
        return DEFAULT_BATTLE_DATA.copy()

    pss = b.get('pss')
    if not isinstance(pss, list) or len(pss) != 7:
        pss = [-1, -1, -1, -1, -1, -1, -1]
    else:
        pss = list(pss)

    ess = b.get('ess')
    if not isinstance(ess, list) or len(ess) != 7:
        ess = [-1, -1, -1, -1, -1, -1, -1]
    else:
        ess = list(ess)

    return {
        'battle_cursor': b.get('bc', -1), 'move_cursor': b.get('mc', -1),
        'party_cursor': b.get('pc', -1),
        'player_species': b.get('ps', -1), 'enemy_species': b.get('es', -1),
        'player_hp': b.get('ph', -1), 'player_max_hp': b.get('pm', -1),
        'enemy_hp': b.get('eh', -1), 'enemy_max_hp': b.get('em', -1),
        'player_level': b.get('pl', -1), 'enemy_level': b.get('el', -1),
        'player_status': b.get('pst', 0), 'enemy_status': b.get('est', 0),
        'battle_type': b.get('bt', 0),
        'move0': b.get('m0', -1), 'move1': b.get('m1', -1),
        'move2': b.get('m2', -1), 'move3': b.get('m3', -1),
        'pp0': b.get('pp0', -1), 'pp1': b.get('pp1', -1),
        'pp2': b.get('pp2', -1), 'pp3': b.get('pp3', -1),
        'player_stat_stages': pss,
        'enemy_move0': b.get('em0', -1), 'enemy_move1': b.get('em1', -1),
        'enemy_move2': b.get('em2', -1), 'enemy_move3': b.get('em3', -1),
        'enemy_pp0': b.get('epp0', -1), 'enemy_pp1': b.get('epp1', -1),
        'enemy_pp2': b.get('epp2', -1), 'enemy_pp3': b.get('epp3', -1),
        'enemy_stat_stages': ess,
    }


def parse_party_data(data):
    pa = data.get('pa')
    if pa is None:
        return DEFAULT_PARTY_DATA.copy()
    count = pa.get('c', 0)
    raw_slots = pa.get('s', [])
    slots = []
    for s in raw_slots:
        if not isinstance(s, dict): continue
        slots.append({
            'level': s.get('l', 0), 'hp': s.get('h', 0), 'max_hp': s.get('m', 0),
            'atk': s.get('a', 0), 'def': s.get('d', 0), 'spd': s.get('sp', 0),
            'spatk': s.get('sa', 0), 'spdef': s.get('sd', 0), 'status': s.get('st', 0),
        })
    return {'count': count, 'slots': slots}


def parse_menu_data(data):
    mu = data.get('mu')
    if mu is None:
        return DEFAULT_MENU_DATA.copy()
    return {'mc': mu.get('mc', -1), 'mm': mu.get('mm', -1),
            'pc': mu.get('pc', -1), 'sc': mu.get('sc', -1)}


def parse_bag_data(data):
    bg = data.get('bg')
    if bg is None:
        return DEFAULT_BAG_DATA.copy()
    items = []
    for item in bg.get('it', []):
        if isinstance(item, dict) and 'id' in item:
            items.append({'id': item.get('id', 0), 'q': item.get('q', 0)})
    return {'pocket': bg.get('pk', -1), 'cursor': bg.get('bc', -1),
            'active': bg.get('a', 0), 'items': items}


def parse_game_state_data(data):
    """
    Parse all fields from game_state.json.

    v17.3: Now extracts "tf" (text_flag) — 0=no dialogue, 1=text box active.
    Falls back to 0 if field is missing (backward compat with old Lua v3).
    """
    raw = data.get("state") or data.get("s") or []
    palette_raw = data.get("palette") or data.get("p") or []
    tiles_raw = data.get("tiles") or data.get("t") or []
    dead = bool(data.get("dead", False))
    battle_data = parse_battle_data(data)
    party_data = parse_party_data(data)
    game_state_raw = data.get("gs", 0)
    menu_data = parse_menu_data(data)
    bag_data = parse_bag_data(data)
    text_flag = int(data.get("tf", 0))  # NEW: text dialogue flag
    return raw, palette_raw, tiles_raw, dead, battle_data, party_data, game_state_raw, menu_data, bag_data, text_flag


def read_game_state(max_retries=3):
    """
    Read and parse game_state.json.

    v17.3: Returns 11 values (added text_flag as 11th).

    Returns:
        (context_state, palette_state, tile_state, dead, raw_position,
         battle_data, party_data, game_state_raw, menu_data, bag_data,
         text_flag)
    """
    if not STATE_FILE.exists():
        return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy(), 0)

    for attempt in range(max_retries):
        try:
            with open(STATE_FILE, "r") as f:
                data = json.loads(f.read())

            (raw, palette_raw, tiles_raw, dead, battle_data, party_data,
             game_state_raw, menu_data, bag_data, text_flag) = parse_game_state_data(data)

            raw_x = int(raw[0]) if len(raw) > 0 else 0
            raw_y = int(raw[1]) if len(raw) > 1 else 0
            raw_position = (raw_x, raw_y)

            context_state = normalize_game_state(np.array(raw, dtype=float))
            palette_state = np.array(palette_raw, dtype=float) if palette_raw else np.zeros(PALETTE_DIM)
            tile_state = np.array(tiles_raw, dtype=float) if tiles_raw else np.zeros(TILE_DIM)

            context_state = _pad_or_trim(context_state, EXPECTED_STATE_DIM)
            palette_state = _pad_or_trim(palette_state, PALETTE_DIM)
            tile_state = _pad_or_trim(tile_state, TILE_DIM)

            return (context_state, palette_state, tile_state, dead, raw_position,
                    battle_data, party_data, game_state_raw, menu_data, bag_data,
                    text_flag)

        except (json.JSONDecodeError, ValueError):
            if attempt < max_retries - 1:
                time.sleep(0.001)
                continue
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy(), 0)
        except Exception:
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy(), 0)


def write_action(action_name):
    if action_name:
        action_name = action_name.upper()
    try:
        with open(ACTION_FILE, "w") as f:
            json.dump({"action": action_name}, f)
            f.flush()
    except Exception as e:
        print(f"[ERROR] Failed to write action: {e}")

In [ ]:
# ============================================================================
# CELL 2: Perceptron Classes + Pool/Pipeline Infrastructure
# ============================================================================
# CHANGES from v17.3:
# 1. NEW: Pool class — fixed-width output layer containing perceptrons
#    - Computes output vector from internal perceptrons (weighted avg / top-k)
#    - Spawn threshold (novelty-based)
#    - Memory paging (serialize pruned perceptrons to residual file)
#    - Authority tracking (how much the pool should be trusted)
# 2. NEW: Pipeline class — ordered sequence of Pools forming a processing chain
#    - Each pool takes input from previous pool's output (or raw state for L1)
#    - Backward credit assignment with decay factor per layer
#    - Fallback to neutral vectors when pools are empty
# 3. UPDATED: Perceptron class
#    - Added pool_id, layer_index for pool membership tracking
#    - Added trigger_context for residual file keying
#    - ensure_weights() unchanged — gets called with pool output width
#    - predict(), update() unchanged internally
#    - Activation discovery system UNCHANGED
# 4. ActivationLibrary UNCHANGED
# 5. ControlSwapPerceptron UNCHANGED
# ============================================================================


class ActivationLibrary:
    """
    Shared library of candidate activation functions.

    Each candidate is a dict:
    {
        'name': str,
        'func': callable(x) -> float,
        'category': str,       — 'standard', 'discovered', 'custom'
        'suited_for': list,    — ['entity', 'action_binary', 'action_continuous']
        'param': float or None — tunable parameter (e.g. threshold for soft_threshold)
    }

    The library starts with standard functions and can grow as perceptrons
    discover that certain shapes work well for certain contexts.
    """

    def __init__(self):
        self.candidates = []
        self._build_standard_library()

    def _build_standard_library(self):
        """Initialize with standard activation functions."""

        self.candidates = [
            {
                'name': 'linear',
                'func': lambda x: x,
                'category': 'standard',
                'suited_for': ['action_continuous', 'entity'],
                'param': None,
            },
            {
                'name': 'tanh',
                'func': lambda x: np.tanh(x),
                'category': 'standard',
                'suited_for': ['entity', 'action_continuous'],
                'param': None,
            },
            {
                'name': 'relu',
                'func': lambda x: max(0.0, x),
                'category': 'standard',
                'suited_for': ['action_continuous'],
                'param': None,
            },
            {
                'name': 'sigmoid',
                'func': lambda x: 1.0 / (1.0 + np.exp(-np.clip(x, -20.0, 20.0))),
                'category': 'standard',
                'suited_for': ['action_binary', 'entity'],
                'param': None,
            },
            {
                'name': 'bounded_linear',
                'func': lambda x: np.clip(x, -1.0, 1.0),
                'category': 'standard',
                'suited_for': ['action_continuous', 'entity'],
                'param': None,
            },
            {
                'name': 'soft_threshold_0.1',
                'func': lambda x: x if abs(x) > 0.1 else 0.0,
                'category': 'standard',
                'suited_for': ['entity', 'action_continuous'],
                'param': 0.1,
            },
            {
                'name': 'soft_threshold_0.3',
                'func': lambda x: x if abs(x) > 0.3 else 0.0,
                'category': 'standard',
                'suited_for': ['entity'],
                'param': 0.3,
            },
            {
                'name': 'leaky_relu',
                'func': lambda x: x if x > 0 else 0.01 * x,
                'category': 'standard',
                'suited_for': ['action_continuous', 'entity'],
                'param': None,
            },
            {
                'name': 'abs',
                'func': lambda x: abs(x),
                'category': 'standard',
                'suited_for': ['entity'],
                'param': None,
            },
            {
                'name': 'squared',
                'func': lambda x: x * abs(x),
                'category': 'standard',
                'suited_for': ['entity', 'action_continuous'],
                'param': None,
            },
        ]

    def get_candidates(self, suited_for=None):
        """
        Get candidate functions, optionally filtered by suitability.

        suited_for: None (all) or a string like 'entity', 'action_binary'
        """
        if suited_for is None:
            return self.candidates

        return [c for c in self.candidates
                if suited_for in c.get('suited_for', [])]

    def get_by_name(self, name):
        """Get a specific candidate by name."""
        for c in self.candidates:
            if c['name'] == name:
                return c
        return None

    def add_discovered(self, name, func, suited_for, param=None):
        """
        Add a newly discovered activation function to the library.

        Called when a perceptron discovers that a custom shape works well
        and that shape isn't already in the library.
        """
        # Don't add duplicates
        if self.get_by_name(name) is not None:
            return

        self.candidates.append({
            'name': name,
            'func': func,
            'category': 'discovered',
            'suited_for': suited_for,
            'param': param,
        })

    def get_names(self):
        return [c['name'] for c in self.candidates]


# Global shared library — all perceptrons reference the same instance
ACTIVATION_LIBRARY = ActivationLibrary()


class Perceptron:
    def __init__(self, kind, action=None, group=None, entity_type=None, chain="shared"):
        self.kind = kind
        self.action = action
        self.group = group
        self.entity_type = entity_type
        self.chain = chain         # "overworld", "battle", "party", "bag", "shared"

        self.utility = 1.0
        self.weights = None

        self.eligibility_fast = 0.0
        self.eligibility_slow = 0.0

        self.familiarity = 0.0
        self.activation_history = deque(maxlen=10)
        self.cluster_activations = deque(maxlen=50)

        self.learning_rate = 0.01
        self.prediction_errors = deque(maxlen=50)

        # === POOL MEMBERSHIP (NEW) ===
        # Which pool this perceptron belongs to (None = legacy flat system)
        self.pool_id = None
        # Which layer index within the pipeline (0-based, None = legacy)
        self.layer_index = None
        # Context that triggered this perceptron's creation
        # Used as key for residual file (e.g. species ID, map region, type pair)
        self.trigger_context = None

        # === EMPIRICAL ACTIVATION DISCOVERY ===
        # Observation buffer: (raw_dot_product, observed_error) pairs
        # Used to evaluate which activation function best fits this
        # perceptron's input-output relationship.
        self.activation_observations = deque(maxlen=200)

        # Current active activation function name
        self.active_activation = 'linear'  # safest default

        # Fitting schedule
        self.ACTIVATION_FIT_INTERVAL = 100   # re-evaluate every N updates
        self.ACTIVATION_MIN_OBSERVATIONS = 30  # minimum data before fitting
        self._activation_update_counter = 0

        # Fit quality tracking
        self.activation_fit_score = 0.0    # how well current activation fits
        self.activation_fit_history = deque(maxlen=10)  # track fit scores over time
        self.activation_change_count = 0   # how many times activation changed

    def _get_activation_func(self):
        """Get the callable for current active activation."""
        candidate = ACTIVATION_LIBRARY.get_by_name(self.active_activation)
        if candidate is not None:
            return candidate['func']
        # Fallback to linear if name not found
        return lambda x: x

    def _apply_activation(self, x):
        """Apply the current discovered activation function."""
        func = self._get_activation_func()
        try:
            return func(x)
        except (ValueError, OverflowError):
            return x  # fallback to linear on numerical issues

    def _get_suited_for_hint(self):
        """
        Determine what kind of activation this perceptron is suited for.
        Used to prioritize candidates during fitting.
        """
        if self.kind == "entity":
            return "entity"

        if self.kind == "action":
            # Binary actions (like A/B/Start) tend toward sigmoid-like
            if self.group == "interact":
                return "action_binary"
            # Movement actions tend toward continuous scoring
            if self.group == "move":
                return "action_continuous"

        return "action_continuous"

    def _evaluate_activation_fit(self, candidate_func):
        """
        Evaluate how well a candidate activation function fits the
        observed (input, error) data.

        Method: for each observation (raw_input, observed_error),
        compute candidate_output = candidate_func(raw_input).
        A good activation should produce outputs that correlate with
        the observed error — when error was high, the activation should
        have responded strongly, and when error was low, weakly.

        Score = correlation between |candidate_output| and |observed_error|

        Higher score = better fit.
        """
        if len(self.activation_observations) < self.ACTIVATION_MIN_OBSERVATIONS:
            return 0.0

        observations = list(self.activation_observations)
        raw_inputs = [obs[0] for obs in observations]
        errors = [obs[1] for obs in observations]

        # Compute candidate outputs
        try:
            candidate_outputs = [candidate_func(x) for x in raw_inputs]
        except (ValueError, OverflowError):
            return -1.0  # candidate is numerically unstable

        # We want: when the perceptron SHOULD respond (high error),
        # the activation produces distinctive (non-zero) output.
        # When the perceptron SHOULDN'T respond (low error),
        # the activation is muted.

        abs_outputs = np.array([abs(o) for o in candidate_outputs])
        abs_errors = np.array([abs(e) for e in errors])

        # Avoid degenerate cases
        if np.std(abs_outputs) < 1e-10 or np.std(abs_errors) < 1e-10:
            return 0.0

        # Pearson correlation between |output| and |error|
        correlation = np.corrcoef(abs_outputs, abs_errors)[0, 1]

        if np.isnan(correlation):
            return 0.0

        # Also reward activations that produce varied outputs
        # (constant output = useless regardless of correlation)
        output_variance = np.std(abs_outputs)
        variance_bonus = min(0.2, output_variance * 0.5)

        # Penalize activations that are numerically extreme
        max_output = max(abs_outputs)
        if max_output > 100:
            return correlation * 0.5  # penalize extreme values

        return correlation + variance_bonus

    def fit_activation(self):
        """
        Evaluate all candidate activation functions and select the best one.

        Called periodically (every ACTIVATION_FIT_INTERVAL updates).
        Only changes activation if a candidate is significantly better
        than the current one (hysteresis to avoid thrashing).
        """
        if len(self.activation_observations) < self.ACTIVATION_MIN_OBSERVATIONS:
            return

        hint = self._get_suited_for_hint()

        # Get candidates — prioritize suited ones but evaluate all
        suited_candidates = ACTIVATION_LIBRARY.get_candidates(suited_for=hint)
        all_candidates = ACTIVATION_LIBRARY.candidates

        # Combine: suited first, then others (suited get a small bonus)
        scored = []
        suited_names = {c['name'] for c in suited_candidates}

        for candidate in all_candidates:
            score = self._evaluate_activation_fit(candidate['func'])
            # Small bonus for candidates marked as suited
            if candidate['name'] in suited_names:
                score += 0.05
            scored.append((candidate['name'], score))

        scored.sort(key=lambda x: x[1], reverse=True)

        if not scored:
            return

        best_name, best_score = scored[0]
        current_score = self._evaluate_activation_fit(self._get_activation_func())

        self.activation_fit_score = current_score
        self.activation_fit_history.append(current_score)

        # Hysteresis: only switch if new candidate is significantly better
        # This prevents thrashing between similar candidates
        SWITCH_THRESHOLD = 0.1  # must be 0.1 better to switch

        if best_name != self.active_activation and best_score > current_score + SWITCH_THRESHOLD:
            old_name = self.active_activation
            self.active_activation = best_name
            self.activation_change_count += 1

            # Log significant changes
            if self.activation_change_count <= 5 or self.activation_change_count % 10 == 0:
                id_str = self.action or self.entity_type or "?"
                print(f"  🧬 ACTIVATION [{self.chain}:{id_str}]: {old_name} → {best_name} "
                      f"(score {current_score:.3f} → {best_score:.3f}, "
                      f"change #{self.activation_change_count})")

    def ensure_weights(self, dim):
        if self.weights is None:
            self.weights = np.random.randn(dim) * 0.001

    def predict(self, state):
        self.ensure_weights(len(state))

        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            raw_dot = np.dot(self.weights[:min_dim], state[:min_dim])
        else:
            raw_dot = np.dot(self.weights, state)

        # Apply discovered activation function
        activated = self._apply_activation(raw_dot)

        if self.kind == "entity":
            novelty_factor = 1.0 / (1.0 + np.sqrt(self.familiarity * 0.5))
            decayed = activated * novelty_factor
            self.activation_history.append(abs(raw_dot))
            self.cluster_activations.append(abs(raw_dot))
            return decayed
        else:
            return activated

    def adapt_learning_rate(self):
        if len(self.prediction_errors) >= 50:
            avg_error = np.mean(self.prediction_errors)
            if avg_error < 0.1:
                self.learning_rate = max(0.001, self.learning_rate * 0.99)
            elif avg_error > 0.5:
                self.learning_rate = min(0.05, self.learning_rate * 1.01)

    def update(self, state, error, gamma_fast=0.5, gamma_slow=0.95, stagnation=0.0):
        self.ensure_weights(len(state))

        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            state = state[:min_dim]
            self.weights = self.weights[:min_dim]

        self.eligibility_fast = gamma_fast * self.eligibility_fast + 1.0
        self.eligibility_slow = gamma_slow * self.eligibility_slow + 1.0

        self.adapt_learning_rate()

        fast_update = 0.7 * self.learning_rate * error * state * self.eligibility_fast
        slow_update = 0.3 * self.learning_rate * error * state * self.eligibility_slow
        self.weights += fast_update + slow_update

        if self.kind == "action":
            if error > 0.01:
                if stagnation > 0.5:
                    self.utility *= 0.97
                elif error > 0.2:
                    self.utility = min(self.utility * 1.02, 2.0)
                else:
                    self.utility *= 0.995

            if self.group == "move":
                self.utility = np.clip(self.utility, 0.1, 2.0)
            else:
                self.utility = np.clip(self.utility, 0.01, 2.0)

        if self.kind == "entity" and len(self.activation_history) > 0:
            recent_avg = np.mean(self.activation_history)
            if recent_avg > 0.1:
                self.familiarity += 0.03

        if self.kind == "entity":
            prediction = self.predict(state)
            self.prediction_errors.append(abs(prediction - error))

        # === RECORD OBSERVATION FOR ACTIVATION FITTING ===
        raw_dot = np.dot(self.weights, state) if len(self.weights) == len(state) else 0.0
        self.activation_observations.append((raw_dot, error))

        # === PERIODIC ACTIVATION FITTING ===
        self._activation_update_counter += 1
        if self._activation_update_counter >= self.ACTIVATION_FIT_INTERVAL:
            self._activation_update_counter = 0
            self.fit_activation()

    def get_activation_state(self):
        """Get activation state for save/load."""
        return {
            'active_activation': self.active_activation,
            'fit_score': float(self.activation_fit_score),
            'change_count': self.activation_change_count,
            'observations_count': len(self.activation_observations),
        }

    def set_activation_state(self, state_dict):
        """Restore activation state from save/load."""
        if state_dict is None:
            return
        self.active_activation = state_dict.get('active_activation', 'linear')
        self.activation_fit_score = state_dict.get('fit_score', 0.0)
        self.activation_change_count = state_dict.get('change_count', 0)
        # Observations are not saved (too large) — will rebuild from live data

    def get_pool_state(self):
        """Get pool membership state for save/load."""
        return {
            'pool_id': self.pool_id,
            'layer_index': self.layer_index,
            'trigger_context': self.trigger_context,
        }

    def set_pool_state(self, state_dict):
        """Restore pool membership state from save/load."""
        if state_dict is None:
            return
        self.pool_id = state_dict.get('pool_id')
        self.layer_index = state_dict.get('layer_index')
        self.trigger_context = state_dict.get('trigger_context')


class ControlSwapPerceptron(Perceptron):
    def __init__(self):
        super().__init__(kind="control_swap", chain="shared")
        self.swap_history = deque(maxlen=100)
        self.confidence = 0.0

    def should_swap(self, state, movement_stagnation):
        if self.weights is None:
            return False, 0.0

        self.ensure_weights(len(state))
        swap_score = self.predict(state)  # uses discovered activation
        stagnation_factor = np.tanh(movement_stagnation / 5.0)
        combined_score = swap_score * 0.7 + stagnation_factor * 0.3

        return combined_score > 0.5, abs(combined_score)

    def record_swap_outcome(self, state, swapped, novelty_gained):
        self.swap_history.append((swapped, novelty_gained))

        if len(self.swap_history) >= 20:
            recent = list(self.swap_history)[-20:]
            successful = sum(1 for swap, nov in recent if swap and nov > 0.2)
            self.confidence = successful / 20.0


# ============================================================================
# POOL CLASS — Fixed-Width Output Layer of Perceptrons
# ============================================================================

class Pool:
    """
    A pool is a layer within a pipeline. It contains zero or more perceptrons
    that share a semantic role. The pool produces a fixed-width output vector
    regardless of how many perceptrons it contains.

    When empty, it produces a neutral (zero) vector — triggering fallback
    to Markov/hardcoded systems downstream.

    Key properties:
    - output_width: fixed dimensionality of output (default 8)
    - perceptrons: list of perceptrons in this pool
    - spawn_threshold: novelty threshold for creating new perceptrons
    - authority: how much this pool's output should be trusted (0.0-1.0)
    - residual: serialized pruned perceptrons keyed by trigger_context
    """

    # Default output width for all pools
    DEFAULT_OUTPUT_WIDTH = 8

    # Maximum perceptrons before clustering/pruning triggers
    DEFAULT_MAX_PERCEPTRONS = 20

    # Authority ramp-up: authority = min(1.0, n_perceptrons * avg_fit / threshold)
    AUTHORITY_FIT_THRESHOLD = 5.0

    # Residual file max entries per pool
    RESIDUAL_MAX_ENTRIES = 50

    def __init__(self, pool_id, name, output_width=None, max_perceptrons=None):
        self.pool_id = pool_id
        self.name = name
        self.output_width = output_width or self.DEFAULT_OUTPUT_WIDTH
        self.max_perceptrons = max_perceptrons or self.DEFAULT_MAX_PERCEPTRONS

        # Perceptrons in this pool (references — actual perceptrons live in Brain.perceptrons)
        self.perceptron_ids = []  # list of id() references for fast lookup

        # Spawn control
        self.spawn_threshold = 0.0005  # will be set adaptively
        self.spawn_count = 0
        self.last_spawn_timestep = 0

        # Authority tracking
        self.authority = 0.0  # 0.0 = no trust (fallback), 1.0 = full trust

        # Error history for this pool (for adaptive spawn threshold)
        self.error_history = deque(maxlen=200)

        # Residual file: pruned perceptrons stored by trigger_context
        # Format: {trigger_context_key: serialized_perceptron_dict}
        self.residual = {}

        # Last computed output (cached for inter-pool wiring)
        self._cached_output = np.zeros(self.output_width)
        self._cached_output_valid = False

    def compute_output(self, input_vector, brain_perceptrons):
        """
        Compute the fixed-width output vector from pool perceptrons.

        Strategy:
        - If pool is empty → return neutral (zeros) vector
        - If pool has perceptrons → each predicts on input_vector,
          then aggregate into fixed-width output via weighted average
          of top-k activations, padded/truncated to output_width.

        Args:
            input_vector: numpy array — either raw state (for L1) or
                         previous pool's output (for L2+)
            brain_perceptrons: list of all brain perceptrons (to resolve refs)

        Returns:
            numpy array of shape (output_width,)
        """
        pool_perceptrons = self._get_perceptrons(brain_perceptrons)

        if not pool_perceptrons:
            self._cached_output = np.zeros(self.output_width)
            self._cached_output_valid = True
            self.authority = 0.0
            return self._cached_output.copy()

        # Collect activations from each perceptron
        activations = []
        weights_sum = 0.0

        for p in pool_perceptrons:
            pred = p.predict(input_vector)
            weight = max(0.01, p.utility)
            activations.append((pred, weight, p))
            weights_sum += weight

        if weights_sum < 1e-10:
            self._cached_output = np.zeros(self.output_width)
            self._cached_output_valid = True
            return self._cached_output.copy()

        # Build output vector
        # Method: distribute perceptron activations across output dimensions
        # Each perceptron contributes its activation weighted by utility
        # to one or more output dimensions (round-robin assignment)
        output = np.zeros(self.output_width)

        for i, (pred, weight, p) in enumerate(activations):
            # Each perceptron maps to an output dimension (round-robin)
            dim_idx = i % self.output_width
            output[dim_idx] += pred * (weight / weights_sum)

        # If we have more output dims than perceptrons, some dims stay 0
        # If we have more perceptrons than output dims, they accumulate

        # Normalize to bounded range for stable inter-pool propagation
        output_norm = np.linalg.norm(output)
        if output_norm > 0:
            output = output / max(1.0, output_norm)

        self._cached_output = output
        self._cached_output_valid = True

        # Update authority
        avg_fit = np.mean([p.activation_fit_score for p in pool_perceptrons])
        n = len(pool_perceptrons)
        self.authority = min(1.0, n * max(0.01, avg_fit) / self.AUTHORITY_FIT_THRESHOLD)

        return output.copy()

    def get_cached_output(self):
        """Return last computed output (for inter-pool reads between compute cycles)."""
        return self._cached_output.copy()

    def invalidate_cache(self):
        """Mark cached output as stale."""
        self._cached_output_valid = False

    def _get_perceptrons(self, brain_perceptrons):
        """Resolve perceptron references from brain's master list."""
        pool_id = self.pool_id
        return [p for p in brain_perceptrons if p.pool_id == pool_id]

    def get_perceptron_count(self, brain_perceptrons):
        """Count perceptrons in this pool."""
        return len(self._get_perceptrons(brain_perceptrons))

    def needs_spawn(self, error):
        """Check if error exceeds spawn threshold."""
        return error > self.spawn_threshold

    def update_spawn_threshold(self, percentile=75):
        """Adaptively set spawn threshold from error history."""
        if len(self.error_history) >= 50:
            self.spawn_threshold = max(0.001, np.percentile(list(self.error_history), percentile))

    def record_error(self, error):
        """Record error for adaptive threshold computation."""
        self.error_history.append(error)

    def should_cluster(self, brain_perceptrons):
        """Check if pool has too many perceptrons and needs clustering."""
        return self.get_perceptron_count(brain_perceptrons) >= self.max_perceptrons

    # =================================================================
    # RESIDUAL FILE (Memory Paging)
    # =================================================================

    def page_to_residual(self, perceptron):
        """
        Serialize a pruned perceptron to the residual store.
        Keyed by trigger_context so it can be retrieved if the same
        context is encountered again.
        """
        key = perceptron.trigger_context
        if key is None:
            key = f"_unnamed_{self.pool_id}_{perceptron.entity_type or perceptron.action or 'unknown'}"

        entry = {
            'kind': perceptron.kind,
            'action': perceptron.action,
            'group': perceptron.group,
            'entity_type': perceptron.entity_type,
            'chain': perceptron.chain,
            'utility': float(perceptron.utility),
            'familiarity': float(perceptron.familiarity),
            'learning_rate': float(perceptron.learning_rate),
            'active_activation': perceptron.active_activation,
            'activation_fit_score': float(perceptron.activation_fit_score),
            'trigger_context': perceptron.trigger_context,
            'weights_shape': len(perceptron.weights) if perceptron.weights is not None else 0,
            'weights_nonzero': [[i, float(v)] for i, v in enumerate(perceptron.weights)
                                if abs(v) > 1e-10] if perceptron.weights is not None else [],
            'paged_at': 0,  # will be set by caller with brain.timestep
        }

        self.residual[str(key)] = entry

        # Evict oldest if over capacity
        if len(self.residual) > self.RESIDUAL_MAX_ENTRIES:
            oldest_key = min(self.residual.keys(),
                            key=lambda k: self.residual[k].get('paged_at', 0))
            del self.residual[oldest_key]

    def restore_from_residual(self, trigger_context):
        """
        Check if a perceptron with this trigger_context exists in residual.
        If found, reconstruct and return it (removing from residual).
        Returns: Perceptron or None
        """
        key = str(trigger_context)
        if key not in self.residual:
            return None

        entry = self.residual.pop(key)

        p = Perceptron(
            kind=entry['kind'],
            action=entry.get('action'),
            group=entry.get('group'),
            entity_type=entry.get('entity_type'),
            chain=entry.get('chain', 'shared'),
        )
        p.utility = entry.get('utility', 1.0)
        p.familiarity = entry.get('familiarity', 0.0)
        p.learning_rate = entry.get('learning_rate', 0.01)
        p.active_activation = entry.get('active_activation', 'linear')
        p.activation_fit_score = entry.get('activation_fit_score', 0.0)
        p.trigger_context = entry.get('trigger_context')
        p.pool_id = self.pool_id
        p.layer_index = None  # caller sets this

        # Restore weights
        ws = entry.get('weights_shape', 0)
        if ws > 0 and entry.get('weights_nonzero'):
            p.weights = np.zeros(ws)
            for idx, val in entry['weights_nonzero']:
                if idx < ws:
                    p.weights[idx] = val

        return p

    def has_residual(self, trigger_context):
        """Check if a trigger_context exists in residual without removing it."""
        return str(trigger_context) in self.residual

    # =================================================================
    # SERIALIZATION
    # =================================================================

    def get_save_state(self):
        """Serialize pool metadata for checkpoint."""
        return {
            'pool_id': self.pool_id,
            'name': self.name,
            'output_width': self.output_width,
            'max_perceptrons': self.max_perceptrons,
            'spawn_threshold': float(self.spawn_threshold),
            'spawn_count': self.spawn_count,
            'authority': float(self.authority),
            'residual': self.residual,
        }

    def load_save_state(self, state_dict):
        """Restore pool metadata from checkpoint."""
        if state_dict is None:
            return
        self.output_width = state_dict.get('output_width', self.DEFAULT_OUTPUT_WIDTH)
        self.max_perceptrons = state_dict.get('max_perceptrons', self.DEFAULT_MAX_PERCEPTRONS)
        self.spawn_threshold = state_dict.get('spawn_threshold', 0.0005)
        self.spawn_count = state_dict.get('spawn_count', 0)
        self.authority = state_dict.get('authority', 0.0)
        self.residual = state_dict.get('residual', {})


# ============================================================================
# PIPELINE CLASS — Ordered Sequence of Pools
# ============================================================================

class Pipeline:
    """
    A pipeline is an ordered sequence of pools (layers) that process
    game state through increasing levels of abstraction.

    Each pipeline corresponds to a game context:
    - battle_pipeline (6 layers + outcome observation)
    - overworld_pipeline (7 layers + outcome observation)
    - bag_pipeline (3 layers)
    - party_pipeline (2 layers)

    Key behaviors:
    - Layer 1 pools take raw game state as input
    - Layer N pools (N>1) take the fixed-width output of Layer N-1
    - When a pool is empty, it passes a neutral vector
    - Credit assignment flows backward with decay factor per layer
    - If ALL pools are empty, the pipeline signals fallback to Markov/hardcoded
    """

    # Default credit assignment decay per layer
    DEFAULT_CREDIT_DECAY = 0.7

    def __init__(self, pipeline_id, name, pool_definitions, credit_decay=None):
        """
        Args:
            pipeline_id: unique string identifier
            name: human-readable name
            pool_definitions: list of dicts, one per layer, each with:
                {
                    'name': str,          — layer name (e.g. "identification")
                    'output_width': int,  — fixed output width (default 8)
                    'max_perceptrons': int, — capacity (default 20)
                }
            credit_decay: float — decay factor per layer for backward credit
        """
        self.pipeline_id = pipeline_id
        self.name = name
        self.credit_decay = credit_decay or self.DEFAULT_CREDIT_DECAY

        # Build pools from definitions
        self.pools = []
        for i, pdef in enumerate(pool_definitions):
            pool_id = f"{pipeline_id}_L{i}_{pdef['name']}"
            pool = Pool(
                pool_id=pool_id,
                name=pdef['name'],
                output_width=pdef.get('output_width', Pool.DEFAULT_OUTPUT_WIDTH),
                max_perceptrons=pdef.get('max_perceptrons', Pool.DEFAULT_MAX_PERCEPTRONS),
            )
            self.pools.append(pool)

        self.num_layers = len(self.pools)

        # Track per-layer outputs for credit assignment
        self._layer_inputs = []   # input to each layer during forward pass
        self._layer_outputs = []  # output of each layer during forward pass

        # Pipeline-level authority (are ANY pools populated?)
        self.active = False

    def forward(self, raw_input, brain_perceptrons):
        """
        Run forward pass through all layers.

        Args:
            raw_input: numpy array — raw game state for this context
            brain_perceptrons: list of all brain perceptrons

        Returns:
            final_output: numpy array — output of the last pool
            active: bool — True if at least one pool had perceptrons
        """
        self._layer_inputs = []
        self._layer_outputs = []

        current_input = raw_input
        any_active = False

        for i, pool in enumerate(self.pools):
            self._layer_inputs.append(current_input.copy())

            output = pool.compute_output(current_input, brain_perceptrons)
            self._layer_outputs.append(output.copy())

            if pool.authority > 0.0:
                any_active = True

            # Next layer's input is this layer's output
            current_input = output

        self.active = any_active

        # Return final layer output
        return self._layer_outputs[-1] if self._layer_outputs else np.zeros(
            self.pools[-1].output_width if self.pools else Pool.DEFAULT_OUTPUT_WIDTH
        ), any_active

    def backward(self, error_signal, brain_perceptrons):
        """
        Backward credit assignment through the pipeline.

        Error flows from the last layer backward to the first.
        Each layer receives: error * decay^(total_layers - layer_index - 1)

        Args:
            error_signal: float — the outcome error to distribute
            brain_perceptrons: list of all brain perceptrons
        """
        if not self._layer_inputs:
            return

        for i in reversed(range(self.num_layers)):
            pool = self.pools[i]

            # Decay: deeper layers (closer to output) get more error
            # Layer i gets: error * decay^(num_layers - i - 1)
            layers_from_output = self.num_layers - i - 1
            layer_error = error_signal * (self.credit_decay ** layers_from_output)

            # Record error for pool's adaptive spawn threshold
            pool.record_error(abs(layer_error))
            pool.update_spawn_threshold()

            # Update all perceptrons in this pool
            layer_input = self._layer_inputs[i]
            pool_perceptrons = [p for p in brain_perceptrons if p.pool_id == pool.pool_id]

            for p in pool_perceptrons:
                p.update(layer_input, layer_error)

    def get_pool_at_layer(self, layer_index):
        """Get pool at a specific layer index."""
        if 0 <= layer_index < len(self.pools):
            return self.pools[layer_index]
        return None

    def get_input_width_for_layer(self, layer_index):
        """
        Get the expected input width for a layer.
        Layer 0: raw state width (caller must provide)
        Layer N: previous pool's output_width
        """
        if layer_index == 0:
            return None  # caller determines from raw state
        return self.pools[layer_index - 1].output_width

    def is_fallback_needed(self):
        """
        Returns True if ALL pools are empty — pipeline has no learned
        perceptrons and should fall through to Markov/hardcoded logic.
        """
        return not self.active

    def get_total_authority(self):
        """Weighted authority across all pools."""
        if not self.pools:
            return 0.0
        return np.mean([p.authority for p in self.pools])

    def get_layer_authorities(self):
        """Get authority per layer for logging."""
        return {pool.name: pool.authority for pool in self.pools}

    # =================================================================
    # SPAWNING INTO POOLS
    # =================================================================

    def spawn_into_pool(self, layer_index, perceptron, brain):
        """
        Add a perceptron to a specific pool layer.
        Checks residual file first for reuse.

        Args:
            layer_index: which layer to spawn into
            perceptron: the Perceptron to add
            brain: Brain instance (to call brain.add())

        Returns:
            The perceptron that was added (may be restored from residual)
        """
        pool = self.pools[layer_index]

        # Check residual first
        if perceptron.trigger_context and pool.has_residual(perceptron.trigger_context):
            restored = pool.restore_from_residual(perceptron.trigger_context)
            if restored is not None:
                restored.pool_id = pool.pool_id
                restored.layer_index = layer_index
                brain.add(restored)
                print(f"  🔄 RESTORED from residual: {pool.name} [{restored.trigger_context}]")
                return restored

        # Fresh spawn
        perceptron.pool_id = pool.pool_id
        perceptron.layer_index = layer_index
        brain.add(perceptron)
        pool.spawn_count += 1

        return perceptron

    def prune_from_pool(self, layer_index, perceptron, brain_perceptrons, timestep=0):
        """
        Remove a perceptron from a pool, paging it to residual.

        Returns: True if removed
        """
        pool = self.pools[layer_index]

        # Page to residual before removing
        entry = pool.page_to_residual(perceptron)
        if entry and hasattr(entry, 'get'):
            entry['paged_at'] = timestep

        # Remove from brain's master list by clearing pool_id
        # (actual removal from brain.perceptrons happens in caller)
        perceptron.pool_id = None
        perceptron.layer_index = None

        return True

    # =================================================================
    # SERIALIZATION
    # =================================================================

    def get_save_state(self):
        """Serialize pipeline metadata for checkpoint."""
        return {
            'pipeline_id': self.pipeline_id,
            'name': self.name,
            'credit_decay': self.credit_decay,
            'pools': [pool.get_save_state() for pool in self.pools],
        }

    def load_save_state(self, state_dict):
        """Restore pipeline metadata from checkpoint."""
        if state_dict is None:
            return
        self.credit_decay = state_dict.get('credit_decay', self.DEFAULT_CREDIT_DECAY)
        pool_states = state_dict.get('pools', [])
        for i, ps in enumerate(pool_states):
            if i < len(self.pools):
                self.pools[i].load_save_state(ps)

    def get_status(self, brain_perceptrons):
        """Get pipeline status for logging."""
        status = {
            'pipeline': self.pipeline_id,
            'active': self.active,
            'total_authority': self.get_total_authority(),
            'layers': [],
        }
        for i, pool in enumerate(self.pools):
            n = pool.get_perceptron_count(brain_perceptrons)
            status['layers'].append({
                'name': pool.name,
                'perceptrons': n,
                'authority': pool.authority,
                'spawn_count': pool.spawn_count,
                'residual_size': len(pool.residual),
                'spawn_threshold': pool.spawn_threshold,
            })
        return status

In [ ]:
# ============================================================================
# CELL 3 PART 1 — PART A: Brain __init__ (All State Variables)
# ============================================================================
# Paste PART A + PART B + PART C together as the complete Cell 3 Part 1.
#
# CHANGES from v17.3:
# 1. NEW: Pipeline definitions for battle, overworld, bag, party
# 2. NEW: Pipeline instances created in __init__
# 3. NEW: Revenge module state (revenge_targets dict)
# 4. NEW: Residual perceptrons file path
# 5. NEW: Pool-aware entity capacity tracking per pipeline
# 6. All existing v17.3 state variables UNCHANGED
# ============================================================================

class Brain:
    def __init__(self):
        # === MASTER PERCEPTRON LIST ===
        # All perceptrons live here. Chain-specific methods filter by .chain
        # Pool-specific methods filter by .pool_id
        self.perceptrons = []

        # =================================================================
        # === PIPELINE DEFINITIONS (NEW) ===
        # =================================================================
        # Each pipeline is an ordered list of pool definitions.
        # The Pipeline and Pool classes (Cell 2) handle output computation,
        # inter-pool wiring, credit assignment, and residual paging.
        #
        # Pool output_width=8 everywhere for uniformity.
        # max_perceptrons per pool varies by expected complexity.
        # =================================================================

        # --- BATTLE PIPELINE (6 layers + outcome observation) ---
        # L0: IDENTIFICATION — what am I fighting, what do I have?
        # L1: THREAT ASSESSMENT — how do I compare to this enemy?
        # L2: STAY-OR-BAIL — should I fight, switch, item, or run?
        # L3: ACTION SELECTION — which move/switch/item specifically?
        # L4: EXECUTION — press the right buttons
        # L5: OUTCOME OBSERVATION — generate learning signals (not action)
        self.battle_pipeline = Pipeline(
            pipeline_id="battle",
            name="Battle Pipeline",
            pool_definitions=[
                {'name': 'identification',    'output_width': 8, 'max_perceptrons': 15},
                {'name': 'threat_assessment',  'output_width': 8, 'max_perceptrons': 20},
                {'name': 'stay_or_bail',       'output_width': 8, 'max_perceptrons': 15},
                {'name': 'action_selection',   'output_width': 8, 'max_perceptrons': 20},
                {'name': 'execution',          'output_width': 8, 'max_perceptrons': 10},
                {'name': 'outcome_observation','output_width': 8, 'max_perceptrons': 10},
            ],
            credit_decay=0.7,
        )

        # --- OVERWORLD PIPELINE (7 layers + outcome observation) ---
        # L0: SPATIAL AWARENESS — where am I, what's around me?
        # L1: AREA CLASSIFICATION — what kind of area is this?
        # L2: FRONTIER DETECTION — where is the edge of what I know?
        # L3: OBJECTIVE MANAGEMENT — what is my current goal?
        # L4: PATHFINDING — how do I get to my objective?
        # L5: EXECUTION — which button right now?
        # L6: OUTCOME OBSERVATION — generate learning signals
        self.overworld_pipeline = Pipeline(
            pipeline_id="overworld",
            name="Overworld Pipeline",
            pool_definitions=[
                {'name': 'spatial_awareness',    'output_width': 8, 'max_perceptrons': 15},
                {'name': 'area_classification',  'output_width': 8, 'max_perceptrons': 10},
                {'name': 'frontier_detection',   'output_width': 8, 'max_perceptrons': 15},
                {'name': 'objective_management', 'output_width': 8, 'max_perceptrons': 15},
                {'name': 'pathfinding',          'output_width': 8, 'max_perceptrons': 10},
                {'name': 'execution',            'output_width': 8, 'max_perceptrons': 10},
                {'name': 'outcome_observation',  'output_width': 8, 'max_perceptrons': 10},
            ],
            credit_decay=0.7,
        )

        # --- BAG PIPELINE (3 layers) ---
        # L0: INVENTORY AWARENESS — what do I have, what does party need?
        # L1: ITEM SELECTION — which item on which mon?
        # L2: EXECUTION — navigate bag menus
        self.bag_pipeline = Pipeline(
            pipeline_id="bag",
            name="Bag Pipeline",
            pool_definitions=[
                {'name': 'inventory_awareness', 'output_width': 8, 'max_perceptrons': 10},
                {'name': 'item_selection',      'output_width': 8, 'max_perceptrons': 10},
                {'name': 'execution',           'output_width': 8, 'max_perceptrons': 8},
            ],
            credit_decay=0.7,
        )

        # --- PARTY PIPELINE (2 layers) ---
        # L0: ASSESSMENT — evaluate each slot
        # L1: EXECUTION — navigate to target slot, confirm
        self.party_pipeline = Pipeline(
            pipeline_id="party",
            name="Party Pipeline",
            pool_definitions=[
                {'name': 'assessment', 'output_width': 8, 'max_perceptrons': 10},
                {'name': 'execution',  'output_width': 8, 'max_perceptrons': 8},
            ],
            credit_decay=0.7,
        )

        # All pipelines indexed by id for generic access
        self.pipelines = {
            'battle': self.battle_pipeline,
            'overworld': self.overworld_pipeline,
            'bag': self.bag_pipeline,
            'party': self.party_pipeline,
        }

        # =================================================================
        # === RESIDUAL PERCEPTRONS FILE (NEW) ===
        # =================================================================
        RESIDUAL_FILE = BASE_PATH / "residual_perceptrons.json"
        self.RESIDUAL_FILE = RESIDUAL_FILE

        # =================================================================
        # === REVENGE MODULE (NEW) ===
        # =================================================================
        # Persistent memory of losses. Drives grinding → retry cycle.
        # Stored in checkpoint, survives restarts.
        #
        # Lifecycle:
        #   Loss → set target level (enemy avg + 2*losses margin)
        #        → grind until team avg >= target
        #        → ready → attempt
        #        → win = cleared / lose = raise target, retry
        #
        # target_id format: "map{map_id}_pos{x}_{y}" for location-based
        #                   "trainer_{species_list_hash}" for trainer-based
        # =================================================================
        self.revenge_targets = {}
        # {
        #     'target_id': {
        #         'map_id': int,
        #         'position': (x, y),
        #         'enemy_type': 'trainer'|'wild'|'gym',
        #         'enemy_species': [list],
        #         'enemy_avg_level': float,
        #         'my_avg_level_at_loss': float,
        #         'target_avg_level': float,  # enemy + margin
        #         'losses_here': int,
        #         'attempts': int,
        #         'status': 'grinding'|'ready'|'cleared',
        #         'strategy_notes': {
        #             'moves_that_failed': [...],
        #             'types_that_worked': [...],
        #         },
        #         'created_at': int,  # timestep
        #         'last_updated': int,
        #     }
        # }

        # Revenge module constants
        self.REVENGE_BASE_MARGIN = 2      # base level margin over enemy
        self.REVENGE_MARGIN_PER_LOSS = 2  # additional margin per repeated loss
        self.REVENGE_MAX_MARGIN = 15      # cap on total margin
        self.REVENGE_MAX_TARGETS = 10     # max tracked revenge targets

        # =================================================================
        # === PER-CHAIN LEARNING STATE HISTORY ===
        # =================================================================
        # Each chain tracks its own previous learning states for error computation
        self.prev_learning_states = deque(maxlen=50)          # overworld (legacy)
        self.prev_battle_learning_states = deque(maxlen=50)   # battle chain
        self.prev_party_learning_states = deque(maxlen=20)    # party chain
        self.prev_bag_learning_states = deque(maxlen=20)      # bag chain

        self.prev_context_states = deque(maxlen=10)
        self.last_positions = deque(maxlen=30)
        self.action_history = deque(maxlen=100)

        self.control_mode = "move"
        self.timestep = 0
        self.last_action = None
        self.last_direction = 0

        self.MOVE_UTILITY_FLOOR = 0.05
        self.INTERACT_UTILITY_FLOOR = 0.15

        # === PER-CHAIN ENTITY CAPACITY ===
        # Each chain has its own entity budget — battle needs fewer than overworld
        self.ENTITY_CAPACITY = {
            'overworld': 20,
            'battle': 10,
            'party': 5,
            'bag': 5,
            'shared': 10,
        }
        self.ENTITY_CAPACITY_GROWTH = 1.5

        # Per-chain spawn/merge counters
        self.entity_spawn_counts = {'overworld': 0, 'battle': 0, 'party': 0, 'bag': 0, 'shared': 0}
        self.entity_merge_counts = {'overworld': 0, 'battle': 0, 'party': 0, 'bag': 0, 'shared': 0}
        self.ENTITY_CLUSTER_SIMILARITY = 0.85
        self.ENTITY_MIN_ACTIVATIONS = 10

        # Per-chain innate entity flags
        self.innate_entities_spawned_overworld = False
        self.innate_entities_spawned_battle = False
        # party and bag chains don't need innate entities (too few dims)

        # Legacy compatibility
        self.entity_capacity = self.ENTITY_CAPACITY['overworld']
        self.entity_spawn_count = 0
        self.entity_merge_count = 0
        self.innate_entities_spawned = False

        # === PERSISTENT EXPLORATION MEMORY ===
        self.EXPLORATION_MEMORY_FILE = BASE_PATH / "exploration_memory.json"
        self.exploration_memory = {}
        self.current_map_id = None
        self.SAVE_INTERVAL = 100

        self.DIRECTION_NAMES = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
        self.DIRECTION_TO_INT = {"DOWN": 0, "UP": 1, "LEFT": 2, "RIGHT": 3}
        self.INT_TO_ACTION = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}

        self.DIRECTION_DELTAS_INT = {0: (0, 1), 1: (0, -1), 2: (-1, 0), 3: (1, 0)}
        self.ACTION_DELTAS = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}
        self.DELTA_TO_DIRECTION = {(0, 1): 0, (0, -1): 1, (-1, 0): 2, (1, 0): 3}

        self.load_exploration_memory()

        # === MARKOV TRANSITION SYSTEM (OVERWORLD) ===
        self.taught_transitions = []
        self.taught_batches = []
        self.taught_metadata = {}
        self.markov_enabled = True
        self.markov_action_count = 0
        self.curiosity_action_count = 0
        self.last_markov_score = 0.0
        self.last_markov_action = None

        # === BATTLE THREAD STATE ===
        self.battle_transitions = []
        self.battle_sequences = []
        self.battle_metadata = {}
        self.battle_loaded = False
        self.battle_action_count = 0
        self.battle_markov_action_count = 0
        self.current_battle_id = 0
        self.battle_frame_count = 0
        self.last_battle_markov_score = 0.0
        self.last_battle_markov_action = None
        self.in_battle_last_frame = False
        self.battle_action_history = deque(maxlen=100)

        # === BATTLE DATA ===
        self.battle_data = DEFAULT_BATTLE_DATA.copy()
        self.prev_battle_data = DEFAULT_BATTLE_DATA.copy()

        # === PARTY DATA ===
        self.party_data = DEFAULT_PARTY_DATA.copy()
        self.prev_party_data = DEFAULT_PARTY_DATA.copy()

        # === MENU DATA (Lua v3) ===
        self.menu_data = DEFAULT_MENU_DATA.copy()
        self.prev_menu_data = DEFAULT_MENU_DATA.copy()

        # === BAG DATA (Lua v3) ===
        self.bag_data = DEFAULT_BAG_DATA.copy()
        self.prev_bag_data = DEFAULT_BAG_DATA.copy()

        # === RAW GAME STATE (Lua v3) ===
        self.game_state_raw = 0

        # === BAG THREAD STATE ===
        self.bag_thread_active = False
        self.bag_thread_context = "none"
        self.bag_thread_entered_at = 0
        self.bag_thread_action_count = 0
        self.bag_thread_last_action = None
        self.BAG_THREAD_TIMEOUT = 180
        self.bag_thread_total_actions = 0
        self.bag_thread_markov_actions = 0

        self.bag_transitions = []
        self.bag_metadata = {}
        self.bag_loaded = False
        self.bag_action_history = deque(maxlen=50)
        self.last_bag_markov_score = 0.0
        self.last_bag_markov_action = None

        # === ITEM KNOWLEDGE ===
        self.item_knowledge = {}
        self.item_knowledge_dirty = False
        self.pending_item_observation = None
        self.ITEM_OBSERVE_WAIT_FRAMES = 15

        # === EVENT TIMELINE (taught — consumed) ===
        self.event_timeline = []
        self.event_segments = []
        self.event_preparation_points = []
        self.event_timeline_metadata = {}
        self.event_timeline_loaded = False

        # === MAP BATTLE STATISTICS (Phase 4) ===
        self.map_battle_stats = {}
        self.map_battle_stats_dirty = False
        self.map_step_counters = {}

        # === PREPARATION STATE MACHINE ===
        self.prep_active = False
        self.prep_phase = "idle"
        self.prep_reason = ""
        self.prep_started_at = 0
        self.prep_target = "bag"
        self.prep_target_mc = 2
        self.PREP_TIMEOUT = 60
        self.PREP_COOLDOWN = 200
        self.prep_last_attempt_at = 0
        self.prep_phase_entered_at = 0
        self.PREP_PHASE_TIMEOUT = 15
        self.prep_total_count = 0
        self.prep_success_count = 0

        # === PARTY MENU THREAD ===
        self.party_menu_active = False
        self.party_menu_context = "none"
        self.party_menu_entered_at = 0
        self.party_menu_target_slot = -1
        self.party_menu_action_count = 0
        self.PARTY_MENU_TIMEOUT = 120
        self.party_menu_battle_cursor_on_entry = -1
        self.party_menu_awaiting_entry = False
        self.party_menu_last_action = None

        # === BATTLE AWARENESS ===
        self.battle_player_species = -1
        self.battle_enemy_species = -1
        self.battle_start_hp = -1
        self.battle_enemy_start_hp = -1
        self.battle_menu_state = "unknown"
        self.battle_cursor_action_count = 0

        self.turn_start_player_hp = -1
        self.turn_start_enemy_hp = -1
        self.turn_start_pp = [-1, -1, -1, -1]
        self.turn_start_enemy_pp = [-1, -1, -1, -1]
        self.turn_start_player_stats = [-1] * 7
        self.turn_start_enemy_stats = [-1] * 7
        self.turn_start_player_status = 0
        self.turn_start_enemy_status = 0
        self.turn_count = 0
        self.last_move_used = -1
        self.last_move_slot = -1

        # === RUNNING DECISION ===
        self.battle_no_damage_turns = 0
        self.battle_hp_trend = deque(maxlen=10)
        self.battle_low_hp_exits = 0
        self.BATTLE_RUN_NO_DAMAGE_THRESHOLD = 4
        self.BATTLE_RUN_HP_RATIO_THRESHOLD = 0.25
        self.BATTLE_RUN_ENABLED = True

        # === FORCED SWITCH ===
        self.forced_switch_pending = False
        self.forced_switch_target_slot = -1

        # === ROSTER ===
        self.roster = {}
        self.roster_dirty = False

        # === MOVE KNOWLEDGE ===
        self.move_knowledge = {}
        self.move_knowledge_dirty = False
        self.enemy_move_knowledge = {}

        # === TAUGHT MODEL REFERENCE ===
        self.taught_reference = {'utilities': {}, 'weights': {}, 'loaded': False}

        # === BLEND SYSTEM ===
        self.blend_tier = 0
        self.last_blend_timestep = 0
        self.BLEND_COOLDOWN = 50
        self.blend_count = 0
        self.BLEND_RATIOS = {1: (0.80, 0.20), 2: (0.60, 0.40), 3: (0.40, 0.60)}
        self.BLEND_TIER_TRIGGERS = {
            1: {'pattern_repeats': 3, 'pos_stagnation': 8, 'consecutive': 12},
            2: {'pattern_repeats': 6, 'pos_stagnation': 15, 'consecutive': 15},
            3: {'pattern_repeats': 10, 'state_stagnation_mult': 2.0}
        }

        # === ACTION EXECUTION CONFIRMATION ===
        self.pending_action = None
        self.pending_action_frames = 0
        self.ACTION_CONFIRM_FRAMES = 3
        self.last_confirmed_action = None

        # === TILE INTERACTION PROBING ===
        self.INTERACTION_VERIFY_FRAMES = 8
        self.MIN_SUCCESS_RATE_THRESHOLD = 0.1
        self.pending_interaction_verify = None
        self.interaction_verify_countdown = 0

        # === MENU ESCAPE B-BOOST ===
        self.menu_trap_frames = 0
        self.menu_trap_b_boost = 1.0
        self.menu_trap_position = None
        self.B_BOOST_INCREMENT = 0.15
        self.B_BOOST_MAX = 3.0
        self.MENU_TRAP_THRESHOLD = 5
        self.original_b_utility = None

        # === ADAPTIVE MODE SWAPPING ===
        self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD = 15
        self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD = 25
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.THRESHOLD_INCREMENT = 15
        self.MAX_THRESHOLD = 150
        self.frames_in_current_mode = 0
        self.swap_chain_count = 0
        self.position_at_mode_swap = None
        self.last_map_id = None
        self.last_battle_state = None

        # === UNPRODUCTIVE MODE SWAP ===
        self.UNPRODUCTIVE_SWAP_THRESHOLD = 3
        self.unproductive_swap_count = 0
        self.utilities_before_swapping = {}
        self.swap_chain_active = False

        # === STATE STAGNATION ===
        self.STATE_STAGNATION_THRESHOLD = 20
        self.state_stagnation_count = 0
        self.last_context_state_hash = None
        self.stagnation_initiator_action = None
        self.STAGNATION_INITIATOR_PENALTY = 0.7

        # === BOTH MODE ===
        self.BOTH_MODE_STAGNATION_THRESHOLD = 35
        self.BOTH_MODE_SWAP_THRESHOLD = 5

        # === PROGRESS TRACKING ===
        self.last_direction_for_progress = None
        self.direction_change_counts_as_progress = True

        # === NOVELTY WEIGHTS ===
        self.UNVISITED_TILE_BONUS = 1.5
        self.OBSTRUCTION_PENALTY = 0.25

        # === TRANSITION SYSTEM ===
        self.TRANSITION_ATTRACTION_WEIGHT = 0.6
        self.TEMP_DEBT_ACCUMULATION = 0.5
        self.TEMP_DEBT_DECAY = 0.02
        self.TEMP_DEBT_MAX = 15.0

        # === DEBT ===
        self.MAX_MAP_DEBT = 10.0
        self.MAX_LOCATION_DEBT = 5.0
        self.DEBT_DECAY_RATE = 0.005

        # === TRANSITION BANS ===
        self.transition_bans = {}
        self.BAN_VICINITY_RADIUS = 3
        self.BAN_COVERAGE_LIFT_THRESHOLD = 0.6
        self.BAN_TIMEOUT_STEPS = 300

        # Multi-scale memory
        self.visited_maps = {}
        self.map_novelty_debt = {}
        self.location_memory = {}
        self.location_novelty = {}
        self.action_execution_count = {}

        self.swap_perceptron = ControlSwapPerceptron()
        self.error_history = deque(maxlen=1000)
        self.numeric_error_history = deque(maxlen=1000)
        self.visual_error_history = deque(maxlen=1000)
        self._entity_norms_cache = {}
        self._cache_valid = False

        # === PER-CHAIN ERROR HISTORY ===
        self.battle_error_history = deque(maxlen=500)
        self.party_error_history = deque(maxlen=200)
        self.bag_error_history = deque(maxlen=200)

        # === REPETITION ===
        self.consecutive_action_count = 0
        self.current_repeated_action = None
        self.LEARNING_SLOWDOWN_START = 3
        self.LEARNING_SLOWDOWN_MAX = 10
        self.PENALTY_THRESHOLD = 12
        self.HARD_RESET_THRESHOLD = 18

        # === PATTERN DETECTION ===
        self.PATTERN_CHECK_WINDOW = 50
        self.PATTERN_MIN_REPEATS = 3
        self.PATTERN_MAX_LENGTH = 10
        self.detected_pattern = None
        self.pattern_repeat_count = 0

        # === PROBE CACHE ===
        self._cached_probe_action = None
        self._cached_probe_dir = None
        self._probe_cache_position = None

        # === NAVIGATION ===
        self.nav_active = False
        self.nav_path = []
        self.nav_path_index = 0
        self.nav_target = None
        self.nav_target_list = []
        self.nav_target_index = 0
        self.nav_struck_targets = set()
        self.nav_steps_taken = 0
        self.nav_stagnation_count = 0
        self.nav_last_position = None

        self.KNOWN_AREA_TRIGGER = 20
        self.known_area_counter = 0
        self.NAV_STAGNATION_LIMIT = 8
        self.NAV_MAX_STEPS = 100
        self.NAV_CURIOSITY_WINDOW = 5
        self.nav_curiosity_countdown = 0
        self.NAV_LEARNING_DAMPENING = 0.3

        # === CROSS-MAP NAV ===
        self.nav_map_chain = []
        self.nav_chain_index = 0
        self.nav_cross_map_target = None
        self.nav_cross_map_target_data = None
        self.nav_paused = False
        self.nav_paused_reason = ""
        self.nav_paused_target_map = None
        self.NAV_PAUSE_CHECK_INTERVAL = 50
        self.nav_pause_check_countdown = 0
        self.NAV_CROSS_MAP_REFRESH_INTERVAL = 40
        self.nav_cross_map_refresh_countdown = 0
        self._map_graph = {}
        self._map_graph_dirty = True

        # === TAUGHT NAV TARGETS ===
        self.taught_nav_targets = {}
        self.taught_nav_global_order = []
        self.nav_visited_targets = set()
        self.taught_nav_loaded = False

        # =====================================================================
        # === EVENT RECORDER STATE (v17.2) ===
        # =====================================================================
        self.event_queue = None
        self.event_recorder_active = False

        self.recorded_battle_events = 0
        self.recorded_bag_events = 0
        self.recorded_map_events = 0
        self.recorded_levelup_events = 0

        self.EVENT_RECORDER_FLUSH_INTERVAL = 30

        # =====================================================================
        # === START MENU THREAD STATE (v17.2) ===
        # =====================================================================
        self.start_menu_active = False
        self.start_menu_context = "none"
        self.start_menu_entered_at = 0
        self.start_menu_action_count = 0
        self.start_menu_last_action = None
        self.start_menu_target_mc = -1
        self.START_MENU_TIMEOUT = 90
        self.start_menu_total_actions = 0
        self.start_menu_markov_actions = 0

        self.start_menu_transitions = []
        self.start_menu_metadata = {}
        self.start_menu_loaded = False
        self.start_menu_action_history = deque(maxlen=50)
        self.last_start_menu_markov_score = 0.0
        self.last_start_menu_markov_action = None

        self.START_MENU_OPTIONS = {
            'pokedex': 0, 'pokemon': 1, 'bag': 2, 'trainer_card': 3,
            'save': 4, 'option': 5, 'exit': 6
        }

        # =====================================================================
        # === EMPIRICAL TYPE CHART DISCOVERY STATE (v17.2) ===
        # =====================================================================
        self.move_type_clusters = {}
        self.species_type_clusters = {}
        self.cluster_effectiveness = {}
        self.move_to_cluster = {}
        self.species_to_cluster = {}

        self.CLUSTERING_BATTLE_INTERVAL = 50
        self.CLUSTERING_MIN_MOVES = 3
        self.CLUSTERING_MIN_SPECIES_PER_MOVE = 3
        self.CLUSTERING_SIMILARITY_THRESHOLD = 0.80
        self.battles_since_last_clustering = 0
        self.clustering_run_count = 0
        self.last_clustering_timestep = 0

        self.type_clusters_dirty = False

        self.type_data = None
        self.type_data_loaded = False

        # =====================================================================
        # === TEXT DIALOGUE STATE (v17.3) ===
        # =====================================================================
        self.text_flag = 0
        self.prev_text_flag = 0

        self.dialogue_active = False
        self.dialogue_is_choice = False
        self.dialogue_is_pure_text = False
        self.dialogue_is_battle_text = False

        self.DIALOGUE_CHOICE_MM_MAX = 3

        self.dialogue_skip_action_count = 0
        self.dialogue_choice_action_count = 0
        self.dialogue_frames_total = 0
        self.dialogue_entered_at = 0
        self.dialogue_last_action = None

    # =========================================================================
    # CHAIN-SPECIFIC PERCEPTRON ACCESS
    # =========================================================================

    def actions(self, chain=None):
        """Get action perceptrons, optionally filtered by chain."""
        if chain is None:
            return [p for p in self.perceptrons if p.kind == "action"]
        return [p for p in self.perceptrons if p.kind == "action" and p.chain == chain]

    def entities(self, chain=None):
        """Get entity perceptrons, optionally filtered by chain."""
        if chain is None:
            return [p for p in self.perceptrons if p.kind == "entity"]
        return [p for p in self.perceptrons if p.kind == "entity" and p.chain == chain]

    def add(self, p):
        """Add perceptron to master list. Invalidates cache."""
        self.perceptrons.append(p)
        self._cache_valid = False

    def get_chain_entity_count(self, chain):
        """Count entities in a specific chain."""
        return len(self.entities(chain=chain))

    def get_chain_entity_capacity(self, chain):
        """Get entity capacity for a chain."""
        return self.ENTITY_CAPACITY.get(chain, 10)

    def get_chain_stats(self):
        """Get perceptron counts per chain for logging."""
        stats = {}
        for chain in ['overworld', 'battle', 'party', 'bag', 'shared']:
            n_act = len(self.actions(chain=chain))
            n_ent = len(self.entities(chain=chain))
            if n_act > 0 or n_ent > 0:
                stats[chain] = {'actions': n_act, 'entities': n_ent}
        return stats

    # =========================================================================
    # POOL-SPECIFIC PERCEPTRON ACCESS (NEW)
    # =========================================================================

    def pool_perceptrons(self, pool_id):
        """Get all perceptrons belonging to a specific pool."""
        return [p for p in self.perceptrons if p.pool_id == pool_id]

    def get_pipeline(self, pipeline_id):
        """Get a pipeline by ID."""
        return self.pipelines.get(pipeline_id)

    def get_pipeline_stats(self):
        """Get pool population stats for all pipelines."""
        stats = {}
        for pid, pipeline in self.pipelines.items():
            p_stats = pipeline.get_status(self.perceptrons)
            stats[pid] = p_stats
        return stats

    def get_pipeline_summary(self):
        """Compact summary for logging."""
        parts = []
        for pid, pipeline in self.pipelines.items():
            total_p = sum(pool.get_perceptron_count(self.perceptrons) for pool in pipeline.pools)
            total_auth = pipeline.get_total_authority()
            if total_p > 0 or total_auth > 0:
                parts.append(f"{pid}:{total_p}p({total_auth:.0%})")
        return ' | '.join(parts) if parts else 'all empty'

    # =========================================================================
    # REVENGE MODULE ACCESS (NEW)
    # =========================================================================

    def get_active_revenge_target(self):
        """Get the highest-priority active revenge target (grinding or ready)."""
        active = [
            (tid, t) for tid, t in self.revenge_targets.items()
            if t['status'] in ('grinding', 'ready')
        ]
        if not active:
            return None, None

        # Priority: ready > grinding, then by losses_here (more losses = more urgent)
        active.sort(key=lambda x: (
            0 if x[1]['status'] == 'ready' else 1,
            -x[1]['losses_here']
        ))
        return active[0]

    def get_revenge_status(self):
        """Return revenge module state for logging."""
        if not self.revenge_targets:
            return {'active': False, 'targets': 0}
        by_status = {}
        for t in self.revenge_targets.values():
            s = t['status']
            by_status[s] = by_status.get(s, 0) + 1
        return {
            'active': any(t['status'] in ('grinding', 'ready') for t in self.revenge_targets.values()),
            'targets': len(self.revenge_targets),
            'by_status': by_status,
        }

    def record_revenge_loss(self, map_id, position, enemy_species, enemy_avg_level,
                            my_avg_level, is_trainer=False, is_gym=False):
        """
        Record a loss for the revenge module.
        Creates or updates a revenge target.
        """
        target_id = f"map{map_id}_pos{position[0]}_{position[1]}"
        enemy_type = 'gym' if is_gym else ('trainer' if is_trainer else 'wild')

        if target_id in self.revenge_targets:
            t = self.revenge_targets[target_id]
            t['losses_here'] += 1
            t['my_avg_level_at_loss'] = my_avg_level
            t['enemy_avg_level'] = enemy_avg_level
            t['enemy_species'] = enemy_species
            # Raise target: enemy avg + base margin + margin_per_loss * losses
            margin = min(self.REVENGE_MAX_MARGIN,
                        self.REVENGE_BASE_MARGIN + self.REVENGE_MARGIN_PER_LOSS * t['losses_here'])
            t['target_avg_level'] = enemy_avg_level + margin
            t['status'] = 'grinding'
            t['last_updated'] = self.timestep
        else:
            margin = self.REVENGE_BASE_MARGIN
            self.revenge_targets[target_id] = {
                'map_id': map_id,
                'position': position,
                'enemy_type': enemy_type,
                'enemy_species': enemy_species if isinstance(enemy_species, list) else [enemy_species],
                'enemy_avg_level': enemy_avg_level,
                'my_avg_level_at_loss': my_avg_level,
                'target_avg_level': enemy_avg_level + margin,
                'losses_here': 1,
                'attempts': 0,
                'status': 'grinding',
                'strategy_notes': {
                    'moves_that_failed': [],
                    'types_that_worked': [],
                },
                'created_at': self.timestep,
                'last_updated': self.timestep,
            }

        # Evict oldest cleared if over capacity
        if len(self.revenge_targets) > self.REVENGE_MAX_TARGETS:
            cleared = [(tid, t) for tid, t in self.revenge_targets.items() if t['status'] == 'cleared']
            if cleared:
                oldest = min(cleared, key=lambda x: x[1]['last_updated'])
                del self.revenge_targets[oldest[0]]

        print(f"  ⚔️🔥 REVENGE TARGET: {target_id} ({enemy_type}) "
              f"losses={self.revenge_targets[target_id]['losses_here']} "
              f"target_lv={self.revenge_targets[target_id]['target_avg_level']:.0f}")

    def check_revenge_readiness(self):
        """
        Check if any grinding revenge targets are now ready
        (team avg level >= target level).
        """
        if not self.revenge_targets:
            return

        party_levels = [s.get('level', 0) for s in self.party_data.get('slots', [])
                       if s.get('hp', 0) > 0]
        if not party_levels:
            return

        team_avg = np.mean(party_levels)

        for tid, t in self.revenge_targets.items():
            if t['status'] == 'grinding' and team_avg >= t['target_avg_level']:
                t['status'] = 'ready'
                t['last_updated'] = self.timestep
                print(f"  ⚔️✅ REVENGE READY: {tid} (team avg {team_avg:.0f} >= target {t['target_avg_level']:.0f})")

    def record_revenge_victory(self, map_id, position):
        """Mark a revenge target as cleared after winning."""
        target_id = f"map{map_id}_pos{position[0]}_{position[1]}"
        if target_id in self.revenge_targets:
            t = self.revenge_targets[target_id]
            t['status'] = 'cleared'
            t['last_updated'] = self.timestep
            print(f"  ⚔️🏆 REVENGE CLEARED: {target_id} after {t['losses_here']} losses, "
                  f"{t['attempts']} attempts")

    # =========================================================================
    # END OF __init__ — PART A
    # Continue with PART B (methods) below
    # =========================================================================

# =========================================================================
    # CELL 3 PART 1 — PART B: Timeline, Map Stats, Preparation,
    #   Start Menu Thread, Dialogue State, Chain Entity Spawning,
    #   Pool-Aware Spawning
    # =========================================================================
    # Contains:
    # - Event timeline (load + query)
    # - Map battle statistics (Phase 4)
    # - should_prepare() + Preparation state machine
    # - Start menu thread (open/close/update/detect)
    # - update_dialogue_state() — text flag interpretation
    # - is_dialogue_skip_state() — query for action selection
    # - update_menu_trap_tracking() exempts dialogue
    # - Chain-specific entity spawning + innate entities per chain
    # - Chain-specific clustering
    # - NEW: Pool-aware entity spawning (spawn_into_pipeline_pool)
    # - NEW: Pipeline pool clustering
    # - NEW: Residual file save/load
    #
    # CHANGES from v17.3:
    # 1. NEW: spawn_into_pipeline_pool() — spawns perceptron into a
    #    specific pipeline pool layer, checking residual first
    # 2. NEW: cluster_pipeline_pool() — clusters perceptrons within a pool
    # 3. NEW: save_residual_file() / load_residual_file() — persistence
    # 4. NEW: get_pool_spawn_context() — determines trigger_context for
    #    residual keying based on game state
    # 5. All other methods UNCHANGED from v17.3
    # =========================================================================

    # =========================================================================
    # EVENT TIMELINE
    # =========================================================================

    def load_event_timeline(self, filepath=None):
        filepath = filepath or EVENT_TIMELINE_FILE
        try:
            if not Path(filepath).exists():
                self.event_timeline_loaded = False
                print(f"  📅 No event timeline found at {filepath}")
                return
            with open(filepath, 'r') as f:
                data = json.load(f)
            self.event_timeline = data.get('events', [])
            self.event_segments = data.get('segments', [])
            self.event_preparation_points = data.get('preparation_points', [])
            self.event_timeline_metadata = data.get('metadata', {})
            self.event_timeline_loaded = True
            n_ev = len(self.event_timeline)
            n_seg = len(self.event_segments)
            n_prep = len(self.event_preparation_points)
            type_counts = {}
            for e in self.event_timeline:
                t = e.get('type', 'unknown')
                type_counts[t] = type_counts.get(t, 0) + 1
            print(f"  📅 Event timeline loaded: {n_ev} events, {n_seg} segments, {n_prep} prep")
            if type_counts:
                print(f"     Types: {', '.join(f'{k}:{v}' for k, v in type_counts.items())}")
            for pp in self.event_preparation_points[:3]:
                print(f"     Prep #{pp.get('before_nav_order','?')}: {pp.get('reason','?')} (HP<{pp.get('party_hp_threshold','?')})")
        except Exception as e:
            print(f"  ⚠️ Error loading event timeline: {e}")
            self.event_timeline = []
            self.event_segments = []
            self.event_preparation_points = []
            self.event_timeline_metadata = {}
            self.event_timeline_loaded = False

    def get_nearest_nav_order(self, raw_x, raw_y, map_id):
        if not self.taught_nav_loaded: return -1
        map_targets = self.taught_nav_targets.get(map_id, [])
        if map_targets:
            best_order, best_dist = -1, float('inf')
            for t in map_targets:
                pos = t.get('position', [0, 0])
                dist = abs(raw_x - pos[0]) + abs(raw_y - pos[1])
                if dist < best_dist:
                    best_dist, best_order = dist, t.get('order', -1)
            return best_order
        if self.nav_visited_targets:
            return max(self.nav_visited_targets)
        return -1

    def get_upcoming_events(self, current_nav_order, lookahead=5):
        if not self.event_timeline_loaded or current_nav_order < 0: return []
        upcoming = [e for e in self.event_timeline
                    if 0 <= e.get('nav_target_order', -1) - current_nav_order <= lookahead]
        upcoming.sort(key=lambda e: (e.get('nav_target_order', 0), e.get('order', 0)))
        return upcoming

    def get_upcoming_battles(self, current_nav_order, lookahead=5):
        return [e for e in self.get_upcoming_events(current_nav_order, lookahead) if e.get('type') == 'battle']

    def get_upcoming_trainer_battles(self, current_nav_order, lookahead=5):
        return [b for b in self.get_upcoming_battles(current_nav_order, lookahead)
                if b.get('battle_info', {}).get('battle_type') == 'trainer']

    def get_segment_difficulty(self, from_nav_order, to_nav_order):
        if not self.event_timeline_loaded: return None
        for seg in self.event_segments:
            if seg.get('from_nav_order') == from_nav_order and seg.get('to_nav_order') == to_nav_order:
                return seg
        for seg in self.event_segments:
            if seg.get('from_nav_order', -1) <= from_nav_order and seg.get('to_nav_order', -1) >= to_nav_order:
                return seg
        return None

    def get_preparation_point(self, nav_order):
        if not self.event_timeline_loaded: return None
        for pp in self.event_preparation_points:
            if abs(nav_order - pp.get('before_nav_order', -999)) <= 1:
                return pp
        return None

    def get_estimated_hp_cost_ahead(self, current_nav_order, lookahead=5):
        if not self.event_timeline_loaded: return 0.0
        for seg in self.event_segments:
            sf, st = seg.get('from_nav_order', -1), seg.get('to_nav_order', -1)
            if sf >= 0 and st >= 0 and sf <= current_nav_order and st <= current_nav_order + lookahead:
                return seg.get('total_hp_cost', 0.0)
        return sum(b.get('battle_info', {}).get('hp_cost', 0.15)
                   for b in self.get_upcoming_battles(current_nav_order, lookahead))

    def get_timeline_status(self):
        if not self.event_timeline_loaded: return {'loaded': False}
        return {'loaded': True, 'events': len(self.event_timeline),
                'segments': len(self.event_segments),
                'prep_points': len(self.event_preparation_points),
                'nav_covered': self.event_timeline_metadata.get('nav_targets_covered', [])}

    # =========================================================================
    # MAP BATTLE STATISTICS (Phase 4)
    # =========================================================================

    def get_map_battle_stats(self, map_id):
        if map_id not in self.map_battle_stats:
            self.map_battle_stats[map_id] = {
                'battles_fought': 0, 'wild_battles': 0, 'trainer_battles': 0, 'losses': 0,
                'total_hp_cost': 0.0, 'avg_hp_cost': 0.0,
                'total_enemy_levels': 0, 'avg_enemy_level': 0.0,
                'species_seen': [], 'total_steps_on_map': 0,
                'encounter_rate': 0.0, 'last_updated': 0,
            }
        return self.map_battle_stats[map_id]

    def update_map_battle_stats(self, map_id, enemy_species, enemy_level,
                                 hp_cost, is_trainer, outcome):
        stats = self.get_map_battle_stats(map_id)
        stats['battles_fought'] += 1
        if is_trainer: stats['trainer_battles'] += 1
        else: stats['wild_battles'] += 1
        if outcome == 'loss': stats['losses'] += 1
        stats['total_hp_cost'] += hp_cost
        stats['avg_hp_cost'] = stats['total_hp_cost'] / max(1, stats['battles_fought'])
        if enemy_level > 0:
            stats['total_enemy_levels'] += enemy_level
            stats['avg_enemy_level'] = stats['total_enemy_levels'] / max(1, stats['battles_fought'])
        if enemy_species > 0 and enemy_species not in stats['species_seen']:
            stats['species_seen'].append(enemy_species)
        steps = self.map_step_counters.get(map_id, 0)
        if steps > 0:
            stats['total_steps_on_map'] = steps
            stats['encounter_rate'] = stats['battles_fought'] / steps
        stats['last_updated'] = self.timestep
        self.map_battle_stats_dirty = True

    def increment_map_steps(self, map_id):
        self.map_step_counters[map_id] = self.map_step_counters.get(map_id, 0) + 1

    def predict_hp_cost_for_map(self, map_id, steps=50):
        stats = self.map_battle_stats.get(map_id)
        if stats is None or stats['battles_fought'] == 0: return 0.0
        rate, cost = stats['encounter_rate'], stats['avg_hp_cost']
        if rate <= 0 or cost <= 0: return 0.0
        return rate * steps * cost

    def predict_hp_cost_for_route(self, map_chain, steps_per_map=50):
        total = 0.0
        for mid in map_chain:
            mc = self.predict_hp_cost_for_map(mid, steps_per_map)
            if mc > 0:
                total += mc
            else:
                all_stats = [s for s in self.map_battle_stats.values() if s['battles_fought'] >= 3]
                if all_stats:
                    ga = sum(s['avg_hp_cost'] for s in all_stats) / len(all_stats)
                    gr = sum(s['encounter_rate'] for s in all_stats) / len(all_stats)
                    total += gr * steps_per_map * ga
                else:
                    total += 0.30
        return total

    def get_autonomous_hp_estimate(self, raw_x, raw_y, map_id):
        stats = self.map_battle_stats.get(map_id)
        if stats and stats['battles_fought'] >= 3:
            mc = self.predict_hp_cost_for_map(map_id, 50)
            conf = min(1.0, stats['battles_fought'] / 10.0)
            if self.nav_map_chain and len(self.nav_map_chain) > 1:
                remaining = []
                for i, cm in enumerate(self.nav_map_chain):
                    if cm == map_id:
                        remaining = self.nav_map_chain[i+1:]
                        break
                if remaining:
                    mc += self.predict_hp_cost_for_route(remaining, 40)
            return mc, conf
        elif stats and stats['battles_fought'] >= 1:
            return self.predict_hp_cost_for_map(map_id, 50), stats['battles_fought'] / 10.0
        else:
            all_stats = [s for s in self.map_battle_stats.values() if s['battles_fought'] >= 3]
            if all_stats:
                ga = sum(s['avg_hp_cost'] for s in all_stats) / len(all_stats)
                gr = sum(s['encounter_rate'] for s in all_stats) / len(all_stats)
                return gr * 50 * ga, 0.2
            return 0.0, 0.0

    def get_map_battle_stats_summary(self):
        if not self.map_battle_stats: return {'maps_with_data': 0, 'total_battles': 0, 'unique_species': 0}
        mwd = len([s for s in self.map_battle_stats.values() if s['battles_fought'] > 0])
        tb = sum(s['battles_fought'] for s in self.map_battle_stats.values())
        species = set()
        for s in self.map_battle_stats.values():
            species.update(s.get('species_seen', []))
        return {'maps_with_data': mwd, 'total_battles': tb, 'unique_species': len(species)}

    # =========================================================================
    # TEXT DIALOGUE STATE (v17.3)
    # =========================================================================

    def update_dialogue_state(self, context_state):
        self.prev_text_flag = self.text_flag

        in_battle = context_state[3] > 0.5
        tf = self.text_flag

        self.dialogue_active = False
        self.dialogue_is_battle_text = False
        self.dialogue_is_choice = False
        self.dialogue_is_pure_text = False

        if tf != 1:
            if self.prev_text_flag == 1 and self.dialogue_entered_at > 0:
                duration = self.timestep - self.dialogue_entered_at
                if duration >= 3:
                    self.dialogue_entered_at = 0
            return

        self.dialogue_active = True

        if self.prev_text_flag != 1:
            self.dialogue_entered_at = self.timestep

        self.dialogue_frames_total += 1

        if in_battle:
            bc = self.battle_data.get('battle_cursor', -1)
            mc = self.battle_data.get('move_cursor', -1)
            if not (0 <= bc <= 3) and not (0 <= mc <= 3):
                self.dialogue_is_battle_text = True
                return
            else:
                self.dialogue_active = False
                return

        mm = self.menu_data.get('mm', -1)
        mc = self.menu_data.get('mc', -1)
        gs = self.game_state_raw
        pc = self.menu_data.get('pc', -1)

        if (0 <= mm <= self.DIALOGUE_CHOICE_MM_MAX and
                0 <= mc <= mm and
                gs != 14 and
                not (gs == 1 and mm >= 4) and
                not (0 <= pc <= 5)):
            self.dialogue_is_choice = True
            return

        self.dialogue_is_pure_text = True

    def is_dialogue_skip_state(self):
        return self.dialogue_is_pure_text or self.dialogue_is_battle_text

    def is_dialogue_choice_state(self):
        return self.dialogue_is_choice

    def get_dialogue_status(self):
        if not self.dialogue_active:
            return {
                'active': False,
                'total_skip_actions': self.dialogue_skip_action_count,
                'total_choice_actions': self.dialogue_choice_action_count,
                'total_frames': self.dialogue_frames_total,
            }
        return {
            'active': True,
            'is_pure_text': self.dialogue_is_pure_text,
            'is_choice': self.dialogue_is_choice,
            'is_battle_text': self.dialogue_is_battle_text,
            'frames_in_current': self.timestep - self.dialogue_entered_at if self.dialogue_entered_at > 0 else 0,
            'total_skip_actions': self.dialogue_skip_action_count,
            'total_choice_actions': self.dialogue_choice_action_count,
            'total_frames': self.dialogue_frames_total,
        }

    # =========================================================================
    # START MENU THREAD (v17.2)
    # =========================================================================

    def open_start_menu(self, context, target_mc=-1):
        self.start_menu_active = True
        self.start_menu_context = context
        self.start_menu_entered_at = self.timestep
        self.start_menu_action_count = 0
        self.start_menu_last_action = None
        self.start_menu_target_mc = target_mc
        self.start_menu_action_history.clear()

        target_name = "?"
        for name, mc_val in self.START_MENU_OPTIONS.items():
            if mc_val == target_mc:
                target_name = name
                break

        print(f"  📋 START MENU OPEN: {context}"
              f"{f' → {target_name}(mc={target_mc})' if target_mc >= 0 else ' (Markov)'}")

    def close_start_menu(self, reason=""):
        if self.start_menu_active:
            duration = self.timestep - self.start_menu_entered_at
            print(f"  📋 START MENU CLOSED: {reason} ({self.start_menu_context} "
                  f"{duration}f {self.start_menu_action_count}act)")
        self.start_menu_active = False
        self.start_menu_context = "none"
        self.start_menu_entered_at = 0
        self.start_menu_action_count = 0
        self.start_menu_last_action = None
        self.start_menu_target_mc = -1

    def is_start_menu_active(self):
        return self.start_menu_active

    def set_start_menu_last_action(self, action_name):
        self.start_menu_last_action = action_name

    def update_start_menu_state(self, context_state):
        in_battle = context_state[3] > 0.5
        gs = self.game_state_raw
        mc = self.menu_data.get('mc', -1)
        mm = self.menu_data.get('mm', -1)
        pc = self.menu_data.get('pc', -1)
        prev_gs = getattr(self, '_prev_game_state_raw', 0)

        if self.start_menu_active:
            if self.timestep - self.start_menu_entered_at > self.START_MENU_TIMEOUT:
                self.close_start_menu("timeout")
                return
            if in_battle:
                self.close_start_menu("battle_started")
                return
            if gs != 1:
                if gs == 14 and self.start_menu_context == "preparation":
                    self.close_start_menu("bag_opened_success")
                    return
                self.close_start_menu("gs_left_1")
                return
            if 0 <= pc <= 5:
                self.close_start_menu("party_took_over")
                return
            return

        if self.party_menu_active or self.bag_thread_active:
            return
        if in_battle:
            return
        if self.prep_active:
            return

        if gs == 1 and prev_gs != 1:
            if 0 <= mc <= 6 and mm >= 4:
                if not (0 <= pc <= 5):
                    self.open_start_menu("unknown")
                    return

        if gs == 1 and not self.start_menu_active:
            if 0 <= mc <= 6 and mm >= 4 and not (0 <= pc <= 5):
                if self.state_stagnation_count >= 3:
                    self.open_start_menu("unknown")
                    return

    def get_start_menu_status(self):
        if not self.start_menu_active:
            return {'active': False}
        return {
            'active': True,
            'context': self.start_menu_context,
            'target_mc': self.start_menu_target_mc,
            'frames': self.timestep - self.start_menu_entered_at,
            'actions': self.start_menu_action_count,
            'total_actions': self.start_menu_total_actions,
            'markov_actions': self.start_menu_markov_actions,
        }

    # =========================================================================
    # PREPARATION (Phase 3 + Phase 4 + v17.2 Start Menu Integration)
    # =========================================================================

    def should_prepare(self, raw_x, raw_y, map_id):
        if self.prep_active or self.party_menu_active or self.bag_thread_active: return False, "", ""
        if self.start_menu_active: return False, "", ""
        if self.game_state_raw != 0: return False, "", ""
        if self.timestep - self.prep_last_attempt_at < self.PREP_COOLDOWN: return False, "", ""

        lowest_hp = self.get_lowest_hp_ratio()
        has_status = self.has_status_condition_in_party()
        healing = self.get_healing_items()
        has_healing = len(healing) > 0

        # Tier 1: Timeline
        if self.event_timeline_loaded:
            cn = self.get_nearest_nav_order(raw_x, raw_y, map_id)
            if cn >= 0:
                pp = self.get_preparation_point(cn)
                if pp:
                    thr = pp.get('party_hp_threshold', 0.8)
                    if lowest_hp < thr and has_healing:
                        return True, f"timeline prep #{pp.get('before_nav_order','?')}: {pp.get('reason','')} HP {lowest_hp:.0%}<{thr:.0%}", "bag"
                hc = self.get_estimated_hp_cost_ahead(cn, 5)
                if hc > 0:
                    if lowest_hp < hc and has_healing:
                        return True, f"timeline survival: HP {lowest_hp:.0%}<cost {hc:.0%}", "bag"
                    tr = self.get_upcoming_trainer_battles(cn, 3)
                    if tr and lowest_hp < 0.5 and has_healing:
                        return True, f"timeline trainer: {len(tr)} ahead HP {lowest_hp:.0%}", "bag"

        # Tier 2: Autonomous
        ac, acf = self.get_autonomous_hp_estimate(raw_x, raw_y, map_id)
        if acf >= 0.3 and ac > 0:
            if lowest_hp < ac and has_healing:
                return True, f"autonomous: HP {lowest_hp:.0%}<cost {ac:.0%} (conf {acf:.0%})", "bag"
            stats = self.map_battle_stats.get(map_id)
            if stats and stats['trainer_battles'] > 0 and lowest_hp < 0.5 and has_healing:
                return True, f"autonomous trainer map: HP {lowest_hp:.0%}", "bag"

        # Universal
        if lowest_hp < 0.25 and has_healing:
            return True, f"critical HP: {lowest_hp:.0%}", "bag"
        if has_status:
            for iid, qty, cat, conf in healing:
                if cat in ('heal_status', 'heal_both') and conf >= 0.4:
                    return True, f"status condition, cure item {iid}", "bag"

        return False, "", ""

    def start_preparation(self, reason, target="bag"):
        self.prep_active = True
        self.prep_reason = reason
        self.prep_started_at = self.timestep
        self.prep_last_attempt_at = self.timestep
        self.prep_target = target
        self.prep_phase_entered_at = self.timestep
        self.prep_total_count += 1

        if target == "bag":
            self.prep_target_mc = 2
        elif target == "pokemon":
            self.prep_target_mc = 1
        else:
            self.prep_target_mc = 2

        if self.start_menu_loaded:
            self.prep_phase = "pressing_start"
        else:
            self.prep_phase = "pressing_start"

        print(f"  🎯 PREP START: {target} | {reason}"
              f"{' (Markov nav)' if self.start_menu_loaded else ' (hardcoded)'}")

    def abort_preparation(self, reason=""):
        if self.prep_active:
            print(f"  🎯 PREP ABORT: {reason} ({self.prep_phase} {self.timestep - self.prep_started_at}f)")
        self.prep_active = False
        self.prep_phase = "idle"
        self.prep_reason = ""
        if self.start_menu_active and self.start_menu_context == "preparation":
            self.close_start_menu("prep_aborted")

    def is_preparation_active(self):
        return self.prep_active

    def update_preparation_state(self, context_state):
        if not self.prep_active: return
        in_battle = context_state[3] > 0.5
        gs = self.game_state_raw
        mc = self.menu_data.get('mc', -1)

        if self.bag_thread_active and self.prep_target == "bag":
            self.prep_success_count += 1
            print(f"  🎯 PREP SUCCESS: bag ({self.timestep - self.prep_started_at}f)")
            self.prep_active = False; self.prep_phase = "idle"; return
        if self.party_menu_active and self.prep_target == "party":
            self.prep_success_count += 1
            print(f"  🎯 PREP SUCCESS: party ({self.timestep - self.prep_started_at}f)")
            self.prep_active = False; self.prep_phase = "idle"; return

        if in_battle: self.abort_preparation("battle"); return
        if self.timestep - self.prep_started_at > self.PREP_TIMEOUT: self.abort_preparation("timeout"); return
        if self.timestep - self.prep_phase_entered_at > self.PREP_PHASE_TIMEOUT: self.abort_preparation(f"phase_timeout({self.prep_phase})"); return

        if gs == 1 and self.start_menu_loaded:
            if not self.start_menu_active:
                self.open_start_menu("preparation", target_mc=self.prep_target_mc)
                self.prep_phase = "start_menu_navigating"
                self.prep_phase_entered_at = self.timestep
                return
            elif self.start_menu_active and self.start_menu_context == "preparation":
                if mc == self.prep_target_mc:
                    self.prep_phase = "pressing_a"
                    self.prep_phase_entered_at = self.timestep
                return

        if self.prep_phase == "pressing_start":
            if gs == 1: self.prep_phase = "navigating_menu"; self.prep_phase_entered_at = self.timestep
        elif self.prep_phase == "waiting_for_menu":
            if gs == 1: self.prep_phase = "navigating_menu"; self.prep_phase_entered_at = self.timestep
        elif self.prep_phase == "navigating_menu":
            if mc == self.prep_target_mc: self.prep_phase = "pressing_a"; self.prep_phase_entered_at = self.timestep
            elif gs != 1: self.abort_preparation("menu_closed")
        elif self.prep_phase == "pressing_a":
            if gs == 14 and self.prep_target == "bag": self.prep_phase = "waiting_for_open"; self.prep_phase_entered_at = self.timestep
            elif gs != 1: self.prep_phase = "waiting_for_open"; self.prep_phase_entered_at = self.timestep
        elif self.prep_phase == "start_menu_navigating":
            if gs != 1:
                self.prep_phase = "waiting_for_open"
                self.prep_phase_entered_at = self.timestep

    def get_preparation_action(self):
        if not self.prep_active: return None

        if self.start_menu_active and self.start_menu_context == "preparation":
            return None

        mc = self.menu_data.get('mc', -1)
        if self.prep_phase in ("pressing_start", "waiting_for_menu"):
            return "START"
        elif self.prep_phase == "navigating_menu":
            if mc < 0: return "START"
            if mc == self.prep_target_mc: return "A"
            return "DOWN" if mc < self.prep_target_mc else "UP"
        elif self.prep_phase == "pressing_a":
            return "A"
        elif self.prep_phase == "start_menu_navigating":
            if not self.start_menu_active:
                return "START"
            return None
        return None

    def get_preparation_status(self):
        if not self.prep_active: return {'active': False}
        return {'active': True, 'phase': self.prep_phase, 'target': self.prep_target,
                'reason': self.prep_reason, 'frames': self.timestep - self.prep_started_at,
                'total_count': self.prep_total_count, 'success_count': self.prep_success_count,
                'start_menu_nav': self.start_menu_active and self.start_menu_context == "preparation"}

    # =========================================================================
    # CHAIN-SPECIFIC ENTITY SPAWNING (legacy — still used as fallback)
    # =========================================================================

    def spawn_innate_overworld_entities(self, learning_state):
        if self.innate_entities_spawned_overworld:
            return
        for etype, indices in [("sense_menu", [5, 6]), ("sense_battle", [3, 4]),
                                ("sense_movement", [0, 1]), ("sense_map_transition", [2])]:
            entity = Perceptron("entity", entity_type=etype, chain="overworld")
            entity.ensure_weights(len(learning_state))
            entity.weights = np.zeros(len(learning_state))
            for idx in indices:
                entity.weights[idx] = 0.5 if len(indices) > 1 else 1.0
            self.add(entity)
        self.innate_entities_spawned_overworld = True
        self.innate_entities_spawned = True
        print(f"  🧬 Innate overworld entities spawned (4)")

    def spawn_innate_battle_entities(self, learning_state):
        if self.innate_entities_spawned_battle:
            return
        innate_battle = [
            ("battle_hp_crisis", [1]),
            ("battle_enemy_weak", [20]),
            ("battle_species_match", [0, 19]),
            ("battle_status", [3, 22]),
            ("battle_trainer", [38]),
        ]

        for etype, indices in innate_battle:
            entity = Perceptron("entity", entity_type=etype, chain="battle")
            entity.ensure_weights(len(learning_state))
            entity.weights = np.zeros(len(learning_state))
            for idx in indices:
                if idx < len(learning_state):
                    entity.weights[idx] = 0.5 if len(indices) > 1 else 1.0
            self.add(entity)
        self.innate_entities_spawned_battle = True
        print(f"  🧬 Innate battle entities spawned ({len(innate_battle)})")

    def spawn_entity_for_chain(self, chain, learning_state, context_state=None, raw_position=None):
        count = self.entity_spawn_counts.get(chain, 0)
        entity = Perceptron("entity", entity_type=f"{chain}_spawned_{count}", chain=chain)
        entity.ensure_weights(len(learning_state))

        state_norm = np.linalg.norm(learning_state)
        if state_norm > 0:
            entity.weights = (learning_state / state_norm) * 0.1
        else:
            entity.weights = np.random.randn(len(learning_state)) * 0.001

        entity.utility = 1.0
        self.add(entity)
        self.entity_spawn_counts[chain] = count + 1

        if chain == "overworld":
            self.entity_spawn_count = self.entity_spawn_counts['overworld']

        self.check_chain_entity_capacity(chain)

    def check_chain_entity_capacity(self, chain):
        n_entities = self.get_chain_entity_count(chain)
        capacity = self.get_chain_entity_capacity(chain)

        if n_entities < capacity:
            return

        before = n_entities
        self.cluster_chain_entities(chain)
        after = self.get_chain_entity_count(chain)

        if after >= before * 0.9:
            old_cap = self.ENTITY_CAPACITY[chain]
            self.ENTITY_CAPACITY[chain] = int(old_cap * self.ENTITY_CAPACITY_GROWTH)
            print(f"  🧩 [{chain}] Entity capacity: {old_cap} → {self.ENTITY_CAPACITY[chain]} "
                  f"(clustering {before} → {after})")

    def cluster_chain_entities(self, chain):
        chain_entities = self.entities(chain=chain)

        innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition",
                        "battle_hp_crisis", "battle_enemy_weak", "battle_species_match",
                        "battle_status", "battle_trainer"}
        spawned = [e for e in chain_entities if e.entity_type not in innate_types]
        innate = [e for e in chain_entities if e.entity_type in innate_types]

        if len(spawned) < 2:
            return

        clusterable = [e for e in spawned if len(e.cluster_activations) >= self.ENTITY_MIN_ACTIVATIONS]
        too_young = [e for e in spawned if len(e.cluster_activations) < self.ENTITY_MIN_ACTIVATIONS]

        if len(clusterable) < 2:
            return

        max_len = max(len(e.cluster_activations) for e in clusterable)
        activation_vecs = []
        for e in clusterable:
            vec = list(e.cluster_activations)
            while len(vec) < max_len:
                vec.append(0.0)
            activation_vecs.append(np.array(vec))

        merged_indices = set()
        merge_groups = []

        for i in range(len(clusterable)):
            if i in merged_indices: continue
            group = [i]
            vec_i = activation_vecs[i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10: continue

            for j in range(i + 1, len(clusterable)):
                if j in merged_indices: continue
                vec_j = activation_vecs[j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10: continue
                cosine_sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)
                if cosine_sim >= self.ENTITY_CLUSTER_SIMILARITY:
                    group.append(j)
                    merged_indices.add(j)

            if len(group) > 1:
                merged_indices.add(i)
                merge_groups.append(group)

        if not merge_groups:
            return

        new_entities = []
        merged_set = set()

        for group in merge_groups:
            group_ents = [clusterable[idx] for idx in group]
            min_dim = min(len(e.weights) for e in group_ents if e.weights is not None)
            if min_dim == 0: continue

            avg_w = np.zeros(min_dim)
            for e in group_ents:
                avg_w += e.weights[:min_dim]
            avg_w /= len(group_ents)

            merge_count = self.entity_merge_counts.get(chain, 0)
            merged = Perceptron("entity", entity_type=f"{chain}_merged_{merge_count}", chain=chain)
            merged.weights = avg_w
            merged.utility = max(e.utility for e in group_ents)
            merged.familiarity = np.mean([e.familiarity for e in group_ents])
            merged.learning_rate = np.mean([e.learning_rate for e in group_ents])
            best_ent = max(group_ents, key=lambda e: e.activation_fit_score)
            merged.active_activation = best_ent.active_activation
            merged.activation_fit_score = best_ent.activation_fit_score

            new_entities.append(merged)
            self.entity_merge_counts[chain] = merge_count + 1

            for idx in group:
                merged_set.add(id(clusterable[idx]))

        kept_spawned = [e for e in clusterable if id(e) not in merged_set]

        other_perceptrons = [p for p in self.perceptrons
                             if p.kind != "entity" or p.chain != chain]
        self.perceptrons = other_perceptrons + innate + kept_spawned + too_young + new_entities
        self._cache_valid = False

        total_merged = sum(len(g) for g in merge_groups)
        print(f"  🧩 [{chain}] CLUSTERED: {total_merged} → {len(new_entities)} | "
              f"Total: {self.get_chain_entity_count(chain)}")

    # =========================================================================
    # POOL-AWARE ENTITY SPAWNING (NEW)
    # =========================================================================

    def get_pool_spawn_context(self, pipeline_id, layer_index, game_state_data=None):
        """
        Determine the trigger_context for a new perceptron being spawned
        into a pipeline pool. This key is used for residual file lookup.

        The trigger_context depends on which pipeline and layer:
        - Battle L0 (identification): species ID (enemy or player)
        - Battle L1 (threat): type matchup pair
        - Battle L2+ : situation hash
        - Overworld L0 (spatial): map region
        - Overworld L1 (area): map_id
        - Overworld L2+ : frontier/objective context
        - Bag/Party: generic counter

        Returns: string key for residual lookup
        """
        if game_state_data is None:
            game_state_data = {}

        if pipeline_id == "battle":
            if layer_index == 0:
                # Identification: key by enemy species
                es = game_state_data.get('enemy_species', -1)
                ps = game_state_data.get('player_species', -1)
                if es > 0:
                    return f"battle_id_es{es}"
                return f"battle_id_ps{ps}"
            elif layer_index == 1:
                # Threat: key by species matchup
                es = game_state_data.get('enemy_species', -1)
                ps = game_state_data.get('player_species', -1)
                return f"battle_threat_{ps}v{es}"
            else:
                return f"battle_L{layer_index}_{self.current_battle_id}"

        elif pipeline_id == "overworld":
            if layer_index == 0:
                # Spatial: key by map region
                map_id = game_state_data.get('map_id', self.current_map_id or 0)
                return f"ow_spatial_map{map_id}"
            elif layer_index == 1:
                # Area classification: key by map
                map_id = game_state_data.get('map_id', self.current_map_id or 0)
                return f"ow_area_map{map_id}"
            else:
                map_id = game_state_data.get('map_id', self.current_map_id or 0)
                return f"ow_L{layer_index}_map{map_id}_{self.timestep}"

        elif pipeline_id == "bag":
            pocket = game_state_data.get('pocket', self.bag_data.get('pocket', -1))
            return f"bag_L{layer_index}_pk{pocket}"

        elif pipeline_id == "party":
            return f"party_L{layer_index}_{self.timestep}"

        return f"{pipeline_id}_L{layer_index}_{self.timestep}"

    def spawn_into_pipeline_pool(self, pipeline_id, layer_index, input_state,
                                  game_state_data=None, entity_type=None):
        """
        Spawn a perceptron into a specific pipeline pool layer.

        1. Determine trigger_context for residual keying
        2. Check residual file via Pipeline.spawn_into_pool()
        3. If not in residual, create fresh perceptron
        4. Initialize weights from input_state
        5. Add to brain's master list

        Args:
            pipeline_id: which pipeline ("battle", "overworld", "bag", "party")
            layer_index: which layer (0-based)
            input_state: numpy array — the input this layer received
            game_state_data: dict of relevant game state for context keying
            entity_type: optional explicit entity_type name

        Returns:
            The spawned perceptron (or restored from residual)
        """
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return None

        if layer_index < 0 or layer_index >= pipeline.num_layers:
            return None

        pool = pipeline.pools[layer_index]

        # Check capacity
        n_current = pool.get_perceptron_count(self.perceptrons)
        if n_current >= pool.max_perceptrons:
            # Try clustering first
            self.cluster_pipeline_pool(pipeline_id, layer_index)
            n_current = pool.get_perceptron_count(self.perceptrons)
            if n_current >= pool.max_perceptrons:
                # Still full — grow capacity
                pool.max_perceptrons = int(pool.max_perceptrons * 1.5)
                print(f"  🧩 [{pipeline_id}.{pool.name}] Pool capacity grown to {pool.max_perceptrons}")

        # Determine trigger context
        trigger_context = self.get_pool_spawn_context(pipeline_id, layer_index, game_state_data)

        # Build entity type name
        if entity_type is None:
            entity_type = f"{pipeline_id}_{pool.name}_{pool.spawn_count}"

        # Create perceptron
        p = Perceptron("entity", entity_type=entity_type, chain=pipeline_id)
        p.trigger_context = trigger_context

        # Initialize weights from input state
        p.ensure_weights(len(input_state))
        state_norm = np.linalg.norm(input_state)
        if state_norm > 0:
            p.weights = (input_state / state_norm) * 0.1
        else:
            p.weights = np.random.randn(len(input_state)) * 0.001
        p.utility = 1.0

        # Spawn via pipeline (checks residual first)
        result = pipeline.spawn_into_pool(layer_index, p, self)

        return result

    def cluster_pipeline_pool(self, pipeline_id, layer_index):
        """
        Cluster perceptrons within a specific pipeline pool.
        Similar to cluster_chain_entities but scoped to a single pool.
        Pruned perceptrons are paged to the pool's residual store.
        """
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return

        pool = pipeline.pools[layer_index]
        pool_perceptrons = [p for p in self.perceptrons if p.pool_id == pool.pool_id]

        if len(pool_perceptrons) < 2:
            return

        clusterable = [p for p in pool_perceptrons
                       if len(p.cluster_activations) >= self.ENTITY_MIN_ACTIVATIONS]
        too_young = [p for p in pool_perceptrons
                     if len(p.cluster_activations) < self.ENTITY_MIN_ACTIVATIONS]

        if len(clusterable) < 2:
            return

        max_len = max(len(p.cluster_activations) for p in clusterable)
        activation_vecs = []
        for p in clusterable:
            vec = list(p.cluster_activations)
            while len(vec) < max_len:
                vec.append(0.0)
            activation_vecs.append(np.array(vec))

        merged_indices = set()
        merge_groups = []

        for i in range(len(clusterable)):
            if i in merged_indices:
                continue
            group = [i]
            vec_i = activation_vecs[i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10:
                continue

            for j in range(i + 1, len(clusterable)):
                if j in merged_indices:
                    continue
                vec_j = activation_vecs[j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10:
                    continue
                cosine_sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)
                if cosine_sim >= self.ENTITY_CLUSTER_SIMILARITY:
                    group.append(j)
                    merged_indices.add(j)

            if len(group) > 1:
                merged_indices.add(i)
                merge_groups.append(group)

        if not merge_groups:
            return

        new_perceptrons = []
        paged_set = set()

        for group in merge_groups:
            group_ents = [clusterable[idx] for idx in group]
            min_dim = min(len(p.weights) for p in group_ents if p.weights is not None)
            if min_dim == 0:
                continue

            avg_w = np.zeros(min_dim)
            for p in group_ents:
                avg_w += p.weights[:min_dim]
            avg_w /= len(group_ents)

            # Merged perceptron
            merged = Perceptron("entity",
                               entity_type=f"{pipeline_id}_{pool.name}_merged_{pool.spawn_count}",
                               chain=pipeline_id)
            merged.weights = avg_w
            merged.utility = max(p.utility for p in group_ents)
            merged.familiarity = np.mean([p.familiarity for p in group_ents])
            merged.learning_rate = np.mean([p.learning_rate for p in group_ents])
            best_ent = max(group_ents, key=lambda p: p.activation_fit_score)
            merged.active_activation = best_ent.active_activation
            merged.activation_fit_score = best_ent.activation_fit_score
            merged.pool_id = pool.pool_id
            merged.layer_index = layer_index
            merged.trigger_context = best_ent.trigger_context
            pool.spawn_count += 1

            new_perceptrons.append(merged)

            # Page losers to residual
            for idx in group:
                p = clusterable[idx]
                pool.page_to_residual(p)
                paged_set.add(id(p))

        # Rebuild perceptron list
        kept = [p for p in clusterable if id(p) not in paged_set]
        other = [p for p in self.perceptrons if p.pool_id != pool.pool_id]
        self.perceptrons = other + kept + too_young + new_perceptrons
        self._cache_valid = False

        total_merged = sum(len(g) for g in merge_groups)
        print(f"  🧩 [{pipeline_id}.{pool.name}] POOL CLUSTERED: "
              f"{total_merged} → {len(new_perceptrons)} | "
              f"residual: {len(pool.residual)}")

    # =========================================================================
    # RESIDUAL FILE PERSISTENCE (NEW)
    # =========================================================================

    def save_residual_file(self, filepath=None):
        """Save all pipeline pool residual stores to disk."""
        filepath = filepath or self.RESIDUAL_FILE
        try:
            data = {}
            for pid, pipeline in self.pipelines.items():
                pipeline_residuals = {}
                for i, pool in enumerate(pipeline.pools):
                    if pool.residual:
                        pipeline_residuals[pool.pool_id] = pool.residual
                if pipeline_residuals:
                    data[pid] = pipeline_residuals

            if data:
                with open(filepath, 'w') as f:
                    json.dump(data, f, indent=2)

        except Exception as e:
            print(f"  ⚠️ Error saving residual file: {e}")

    def load_residual_file(self, filepath=None):
        """Load residual stores from disk into pipeline pools."""
        filepath = filepath or self.RESIDUAL_FILE
        try:
            if not Path(filepath).exists():
                return

            with open(filepath, 'r') as f:
                data = json.load(f)

            total_loaded = 0
            for pid, pipeline_residuals in data.items():
                pipeline = self.pipelines.get(pid)
                if pipeline is None:
                    continue
                for pool_id, residual_entries in pipeline_residuals.items():
                    for pool in pipeline.pools:
                        if pool.pool_id == pool_id:
                            pool.residual = residual_entries
                            total_loaded += len(residual_entries)
                            break

            if total_loaded > 0:
                print(f"  🔄 Residual file loaded: {total_loaded} paged perceptrons")

        except Exception as e:
            print(f"  ⚠️ Error loading residual file: {e}")

    # =========================================================================
    # MENU TRAP B-BOOST (moved here from Part 4 for dialogue exemption)
    # =========================================================================

    def update_menu_trap_tracking(self, context_state, action_taken, raw_position=None):
        if context_state[3] > 0.5:
            self.reset_menu_trap_boost()
            return

        if self.party_menu_active:
            self.reset_menu_trap_boost()
            return

        if self.bag_thread_active:
            self.reset_menu_trap_boost()
            return

        if self.start_menu_active:
            self.reset_menu_trap_boost()
            return

        if self.dialogue_active:
            self.reset_menu_trap_boost()
            return

        gs = self.game_state_raw
        if gs == 14:
            self.reset_menu_trap_boost()
            return

        if gs == 1:
            mc = self.menu_data.get('mc', -1)
            mm = self.menu_data.get('mm', -1)
            pc = self.menu_data.get('pc', -1)

            if 0 <= pc <= 6:
                self.reset_menu_trap_boost()
                return

            if 0 <= mc <= 6 and mm >= 4:
                self.reset_menu_trap_boost()
                return

        current_pos = raw_position if raw_position else (round(context_state[0] * 255), round(context_state[1] * 255))
        if self.menu_trap_position is not None and current_pos != self.menu_trap_position:
            self.reset_menu_trap_boost()
            return
        if self.get_context_state_hash(context_state) == self.last_context_state_hash:
            if action_taken in ["A", "B", "Start", "Select"]:
                self.menu_trap_frames += 1
                self.menu_trap_position = current_pos
                if self.menu_trap_frames > self.MENU_TRAP_THRESHOLD:
                    if self.original_b_utility is None:
                        for a in self.actions():
                            if a.action == 'B':
                                self.original_b_utility = a.utility
                                break
                    self.menu_trap_b_boost = min(self.B_BOOST_MAX, self.menu_trap_b_boost + self.B_BOOST_INCREMENT)
        elif current_pos != self.menu_trap_position:
            self.reset_menu_trap_boost()

    def reset_menu_trap_boost(self):
        if self.menu_trap_b_boost > 1.0 and self.original_b_utility is not None:
            for a in self.actions():
                if a.action == 'B':
                    a.utility = self.original_b_utility
                    break
        self.menu_trap_frames = 0
        self.menu_trap_b_boost = 1.0
        self.menu_trap_position = None
        self.original_b_utility = None

    # =========================================================================
    # END OF PART B
    # Continue with PART C below
    # =========================================================================

# =========================================================================
    # CELL 3 PART 1 — PART C: Data Management, Item Knowledge, Battle,
    #   Roster, Move Knowledge, Persistence, Type Discovery, Event Helpers
    # =========================================================================
    # CHANGES from v17.3:
    # 1. UPDATED: get_best_move_for_enemy() — adds Tier 0: pipeline battle
    #    pool query before existing 3-tier fallback. Pipeline output used
    #    when battle pipeline has sufficient authority.
    # 2. NEW: query_battle_pipeline_move_scores() — asks battle pipeline
    #    Layer 3 (action_selection) for move preferences
    # 3. NEW: get_pipeline_pool_signal() — generic pool output query
    # 4. All other methods UNCHANGED from v17.3
    # =========================================================================

    # =========================================================================
    # CHAIN-SPECIFIC HELPERS
    # =========================================================================

    def get_chain_error_history(self, chain):
        if chain == "battle":
            return self.battle_error_history
        elif chain == "party":
            return self.party_error_history
        elif chain == "bag":
            return self.bag_error_history
        else:
            return self.error_history

    def get_chain_spawn_threshold(self, chain, percentile=75):
        history = self.get_chain_error_history(chain)
        if len(history) >= 50:
            return max(0.001, np.percentile(history, percentile))
        return 0.0005

    def get_chain_entity_signal(self, chain, learning_state):
        chain_ents = self.entities(chain=chain)
        if not chain_ents:
            return 0.5
        return np.mean([abs(e.predict(learning_state)) * e.utility for e in chain_ents])

    # =========================================================================
    # PIPELINE POOL SIGNAL QUERY (NEW)
    # =========================================================================

    def get_pipeline_pool_signal(self, pipeline_id, layer_index, input_state=None):
        """
        Get the output of a specific pipeline pool layer.

        If input_state is provided, runs a fresh compute.
        Otherwise returns the cached output from the last forward pass.

        Returns: (output_vector, authority) tuple
        """
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return np.zeros(Pool.DEFAULT_OUTPUT_WIDTH), 0.0

        if layer_index < 0 or layer_index >= pipeline.num_layers:
            return np.zeros(Pool.DEFAULT_OUTPUT_WIDTH), 0.0

        pool = pipeline.pools[layer_index]

        if input_state is not None:
            output = pool.compute_output(input_state, self.perceptrons)
            return output, pool.authority
        else:
            return pool.get_cached_output(), pool.authority

    def query_battle_pipeline_move_scores(self, battle_state):
        """
        Query the battle pipeline's action_selection layer (L3) for
        move preference scores.

        Runs a partial forward pass through battle pipeline layers 0-3,
        then interprets the L3 output as move slot preferences.

        Args:
            battle_state: numpy array from build_learning_state_battle()

        Returns:
            list of (slot_index, score) tuples, sorted by score descending,
            or None if pipeline has insufficient authority
        """
        pipeline = self.battle_pipeline

        # Check if pipeline has any populated pools
        total_pool_perceptrons = sum(
            pool.get_perceptron_count(self.perceptrons)
            for pool in pipeline.pools[:4]  # L0-L3
        )
        if total_pool_perceptrons == 0:
            return None

        # Run forward pass through L0-L3
        current_input = battle_state
        for i in range(min(4, pipeline.num_layers)):
            pool = pipeline.pools[i]
            current_input = pool.compute_output(current_input, self.perceptrons)

        # Check authority of action_selection pool (L3)
        action_pool = pipeline.pools[3] if pipeline.num_layers > 3 else None
        if action_pool is None or action_pool.authority < 0.1:
            return None

        # Interpret L3 output as move slot scores
        # Output width is 8; first 4 dims map to move slots 0-3
        output = current_input
        move_scores = []
        for slot in range(4):
            if slot < len(output):
                move_scores.append((slot, float(output[slot])))
            else:
                move_scores.append((slot, 0.0))

        move_scores.sort(key=lambda x: x[1], reverse=True)
        return move_scores

    # =========================================================================
    # MEMORY MONITORING — ACTIVATION OBSERVATIONS
    # =========================================================================

    def get_total_activation_observations(self):
        return sum(len(p.activation_observations) for p in self.perceptrons)

    def get_activation_observation_stats(self):
        stats = {}
        for chain in ['overworld', 'battle', 'party', 'bag', 'shared']:
            chain_perceptrons = [p for p in self.perceptrons if p.chain == chain]
            if not chain_perceptrons:
                continue
            obs_count = sum(len(p.activation_observations) for p in chain_perceptrons)
            n_perceptrons = len(chain_perceptrons)
            estimated_bytes = obs_count * 80
            stats[chain] = {
                'perceptrons': n_perceptrons,
                'observations': obs_count,
                'avg_per_perceptron': obs_count / max(1, n_perceptrons),
                'estimated_bytes': estimated_bytes,
            }
        total_obs = sum(s['observations'] for s in stats.values())
        total_bytes = sum(s['estimated_bytes'] for s in stats.values())
        stats['_total'] = {
            'observations': total_obs,
            'estimated_bytes': total_bytes,
            'estimated_mb': total_bytes / (1024 * 1024),
        }
        return stats

    # =========================================================================
    # KNOWLEDGE FILE SIZE MONITORING
    # =========================================================================

    def get_knowledge_file_sizes(self):
        files = {
            'model_checkpoint': MODEL_CHECKPOINT_FILE,
            'exploration_memory': self.EXPLORATION_MEMORY_FILE,
            'roster': ROSTER_FILE,
            'move_knowledge': MOVE_KNOWLEDGE_FILE,
            'item_knowledge': ITEM_KNOWLEDGE_FILE,
            'type_clusters': TYPE_CLUSTERS_FILE,
            'residual_perceptrons': self.RESIDUAL_FILE,
        }
        sizes = {}
        for name, filepath in files.items():
            try:
                if Path(filepath).exists():
                    sizes[name] = Path(filepath).stat().st_size
                else:
                    sizes[name] = 0
            except Exception:
                sizes[name] = -1
        sizes['_total'] = sum(v for v in sizes.values() if v > 0)
        return sizes

    # =========================================================================
    # KNOWLEDGE PRUNING
    # =========================================================================

    def prune_stale_move_knowledge(self, min_uses=2, max_entries=500):
        if len(self.move_knowledge) <= max_entries:
            return 0
        pruned = 0
        to_remove = []
        for move_id, data in self.move_knowledge.items():
            if data.get('total_uses', 0) < min_uses:
                to_remove.append(move_id)
        for move_id in to_remove:
            del self.move_knowledge[move_id]
            pruned += 1
        if pruned > 0:
            self.move_knowledge_dirty = True
            print(f"  🧹 Pruned {pruned} move knowledge entries (<{min_uses} uses)")
        return pruned

    def prune_stale_item_knowledge(self, min_uses=1, max_entries=200):
        if len(self.item_knowledge) <= max_entries:
            return 0
        pruned = 0
        to_remove = []
        for item_id, data in self.item_knowledge.items():
            if data.get('uses', 0) < min_uses and data.get('category', 'unknown') == 'unknown':
                to_remove.append(item_id)
        for item_id in to_remove:
            del self.item_knowledge[item_id]
            pruned += 1
        if pruned > 0:
            self.item_knowledge_dirty = True
            print(f"  🧹 Pruned {pruned} item knowledge entries (unused+unknown)")
        return pruned

    # =========================================================================
    # EVENT RECORDER HELPERS
    # =========================================================================

    def push_event(self, event_type, event_data):
        if self.event_queue is None or not self.event_recorder_active:
            return False

        event = {
            'type': event_type,
            'timestep': self.timestep,
            'time': time.time(),
            'map_id': self.current_map_id,
            'data': event_data,
        }

        try:
            self.event_queue.put_nowait(event)
            if event_type == 'battle_end':
                self.recorded_battle_events += 1
            elif event_type == 'bag_session':
                self.recorded_bag_events += 1
            elif event_type == 'map_transition':
                self.recorded_map_events += 1
            elif event_type == 'level_up':
                self.recorded_levelup_events += 1
            return True
        except Exception:
            return False

    def get_event_recorder_stats(self):
        return {
            'active': self.event_recorder_active,
            'battles': self.recorded_battle_events,
            'bags': self.recorded_bag_events,
            'maps': self.recorded_map_events,
            'levelups': self.recorded_levelup_events,
            'total': (self.recorded_battle_events + self.recorded_bag_events +
                      self.recorded_map_events + self.recorded_levelup_events),
        }

    # =========================================================================
    # EMPIRICAL TYPE CHART DISCOVERY — TRACK A
    # =========================================================================

    def run_type_clustering(self):
        eligible_moves = {}
        for move_id, data in self.move_knowledge.items():
            vs = data.get('vs_species', {})
            reliable_species = {sp: sd for sp, sd in vs.items()
                                if sd.get('uses', 0) >= 2}
            if len(reliable_species) >= self.CLUSTERING_MIN_SPECIES_PER_MOVE:
                eligible_moves[move_id] = reliable_species

        if len(eligible_moves) < self.CLUSTERING_MIN_MOVES:
            return

        all_species = set()
        for vs in eligible_moves.values():
            all_species.update(vs.keys())
        all_species = sorted(all_species)
        species_idx = {sp: i for i, sp in enumerate(all_species)}
        n_species = len(all_species)

        if n_species < 2:
            return

        move_ids = sorted(eligible_moves.keys())
        n_moves = len(move_ids)
        damage_matrix = np.full((n_moves, n_species), 0.5)

        for mi, move_id in enumerate(move_ids):
            vs = eligible_moves[move_id]
            max_dmg = max((sd.get('avg', 0.0) for sd in vs.values()), default=1.0)
            if max_dmg <= 0:
                max_dmg = 1.0
            for sp, sd in vs.items():
                si = species_idx[sp]
                damage_matrix[mi, si] = np.clip(sd.get('avg', 0.0) / max_dmg, 0.0, 1.0)

        # Cluster moves
        move_clusters = {}
        move_assigned = set()
        cluster_id = 0

        for i in range(n_moves):
            if i in move_assigned:
                continue
            group = [i]
            vec_i = damage_matrix[i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10:
                continue

            for j in range(i + 1, n_moves):
                if j in move_assigned:
                    continue
                vec_j = damage_matrix[j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10:
                    continue
                sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)
                if sim >= self.CLUSTERING_SIMILARITY_THRESHOLD:
                    group.append(j)
                    move_assigned.add(j)

            if len(group) >= 1:
                move_assigned.add(i)
                move_clusters[cluster_id] = [move_ids[idx] for idx in group]
                cluster_id += 1

        # Cluster species
        species_clusters = {}
        species_assigned = set()
        sp_cluster_id = 0

        for i in range(n_species):
            if i in species_assigned:
                continue
            group = [i]
            vec_i = damage_matrix[:, i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10:
                continue

            for j in range(i + 1, n_species):
                if j in species_assigned:
                    continue
                vec_j = damage_matrix[:, j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10:
                    continue
                sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)
                if sim >= self.CLUSTERING_SIMILARITY_THRESHOLD:
                    group.append(j)
                    species_assigned.add(j)

            if len(group) >= 1:
                species_assigned.add(i)
                species_clusters[sp_cluster_id] = [all_species[idx] for idx in group]
                sp_cluster_id += 1

        # Cluster effectiveness
        cluster_eff = {}
        for mc_id, mc_moves in move_clusters.items():
            mc_indices = [move_ids.index(m) for m in mc_moves if m in move_ids]
            for sc_id, sc_species in species_clusters.items():
                sc_indices = [species_idx[sp] for sp in sc_species if sp in species_idx]
                if not mc_indices or not sc_indices:
                    continue
                values = []
                for mi in mc_indices:
                    for si in sc_indices:
                        val = damage_matrix[mi, si]
                        if val != 0.5:
                            values.append(val)
                if values:
                    cluster_eff[(mc_id, sc_id)] = float(np.mean(values))

        # Reverse lookups
        new_move_to_cluster = {}
        for mc_id, mc_moves in move_clusters.items():
            for m in mc_moves:
                new_move_to_cluster[m] = mc_id

        new_species_to_cluster = {}
        for sc_id, sc_species in species_clusters.items():
            for sp in sc_species:
                new_species_to_cluster[sp] = sc_id

        # Store
        self.move_type_clusters = {int(k): v for k, v in move_clusters.items()}
        self.species_type_clusters = {int(k): v for k, v in species_clusters.items()}
        self.cluster_effectiveness = {f"{k[0]}_{k[1]}": v for k, v in cluster_eff.items()}
        self.move_to_cluster = {int(k): int(v) for k, v in new_move_to_cluster.items()}
        self.species_to_cluster = {int(k): int(v) for k, v in new_species_to_cluster.items()}

        self.clustering_run_count += 1
        self.last_clustering_timestep = self.timestep
        self.battles_since_last_clustering = 0
        self.type_clusters_dirty = True

        print(f"  🧬 TYPE CLUSTERING #{self.clustering_run_count}:")
        print(f"     Moves: {n_moves} → {len(move_clusters)} clusters")
        print(f"     Species: {n_species} → {len(species_clusters)} clusters")
        print(f"     Effectiveness entries: {len(cluster_eff)}")

        for mc_id, mc_moves in sorted(move_clusters.items(), key=lambda x: len(x[1]), reverse=True)[:3]:
            print(f"     Move cluster {mc_id}: {mc_moves}")
        for sc_id, sc_species in sorted(species_clusters.items(), key=lambda x: len(x[1]), reverse=True)[:3]:
            print(f"     Species cluster {sc_id}: {sc_species}")

    def get_cluster_effectiveness_for_move(self, move_id, species_id):
        mc_id = self.move_to_cluster.get(move_id)
        sc_id = self.species_to_cluster.get(species_id)

        if mc_id is None or sc_id is None:
            return None

        key = f"{mc_id}_{sc_id}"
        return self.cluster_effectiveness.get(key)

    def get_type_chart_status(self):
        return {
            'clustering_runs': self.clustering_run_count,
            'move_clusters': len(self.move_type_clusters),
            'species_clusters': len(self.species_type_clusters),
            'effectiveness_entries': len(self.cluster_effectiveness),
            'moves_clustered': len(self.move_to_cluster),
            'species_clustered': len(self.species_to_cluster),
            'battles_since_clustering': self.battles_since_last_clustering,
            'track_b_loaded': self.type_data_loaded,
        }

    # =========================================================================
    # TYPE CHART PERSISTENCE
    # =========================================================================

    def load_type_clusters(self, filepath=None):
        filepath = filepath or TYPE_CLUSTERS_FILE
        try:
            if not Path(filepath).exists():
                print(f"  🧬 No type clusters file")
                return

            with open(filepath, 'r') as f:
                data = json.load(f)

            self.move_type_clusters = {int(k): v for k, v in data.get('move_type_clusters', {}).items()}
            self.species_type_clusters = {int(k): v for k, v in data.get('species_type_clusters', {}).items()}
            self.cluster_effectiveness = data.get('cluster_effectiveness', {})
            self.move_to_cluster = {int(k): int(v) for k, v in data.get('move_to_cluster', {}).items()}
            self.species_to_cluster = {int(k): int(v) for k, v in data.get('species_to_cluster', {}).items()}
            self.clustering_run_count = data.get('clustering_run_count', 0)
            self.type_clusters_dirty = False

            print(f"  🧬 Type clusters loaded: {len(self.move_type_clusters)} move clusters, "
                  f"{len(self.species_type_clusters)} species clusters, "
                  f"{len(self.cluster_effectiveness)} effectiveness entries")

        except Exception as e:
            print(f"  ⚠️ Error loading type clusters: {e}")

    def save_type_clusters(self, filepath=None):
        filepath = filepath or TYPE_CLUSTERS_FILE
        try:
            data = {
                'move_type_clusters': {str(k): v for k, v in self.move_type_clusters.items()},
                'species_type_clusters': {str(k): v for k, v in self.species_type_clusters.items()},
                'cluster_effectiveness': self.cluster_effectiveness,
                'move_to_cluster': {str(k): v for k, v in self.move_to_cluster.items()},
                'species_to_cluster': {str(k): v for k, v in self.species_to_cluster.items()},
                'clustering_run_count': self.clustering_run_count,
                'last_clustering_timestep': self.last_clustering_timestep,
            }
            with open(filepath, 'w') as f:
                json.dump(data, f, indent=2)
            self.type_clusters_dirty = False
        except Exception as e:
            print(f"  ⚠️ Error saving type clusters: {e}")

    def load_ground_truth_types(self, filepath=None):
        self.type_data = load_type_data(filepath)
        self.type_data_loaded = self.type_data is not None and self.type_data.get('loaded', False)

    # =========================================================================
    # MENU DATA
    # =========================================================================

    def update_menu_data(self, menu_data):
        self.prev_menu_data = self.menu_data.copy()
        self.menu_data = menu_data.copy()

    # =========================================================================
    # BAG DATA
    # =========================================================================

    def update_bag_data(self, bag_data):
        self.prev_bag_data = self.bag_data.copy()
        self.bag_data = {
            'pocket': bag_data.get('pocket', -1),
            'cursor': bag_data.get('cursor', -1),
            'active': bag_data.get('active', 0),
            'items': [item.copy() for item in bag_data.get('items', [])],
        }

    # =========================================================================
    # BAG THREAD
    # =========================================================================

    def open_bag_thread(self, context):
        self.bag_thread_active = True
        self.bag_thread_context = context
        self.bag_thread_entered_at = self.timestep
        self.bag_thread_action_count = 0
        self.bag_thread_last_action = None
        self.bag_action_history.clear()
        pk_names = {0:"Items",1:"KeyItems",2:"Pokeballs",3:"TMs",4:"Berries"}
        print(f"  {'🎒' if context=='overworld' else '🎒⚔️'} BAG OPEN: {context} | "
              f"{pk_names.get(self.bag_data.get('pocket',-1),'?')} items={len(self.bag_data.get('items',[]))}")

    def close_bag_thread(self, reason=""):
        if self.bag_thread_active:
            print(f"  🎒 BAG CLOSED: {reason} ({self.bag_thread_context} "
                  f"{self.timestep - self.bag_thread_entered_at}f {self.bag_thread_action_count}act)")
        self.bag_thread_active = False
        self.bag_thread_context = "none"
        self.bag_thread_entered_at = 0
        self.bag_thread_action_count = 0
        self.bag_thread_last_action = None

    def is_bag_thread_active(self):
        return self.bag_thread_active

    def set_bag_thread_last_action(self, action_name):
        self.bag_thread_last_action = action_name

    def update_bag_thread_state(self, context_state):
        in_battle = context_state[3] > 0.5
        gs = self.game_state_raw
        bgd, prev_bgd = self.bag_data, self.prev_bag_data

        if self.bag_thread_active:
            if self.timestep - self.bag_thread_entered_at > self.BAG_THREAD_TIMEOUT:
                self.close_bag_thread("timeout"); return
        if self.bag_thread_active:
            if self.bag_thread_context == "overworld":
                if gs != 14: self.close_bag_thread("gs_left_14"); return
            elif self.bag_thread_context == "battle":
                if not in_battle: self.close_bag_thread("battle_ended"); return
                if bgd.get('active', 0) == 0 and prev_bgd.get('active', 0) == 0:
                    self.close_bag_thread("bag_deactivated"); return
            return

        if self.party_menu_active: return
        if not in_battle and gs == 14: self.open_bag_thread("overworld"); return
        if in_battle and bgd.get('active', 0) == 1: self.open_bag_thread("battle"); return

    # =========================================================================
    # ITEM KNOWLEDGE
    # =========================================================================

    def get_item_knowledge(self, item_id):
        if item_id not in self.item_knowledge:
            self.item_knowledge[item_id] = {
                'uses': 0, 'category': 'unknown', 'confidence': 0.0,
                'avg_hp_restored': 0.0, 'total_hp_restored': 0,
                'status_cured': [], 'catch_attempts': 0,
                'catch_successes': 0, 'last_used_timestep': 0,
            }
        return self.item_knowledge[item_id]

    def get_item_at_cursor(self):
        items = self.bag_data.get('items', [])
        cursor = self.bag_data.get('cursor', -1)
        if 0 <= cursor < len(items): return items[cursor].get('id', -1)
        return -1

    def snapshot_party_for_item(self):
        return {
            'slots': [{'hp': s.get('hp',0), 'max_hp': s.get('max_hp',0), 'status': s.get('status',0)}
                       for s in self.party_data.get('slots', [])],
            'count': self.party_data.get('count', 0)
        }

    def start_item_observation(self, item_id, target_slot=-1):
        in_battle = self.battle_data.get('battle_cursor', -1) != -1
        self.pending_item_observation = {
            'item_id': item_id, 'item_pocket': self.bag_data.get('pocket', -1),
            'target_slot': target_slot, 'party_snapshot': self.snapshot_party_for_item(),
            'party_count': self.party_data.get('count', 0), 'in_battle': in_battle,
            'enemy_hp_before': self.battle_data.get('enemy_hp', -1) if in_battle else -1,
            'timestep': self.timestep, 'frames_waiting': 0,
        }
        print(f"  🎒📝 ITEM OBS: item {item_id} pocket {self.bag_data.get('pocket',-1)} slot {target_slot}")

    def check_item_observation(self):
        if self.pending_item_observation is None: return False
        obs = self.pending_item_observation
        obs['frames_waiting'] += 1
        if obs['frames_waiting'] > self.ITEM_OBSERVE_WAIT_FRAMES:
            self.pending_item_observation = None; return True

        snap = obs['party_snapshot']
        cur_slots = self.party_data.get('slots', [])
        cur_count = self.party_data.get('count', 0)

        hp_r, hp_t = 0, -1
        for i, (sn, cu) in enumerate(zip(snap['slots'], cur_slots)):
            d = cu.get('hp', 0) - sn['hp']
            if d > 0: hp_r, hp_t = d, i; break

        st_c, st_t = 0, -1
        for i, (sn, cu) in enumerate(zip(snap['slots'], cur_slots)):
            if sn['status'] != 0 and cu.get('status', 0) == 0:
                st_c, st_t = sn['status'], i; break

        caught = cur_count > obs['party_count'] and obs['party_count'] > 0
        if not (hp_r > 0 or st_c != 0 or caught): return False

        target = hp_t if hp_t >= 0 else (st_t if st_t >= 0 else obs['target_slot'])
        self.record_item_observation(obs['item_id'], hp_r, st_c, caught, target, obs['in_battle'])
        self.pending_item_observation = None
        return True

    def record_item_observation(self, item_id, hp_restored=0, status_cured=0,
                                 caught=False, target_slot=-1, in_battle=False):
        if item_id <= 0: return
        ik = self.get_item_knowledge(item_id)
        ik['uses'] += 1; ik['last_used_timestep'] = self.timestep
        if hp_restored > 0:
            ik['total_hp_restored'] += hp_restored
            ik['avg_hp_restored'] = ik['total_hp_restored'] / max(1, ik['uses'])
        if status_cured != 0 and status_cured not in ik['status_cured']:
            ik['status_cured'].append(status_cured)
        if in_battle:
            ik['catch_attempts'] += 1
            if caught: ik['catch_successes'] += 1
        ik['category'] = self._categorize_item(ik)
        ik['confidence'] = min(1.0, ik['uses'] / 5.0)
        self.item_knowledge_dirty = True
        fx = []
        if hp_restored > 0: fx.append(f"+{hp_restored}HP")
        if status_cured != 0: fx.append(f"cured {status_cured}")
        if caught: fx.append("CAUGHT!")
        print(f"  🎒📖 ITEM: {item_id} → {ik['category']} ({ik['confidence']:.0%}) | {', '.join(fx) if fx else 'none'}")

    def _categorize_item(self, ik):
        if ik['uses'] == 0: return 'unknown'
        hp = ik['total_hp_restored'] > 0
        st = len(ik['status_cured']) > 0
        ca = ik['catch_successes'] > 0 or ik['catch_attempts'] >= 2
        if ca: return 'catch'
        if hp and st: return 'heal_both'
        if hp: return 'heal_hp'
        if st: return 'heal_status'
        if ik['uses'] >= 3: return 'other'
        return 'unknown'

    def get_party_hp_ratios(self):
        return [s.get('hp',0)/s.get('max_hp',1) if s.get('max_hp',0) > 0 else 0.0
                for s in self.party_data.get('slots', [])]

    def get_party_status_flags(self):
        return [s.get('status', 0) for s in self.party_data.get('slots', [])]

    def get_lowest_hp_ratio(self):
        living = [r for r in self.get_party_hp_ratios() if r > 0.0]
        return min(living) if living else 0.0

    def has_status_condition_in_party(self):
        return any(s.get('hp',0) > 0 and s.get('status',0) != 0 for s in self.party_data.get('slots', []))

    def get_healing_items(self):
        return [(it.get('id',0), it.get('q',0), self.item_knowledge[it['id']]['category'],
                 self.item_knowledge[it['id']]['confidence'])
                for it in self.bag_data.get('items', [])
                if it.get('id',0) > 0 and it.get('q',0) > 0 and
                it['id'] in self.item_knowledge and
                self.item_knowledge[it['id']]['category'] in ('heal_hp','heal_status','heal_both')]

    def get_catch_items(self):
        return [(it.get('id',0), it.get('q',0), self.item_knowledge[it['id']]['confidence'])
                for it in self.bag_data.get('items', [])
                if it.get('id',0) > 0 and it.get('q',0) > 0 and
                it['id'] in self.item_knowledge and
                self.item_knowledge[it['id']]['category'] == 'catch']

    # =========================================================================
    # ITEM KNOWLEDGE PERSISTENCE
    # =========================================================================

    def load_item_knowledge(self, filepath=None):
        filepath = filepath or ITEM_KNOWLEDGE_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    self.item_knowledge = {int(k): v for k, v in json.load(f).items()}
                self.item_knowledge_dirty = False
                known = sum(1 for v in self.item_knowledge.values() if v.get('category','unknown') != 'unknown')
                print(f"  🎒📖 Items loaded: {len(self.item_knowledge)} ({known} categorized)")
            else:
                print(f"  🎒📖 No item knowledge file")
        except Exception as e:
            print(f"  ⚠️ Item knowledge error: {e}")
            self.item_knowledge = {}

    def save_item_knowledge(self, filepath=None):
        filepath = filepath or ITEM_KNOWLEDGE_FILE
        try:
            with open(filepath, 'w') as f:
                json.dump({str(k): v for k, v in self.item_knowledge.items()}, f, indent=2)
            self.item_knowledge_dirty = False
        except Exception as e:
            print(f"  ⚠️ Item save error: {e}")

    # =========================================================================
    # PARTY DATA
    # =========================================================================

    def update_party_data(self, party_data):
        self.prev_party_data = self.party_data.copy()
        self.party_data = party_data.copy()

    def get_party_slot_fingerprint(self, slot_index):
        slots = self.party_data.get('slots', [])
        if slot_index < 0 or slot_index >= len(slots): return None
        s = slots[slot_index]
        return (s['atk'], s['def'], s['spd'], s['spatk'], s['spdef'])

    def get_living_party_slots(self):
        return [(i, s) for i, s in enumerate(self.party_data.get('slots', []))
                if s.get('hp', 0) > 0 and s.get('max_hp', 0) > 0]

    def get_active_slot_index(self):
        bd = self.battle_data
        if bd['player_hp'] <= 0 and bd['player_max_hp'] <= 0: return -1
        for i, s in enumerate(self.party_data.get('slots', [])):
            if (s.get('hp',-1) == bd['player_hp'] and s.get('max_hp',-1) == bd['player_max_hp']
                and s.get('level',-1) == bd['player_level']):
                return i
        return -1

    def get_team_avg_level(self):
        """Get average level of living party members."""
        levels = [s.get('level', 0) for s in self.party_data.get('slots', [])
                  if s.get('hp', 0) > 0]
        return np.mean(levels) if levels else 0.0

    # =========================================================================
    # BATTLE DATA
    # =========================================================================

    def update_battle_data(self, battle_data):
        self.prev_battle_data = self.battle_data.copy()
        for k in ('player_stat_stages', 'enemy_stat_stages'):
            v = self.battle_data.get(k)
            if isinstance(v, list): self.prev_battle_data[k] = list(v)
        self.battle_data = battle_data.copy()
        for k in ('player_stat_stages', 'enemy_stat_stages'):
            v = battle_data.get(k)
            if isinstance(v, list): self.battle_data[k] = list(v)

    def on_battle_start_with_data(self):
        bd = self.battle_data
        if bd['player_species'] > 0: self.battle_player_species = bd['player_species']
        if bd['enemy_species'] > 0: self.battle_enemy_species = bd['enemy_species']
        if bd['player_hp'] > 0: self.battle_start_hp = bd['player_hp']
        if bd['enemy_hp'] > 0: self.battle_enemy_start_hp = bd['enemy_hp']
        self.battle_menu_state = "unknown"; self.battle_cursor_action_count = 0
        self.turn_count = 0
        self.turn_start_player_hp = bd['player_hp']; self.turn_start_enemy_hp = bd['enemy_hp']
        self.turn_start_pp = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
        self.turn_start_enemy_pp = [bd['enemy_pp0'], bd['enemy_pp1'], bd['enemy_pp2'], bd['enemy_pp3']]
        self.turn_start_player_stats = list(bd.get('player_stat_stages', [-1]*7))
        self.turn_start_enemy_stats = list(bd.get('enemy_stat_stages', [-1]*7))
        self.turn_start_player_status = bd.get('player_status', 0)
        self.turn_start_enemy_status = bd.get('enemy_status', 0)
        self.last_move_used = -1; self.last_move_slot = -1
        self.battle_no_damage_turns = 0; self.battle_hp_trend.clear()
        self.close_party_menu("battle_start"); self.close_bag_thread("battle_start")
        self.abort_preparation("battle_start")
        self.forced_switch_pending = False; self.forced_switch_target_slot = -1
        self.update_roster_from_battle()

    def on_battle_end_with_data(self):
        outcome = 'unknown'
        if self.battle_start_hp > 0:
            lp = self.prev_battle_data.get('player_hp', -1)
            le = self.prev_battle_data.get('enemy_hp', -1)
            if le == 0: outcome = 'win'
            elif lp == 0: outcome = 'loss'
            elif lp > 0 and le > 0: outcome = 'run'

        pc, cc = self.prev_party_data.get('count', 0), self.party_data.get('count', 0)
        if cc > pc and pc > 0: outcome = 'catch'; print(f"  🎊 CATCH! {pc}→{cc}")

        if self.battle_start_hp > 0:
            lp = self.prev_battle_data.get('player_hp', -1)
            if lp > 0:
                if lp / self.battle_start_hp < 0.3: self.battle_low_hp_exits += 1
                else: self.battle_low_hp_exits = max(0, self.battle_low_hp_exits - 1)

        if self.current_map_id is not None and self.battle_start_hp > 0:
            lp = self.prev_battle_data.get('player_hp', -1)
            pmhp = self.battle_data.get('player_max_hp', self.battle_start_hp)
            hp_cost = max(0.0, (self.battle_start_hp - lp) / pmhp) if lp >= 0 and pmhp > 0 else 0.0
            self.update_map_battle_stats(self.current_map_id, self.battle_enemy_species,
                                          self.prev_battle_data.get('enemy_level', -1),
                                          hp_cost, self.is_trainer_battle(), outcome)

        # Revenge module integration
        if outcome == 'loss' and self.current_map_id is not None:
            enemy_level = self.prev_battle_data.get('enemy_level', -1)
            if enemy_level > 0:
                pos = self.last_positions[-1] if self.last_positions else (0, 0)
                self.record_revenge_loss(
                    map_id=self.current_map_id,
                    position=pos,
                    enemy_species=[self.battle_enemy_species] if self.battle_enemy_species > 0 else [],
                    enemy_avg_level=float(enemy_level),
                    my_avg_level=self.get_team_avg_level(),
                    is_trainer=self.is_trainer_battle(),
                )
        elif outcome == 'win' and self.current_map_id is not None:
            pos = self.last_positions[-1] if self.last_positions else (0, 0)
            self.record_revenge_victory(self.current_map_id, pos)

        self.battles_since_last_clustering += 1

        self.battle_player_species = -1; self.battle_enemy_species = -1
        self.battle_start_hp = -1; self.battle_enemy_start_hp = -1
        self.battle_menu_state = "unknown"; self.battle_cursor_action_count = 0
        self.turn_count = 0
        self.turn_start_player_hp = -1; self.turn_start_enemy_hp = -1
        self.turn_start_pp = [-1]*4; self.turn_start_enemy_pp = [-1]*4
        self.turn_start_player_stats = [-1]*7; self.turn_start_enemy_stats = [-1]*7
        self.turn_start_player_status = 0; self.turn_start_enemy_status = 0
        self.last_move_used = -1; self.last_move_slot = -1; self.battle_no_damage_turns = 0
        self.close_party_menu("battle_end"); self.close_bag_thread("battle_end")
        self.forced_switch_pending = False; self.forced_switch_target_slot = -1
        return outcome

    def has_battle_data(self):
        return self.battle_data.get('battle_cursor', -1) != -1 or self.battle_data.get('player_hp', -1) != -1

    def is_trainer_battle(self):
        return (self.battle_data.get('battle_type', 0) & 8) != 0

    def infer_battle_menu_state(self):
        bd, prev = self.battle_data, self.prev_battle_data
        if not self.has_battle_data() or self.party_menu_active or self.bag_thread_active: return "unknown"
        bc, mc = bd['battle_cursor'], bd['move_cursor']
        pbc, pmc = prev['battle_cursor'], prev['move_cursor']
        if 0 <= bc <= 3 and bc != pbc: return "main_menu"
        if 0 <= mc <= 3 and mc != pmc: return "move_select"
        if 0 <= bc <= 3: return "main_menu"
        return "unknown"

    # =========================================================================
    # PARTY MENU THREAD
    # =========================================================================

    def open_party_menu(self, context, target_slot=-1):
        self.party_menu_active = True; self.party_menu_context = context
        self.party_menu_entered_at = self.timestep; self.party_menu_target_slot = target_slot
        self.party_menu_action_count = 0; self.party_menu_last_action = None
        if context == "battle_forced":
            self.forced_switch_pending = True; self.forced_switch_target_slot = target_slot
        emoji = {"battle_voluntary":"🔄","battle_forced":"🔄⚠️","overworld":"📋"}.get(context,"❓")
        print(f"  {emoji} PARTY MENU: {context}{f' →slot{target_slot}' if target_slot>=0 else ''}")

    def close_party_menu(self, reason=""):
        if self.party_menu_active:
            print(f"  📋 PARTY CLOSED: {reason} ({self.party_menu_context} "
                  f"{self.timestep-self.party_menu_entered_at}f {self.party_menu_action_count}act)")
        self.party_menu_active = False; self.party_menu_context = "none"
        self.party_menu_entered_at = 0; self.party_menu_target_slot = -1
        self.party_menu_action_count = 0; self.party_menu_awaiting_entry = False
        self.party_menu_battle_cursor_on_entry = -1; self.party_menu_last_action = None

    def is_party_menu_active(self): return self.party_menu_active

    def update_party_menu_state(self, context_state):
        in_battle = context_state[3] > 0.5
        bd, prev_bd = self.battle_data, self.prev_battle_data
        md, gs = self.menu_data, self.game_state_raw

        if self.party_menu_active:
            if self.timestep - self.party_menu_entered_at > self.PARTY_MENU_TIMEOUT:
                self.close_party_menu("timeout"); return
        if self.party_menu_active:
            if in_battle:
                bc = bd.get('battle_cursor', -1)
                if 0 <= bc <= 3 and 0 <= prev_bd.get('battle_cursor', -1) <= 3:
                    self.close_party_menu("returned_to_battle"); return
            else:
                if self.party_menu_context in ("battle_voluntary","battle_forced"):
                    self.close_party_menu("battle_ended"); return
                if self.party_menu_context == "overworld":
                    if gs != 1: self.close_party_menu("gs_changed"); return
                    if not (0 <= md.get('pc',-1) <= 6): self.close_party_menu("pc_invalid"); return
                if context_state[4] <= 0.5 and self.party_menu_context == "overworld":
                    self.close_party_menu("menu_closed"); return
            return

        if in_battle and self.has_battle_data():
            bc = bd.get('battle_cursor', -1)
            pbc = prev_bd.get('battle_cursor', -1)
            if bd['player_hp'] == 0 and prev_bd.get('player_hp', -1) > 0:
                self.open_party_menu("battle_forced", self.get_best_switch_slot()); return
            if self.party_menu_awaiting_entry:
                self.party_menu_awaiting_entry = False
                if not (0 <= bc <= 3): self.open_party_menu("battle_voluntary")
                return
            if self.party_menu_last_action == "A" and pbc == 2 and not (0 <= bc <= 3):
                self.open_party_menu("battle_voluntary"); return

        if not in_battle:
            pc = md.get('pc', -1)
            ppc = self.prev_menu_data.get('pc', -1)
            if gs == 1 and 0 <= pc <= 5:
                if not (0 <= ppc <= 5):
                    self.open_party_menu("overworld"); return

    def set_party_menu_last_action(self, action_name):
        self.party_menu_last_action = action_name

    # =========================================================================
    # FORCED SWITCH + RUNNING
    # =========================================================================

    def detect_forced_switch(self):
        return self.party_menu_active and self.party_menu_context == "battle_forced"

    def get_best_switch_slot(self):
        living = self.get_living_party_slots()
        if not living: return -1
        active = self.get_active_slot_index()
        cands = [(i, s) for i, s in living if i != active]
        if not cands: return living[0][0] if living else -1
        return max(cands, key=lambda x: x[1].get('hp', 0))[0]

    def should_run(self):
        if not self.BATTLE_RUN_ENABLED or self.is_trainer_battle(): return False
        bd = self.battle_data
        if self.battle_no_damage_turns >= self.BATTLE_RUN_NO_DAMAGE_THRESHOLD: return True
        if bd['player_hp'] > 0 and bd['player_max_hp'] > 0:
            if bd['player_hp'] / bd['player_max_hp'] < self.BATTLE_RUN_HP_RATIO_THRESHOLD: return True
        if len(self.battle_hp_trend) >= 3 and self.battle_no_damage_turns >= 2:
            t = list(self.battle_hp_trend)
            if all(t[i] >= t[i+1] for i in range(len(t)-1)): return True
        if self.battle_low_hp_exits >= 3: return True
        return False

    # =========================================================================
    # TURN TRACKING
    # =========================================================================

    def detect_turn_resolved(self):
        bd = self.battle_data
        if bd['player_hp'] < 0 or self.turn_start_player_hp < 0: return False
        cpp = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
        for i in range(4):
            if self.turn_start_pp[i] > 0 and cpp[i] >= 0 and cpp[i] < self.turn_start_pp[i]: return True
        phc = bd['player_hp'] != self.turn_start_player_hp and self.turn_start_player_hp >= 0
        ehc = bd['enemy_hp'] != self.turn_start_enemy_hp and self.turn_start_enemy_hp >= 0
        return phc and ehc

    def on_battle_turn_end(self):
        bd = self.battle_data
        self.turn_count += 1
        cpp = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
        used_slot = -1
        for i in range(4):
            if self.turn_start_pp[i] > 0 and cpp[i] >= 0 and cpp[i] < self.turn_start_pp[i]:
                used_slot = i; break
        move_id = bd.get(['move0','move1','move2','move3'][used_slot], -1) if used_slot >= 0 else -1
        dmg = max(0, self.turn_start_enemy_hp - bd['enemy_hp']) if self.turn_start_enemy_hp >= 0 and bd['enemy_hp'] >= 0 else 0

        missed = False
        if used_slot >= 0 and dmg == 0:
            esc = any(self.turn_start_enemy_stats[j] >= 0 and bd.get('enemy_stat_stages',[-1]*7)[j] >= 0
                      and bd.get('enemy_stat_stages',[-1]*7)[j] != self.turn_start_enemy_stats[j] for j in range(7))
            si = bd.get('enemy_status',0) != self.turn_start_enemy_status and bd.get('enemy_status',0) != 0
            psc = any(self.turn_start_player_stats[j] >= 0 and bd.get('player_stat_stages',[-1]*7)[j] >= 0
                      and bd.get('player_stat_stages',[-1]*7)[j] != self.turn_start_player_stats[j] for j in range(7))
            if not esc and not si and not psc: missed = True

        if move_id > 0 and self.battle_enemy_species > 0:
            cp = bd.get('player_stat_stages', [-1]*7)
            ce = bd.get('enemy_stat_stages', [-1]*7)
            ssc = sum(1 for j in range(7) if self.turn_start_player_stats[j]>=0 and cp[j]>=0 and cp[j]!=self.turn_start_player_stats[j])
            esc = sum(1 for j in range(7) if self.turn_start_enemy_stats[j]>=0 and ce[j]>=0 and ce[j]!=self.turn_start_enemy_stats[j])
            si = 1 if (bd.get('enemy_status',0) != self.turn_start_enemy_status and bd.get('enemy_status',0) != 0) else 0
            self.record_move_result(move_id, self.battle_enemy_species, dmg, missed, ssc, esc, si)

        em = self.detect_enemy_move()
        if em > 0:
            dt = max(0, self.turn_start_player_hp - bd['player_hp']) if self.turn_start_player_hp >= 0 and bd['player_hp'] >= 0 else 0
            ps = 1 if (bd.get('player_status',0) != self.turn_start_player_status and bd.get('player_status',0) != 0) else 0
            self.record_enemy_move_observation(em, dt, ps, 0)

        if dmg == 0 and used_slot >= 0: self.battle_no_damage_turns += 1
        else: self.battle_no_damage_turns = 0
        if bd['player_hp'] > 0: self.battle_hp_trend.append(bd['player_hp'])

        if self.battle_enemy_species > 0 and bd['enemy_species'] > 0 and bd['enemy_species'] != self.battle_enemy_species:
            print(f"  ⚔️ ENEMY SWITCH: {self.battle_enemy_species}→{bd['enemy_species']}")
            self.battle_enemy_species = bd['enemy_species']; self.battle_enemy_start_hp = bd['enemy_hp']

        self.turn_start_player_hp = bd['player_hp']; self.turn_start_enemy_hp = bd['enemy_hp']
        self.turn_start_pp = [bd['pp0'],bd['pp1'],bd['pp2'],bd['pp3']]
        self.turn_start_enemy_pp = [bd['enemy_pp0'],bd['enemy_pp1'],bd['enemy_pp2'],bd['enemy_pp3']]
        self.turn_start_player_stats = list(bd.get('player_stat_stages',[-1]*7))
        self.turn_start_enemy_stats = list(bd.get('enemy_stat_stages',[-1]*7))
        self.turn_start_player_status = bd.get('player_status', 0)
        self.turn_start_enemy_status = bd.get('enemy_status', 0)

    def detect_enemy_move(self):
        bd = self.battle_data
        epp = [bd['enemy_pp0'],bd['enemy_pp1'],bd['enemy_pp2'],bd['enemy_pp3']]
        for i in range(4):
            if self.turn_start_enemy_pp[i] > 0 and epp[i] >= 0 and epp[i] < self.turn_start_enemy_pp[i]:
                return bd.get(['enemy_move0','enemy_move1','enemy_move2','enemy_move3'][i], -1)
        return -1

    # =========================================================================
    # ROSTER
    # =========================================================================

    def update_roster_from_battle(self):
        bd = self.battle_data
        if bd['player_species'] <= 0: return
        active = self.get_active_slot_index()
        if active < 0:
            for i, s in enumerate(self.party_data.get('slots', [])):
                if s.get('level',-1) == bd.get('player_level',-1) and s.get('max_hp',-1) == bd['player_max_hp']:
                    active = i; break
        if active < 0: return
        self.roster[active] = {
            'species': bd['player_species'],
            'moves': [bd[f'move{i}'] for i in range(4) if bd.get(f'move{i}',-1) > 0],
            'fingerprint': self.get_party_slot_fingerprint(active),
            'level': bd.get('player_level', -1), 'last_updated': self.timestep
        }
        self.roster_dirty = True

    def get_roster_moves_for_slot(self, si):
        e = self.roster.get(si); return e.get('moves', []) if e else []

    def get_move_slot_index(self, mid):
        for i in range(4):
            if self.battle_data.get(f'move{i}', -1) == mid: return i
        return -1

    # =========================================================================
    # MOVE KNOWLEDGE
    # =========================================================================

    def record_move_result(self, move_id, enemy_species, damage, missed, ssc, esc, si):
        if move_id <= 0: return
        mk = self.move_knowledge
        if move_id not in mk:
            mk[move_id] = {'total_uses':0,'total_damage':0,'avg_damage':0.0,'misses':0,'vs_species':{}}
        e = mk[move_id]; e['total_uses'] += 1; e['total_damage'] += damage
        e['avg_damage'] = e['total_damage']/max(1,e['total_uses'])
        if missed: e['misses'] += 1
        if enemy_species not in e['vs_species']:
            e['vs_species'][enemy_species] = {'uses':0,'damage':0,'avg':0.0,'misses':0,
                                               'stat_changes_self':0,'stat_changes_enemy':0,'status_inflicted':0}
        v = e['vs_species'][enemy_species]; v['uses'] += 1; v['damage'] += damage
        v['avg'] = v['damage']/max(1,v['uses'])
        if missed: v['misses'] += 1
        v['stat_changes_self'] += ssc; v['stat_changes_enemy'] += esc; v['status_inflicted'] += si
        self.move_knowledge_dirty = True

    def record_enemy_move_observation(self, emid, dtu, si, sc):
        if emid <= 0: return
        emk = self.enemy_move_knowledge
        if emid not in emk: emk[emid] = {'observed_uses':0,'damage_to_us':0,'status_inflicted':0,'stat_changes':0}
        e = emk[emid]; e['observed_uses'] += 1; e['damage_to_us'] += dtu; e['status_inflicted'] += si; e['stat_changes'] += sc

    def get_best_move_for_enemy(self, es):
        """
        Get ranked moves for battling enemy species.

        5-tier scoring (NEW Tier 0: pipeline):
        0. Battle pipeline action_selection pool (when authority sufficient)
        1. Direct vs_species knowledge (highest confidence)
        2. Track B ground-truth type effectiveness (if available)
        3. Empirical cluster effectiveness (Track A)
        4. Fallback: overall avg_damage per move
        """
        bd = self.battle_data
        mids = [bd[f'move{i}'] for i in range(4)]
        pps = [bd[f'pp{i}'] for i in range(4)]

        # === Tier 0: Pipeline query (NEW) ===
        # If battle pipeline has learned enough, use its move preferences
        battle_state = build_learning_state_battle(bd, self.party_data, self.turn_count)
        pipeline_scores = self.query_battle_pipeline_move_scores(battle_state)

        if pipeline_scores is not None:
            # Pipeline returned scores — use them as primary ranking
            # but still filter by PP availability
            pipeline_cands = []
            for slot, score in pipeline_scores:
                mid = mids[slot] if slot < 4 else -1
                pp = pps[slot] if slot < 4 else 0
                if mid <= 0 or pp == 0:
                    continue
                # Scale pipeline score to be comparable with knowledge scores
                scaled = score * 50.0  # pipeline outputs ~0-1, knowledge scores ~0-50+
                if pp <= 3 and pp > 0:
                    scaled *= 0.8
                pipeline_cands.append((mid, slot, scaled))

            if pipeline_cands:
                # Blend with knowledge fallback: use pipeline ranking but
                # boost candidates that also have good knowledge scores
                knowledge_cands = self._get_knowledge_move_scores(es, mids, pps)
                knowledge_map = {mid: sc for mid, si, sc in knowledge_cands}

                for i, (mid, slot, p_score) in enumerate(pipeline_cands):
                    k_score = knowledge_map.get(mid, 1.0)
                    # Weighted blend: 60% pipeline, 40% knowledge
                    pipeline_cands[i] = (mid, slot, p_score * 0.6 + k_score * 0.4)

                pipeline_cands.sort(key=lambda x: x[2], reverse=True)
                return pipeline_cands

        # === Tiers 1-4: Knowledge fallback (existing logic) ===
        return self._get_knowledge_move_scores(es, mids, pps)

    def _get_knowledge_move_scores(self, es, mids, pps):
        """
        Original knowledge-based move scoring (Tiers 1-4).
        Extracted from get_best_move_for_enemy for reuse.
        """
        bd = self.battle_data
        cands = []
        for si in range(4):
            mid, pp = mids[si], pps[si]
            if mid <= 0 or pp == 0: continue
            sc = 1.0
            source = 'default'

            mk = self.move_knowledge.get(mid)
            if mk:
                vs = mk.get('vs_species',{}).get(es)

                # Tier 1: Direct per-species knowledge
                if vs and vs['uses'] >= 2:
                    sc = vs['avg'] * 10.0
                    if vs['uses'] > 0: sc *= (1.0 - vs['misses']/vs['uses']*0.5)
                    if vs['status_inflicted'] > 0: sc += 2.0
                    if vs['stat_changes_enemy'] > 0: sc += 1.0
                    source = 'direct'
                else:
                    # Tier 2: Track B ground-truth type effectiveness
                    if self.type_data_loaded:
                        type_mult = get_type_effectiveness(self.type_data, mid, es)
                        if type_mult is not None:
                            base = mk['avg_damage'] if mk['total_uses'] >= 2 else 5.0
                            sc = base * type_mult * 8.0
                            if mk['total_uses'] > 0:
                                sc *= (1.0 - mk['misses']/mk['total_uses']*0.5)
                            source = 'type_b'

                    # Tier 3: Empirical cluster effectiveness (Track A)
                    if source == 'default':
                        cluster_eff = self.get_cluster_effectiveness_for_move(mid, es)
                        if cluster_eff is not None:
                            base = mk['avg_damage'] if mk['total_uses'] >= 2 else 5.0
                            mult = cluster_eff / 0.5 if cluster_eff > 0 else 0.1
                            sc = base * mult * 8.0
                            if mk['total_uses'] > 0:
                                sc *= (1.0 - mk['misses']/mk['total_uses']*0.5)
                            source = 'cluster'

                    # Tier 4: Overall avg_damage fallback
                    if source == 'default' and mk['total_uses'] >= 2:
                        sc = mk['avg_damage'] * 8.0
                        if mk['total_uses'] > 0:
                            sc *= (1.0 - mk['misses']/mk['total_uses']*0.5)
                        source = 'avg'

            if pp <= 3 and pp > 0: sc *= 0.8
            cands.append((mid, si, sc))
        cands.sort(key=lambda x: x[2], reverse=True)
        return cands

    def get_moves_with_pp(self):
        bd = self.battle_data
        return [(i, bd.get(f'move{i}',-1), bd.get(f'pp{i}',-1))
                for i in range(4) if bd.get(f'move{i}',-1) > 0 and bd.get(f'pp{i}',-1) > 0]

    # =========================================================================
    # PERSISTENCE
    # =========================================================================

    def load_roster(self, filepath=None):
        filepath = filepath or ROSTER_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    self.roster = {int(k): v for k, v in json.load(f).items()}
                for v in self.roster.values():
                    if isinstance(v.get('fingerprint'), list): v['fingerprint'] = tuple(v['fingerprint'])
                self.roster_dirty = False
                print(f"  📋 Roster: {len(self.roster)} slots")
            else: print(f"  📋 No roster file")
        except Exception as e: print(f"  ⚠️ Roster error: {e}"); self.roster = {}

    def save_roster(self, filepath=None):
        filepath = filepath or ROSTER_FILE
        try:
            sd = {}
            for k, v in self.roster.items():
                e = v.copy()
                if isinstance(e.get('fingerprint'), tuple): e['fingerprint'] = list(e['fingerprint'])
                sd[str(k)] = e
            with open(filepath, 'w') as f: json.dump(sd, f, indent=2)
            self.roster_dirty = False
        except Exception as e: print(f"  ⚠️ Roster save error: {e}")

    def load_move_knowledge(self, filepath=None):
        filepath = filepath or MOVE_KNOWLEDGE_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f: data = json.load(f)
                self.move_knowledge = {}
                for mk, mv in data.get('player_moves', {}).items():
                    mv['vs_species'] = {int(sk): sv for sk, sv in mv.get('vs_species', {}).items()}
                    self.move_knowledge[int(mk)] = mv
                self.enemy_move_knowledge = {int(k): v for k, v in data.get('enemy_moves', {}).items()}
                self.move_knowledge_dirty = False
                print(f"  📖 Moves: {len(self.move_knowledge)} tracked")
            else: print(f"  📖 No move knowledge file")
        except Exception as e: print(f"  ⚠️ Move knowledge error: {e}"); self.move_knowledge = {}; self.enemy_move_knowledge = {}

    def save_move_knowledge(self, filepath=None):
        filepath = filepath or MOVE_KNOWLEDGE_FILE
        try:
            sd = {'player_moves': {}, 'enemy_moves': {str(k): v for k, v in self.enemy_move_knowledge.items()}}
            for mk, mv in self.move_knowledge.items():
                c = mv.copy(); c['vs_species'] = {str(sk): sv for sk, sv in mv.get('vs_species', {}).items()}
                sd['player_moves'][str(mk)] = c
            with open(filepath, 'w') as f: json.dump(sd, f, indent=2)
            self.move_knowledge_dirty = False
        except Exception as e: print(f"  ⚠️ Move save error: {e}")

    def load_map_battle_stats(self, data=None):
        if data is None: return
        try:
            self.map_battle_stats = {int(k): v for k, v in data.items()}
            self.map_step_counters = {mid: s.get('total_steps_on_map', 0) for mid, s in self.map_battle_stats.items()}
            self.map_battle_stats_dirty = False
            mwd = len([s for s in self.map_battle_stats.values() if s['battles_fought'] > 0])
            print(f"  📊 Map battle stats: {mwd} maps")
        except Exception as e: print(f"  ⚠️ Map stats error: {e}"); self.map_battle_stats = {}

    def get_map_battle_stats_for_save(self):
        return {str(k): v for k, v in self.map_battle_stats.items()}

    # =========================================================================
    # END OF PART C — Paste PART A + PART B + PART C as Cell 3 Part 1
    # =========================================================================

# ============================================================================
# CELL 3 PART 2: Taught Reference, Blend System, Markov (OW + Battle + Bag + Start Menu)
# ============================================================================
# CHANGES from v17.1:
# 1. NEW: load_taught_start_menu_transitions() — loads start menu demonstrations
# 2. NEW: compute_start_menu_markov_similarity() — start menu matching
# 3. NEW: get_start_menu_markov_action() — best start menu action from Markov
# 4. All overworld + battle + bag Markov methods UNCHANGED
# 5. All blend system methods UNCHANGED
# ============================================================================

    # =========================================================================
    # TAUGHT MODEL REFERENCE
    # =========================================================================
    
    def load_taught_reference(self, filepath):
        try:
            if not Path(filepath).exists():
                print(f"  No taught reference model found at {filepath}")
                return
            
            with open(filepath, 'r') as f:
                model = json.load(f)
            
            if "perceptrons" not in model:
                print(f"  ⚠️ Taught reference model empty or invalid")
                return
            
            for saved_action in model["perceptrons"].get("actions", []):
                action_name = saved_action.get("action")
                if action_name:
                    self.taught_reference['utilities'][action_name] = saved_action.get("utility", 1.0)
                    
                    if saved_action.get("weights_nonzero"):
                        dim = saved_action.get("weights_shape", 1376)
                        w = np.zeros(dim)
                        for idx, val in saved_action["weights_nonzero"]:
                            if idx < dim:
                                w[idx] = val
                        self.taught_reference['weights'][action_name] = w
            
            self.taught_reference['loaded'] = True
            
            print(f"  📖 Taught reference loaded:")
            print(f"     Actions: {list(self.taught_reference['utilities'].keys())}")
            print(f"     Utilities: {', '.join(f'{k}:{v:.3f}' for k, v in self.taught_reference['utilities'].items())}")
            print(f"     Weights available: {list(self.taught_reference['weights'].keys())}")
            
        except Exception as e:
            print(f"  ⚠️ Error loading taught reference: {e}")
    
    def blend_from_taught(self, tier):
        if not self.taught_reference['loaded']:
            return
        
        if tier not in self.BLEND_RATIOS:
            return
        
        if self.timestep - self.last_blend_timestep < self.BLEND_COOLDOWN:
            return
        
        ai_weight, taught_weight = self.BLEND_RATIOS[tier]
        blend_weights = (tier == 3)
        
        blended_actions = []
        
        for a in self.actions():
            if a.action not in self.taught_reference['utilities']:
                continue
            
            taught_util = self.taught_reference['utilities'][a.action]
            old_util = a.utility
            
            a.utility = ai_weight * a.utility + taught_weight * taught_util
            
            if taught_util > 1.0:
                a.utility = max(a.utility, taught_util * 0.5)
            
            floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
            a.utility = max(a.utility, floor)
            a.utility = min(a.utility, 2.0)
            
            blended_actions.append(f"{a.action}:{old_util:.3f}→{a.utility:.3f}")
            
            if blend_weights and a.action in self.taught_reference['weights']:
                taught_w = self.taught_reference['weights'][a.action]
                if a.weights is not None:
                    min_dim = min(len(a.weights), len(taught_w))
                    a.weights[:min_dim] = (
                        ai_weight * a.weights[:min_dim] + 
                        taught_weight * taught_w[:min_dim]
                    )
        
        self.last_blend_timestep = self.timestep
        self.blend_tier = tier
        self.blend_count += 1
        
        tier_names = {1: "LIGHT", 2: "MEDIUM", 3: "HARD"}
        print(f"  🔀 BLEND [{tier_names.get(tier, '?')}] ({ai_weight:.0%} AI / {taught_weight:.0%} taught)"
              f" | Blend #{self.blend_count}")
        for ba in blended_actions:
            print(f"     {ba}")
        if blend_weights:
            print(f"     + Weights blended for: {list(self.taught_reference['weights'].keys())}")

    # =========================================================================
    # OVERWORLD MARKOV TRANSITION SYSTEM
    # =========================================================================
    
    def load_taught_transitions(self, filepath=None):
        filepath = filepath or TAUGHT_TRANSITIONS_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    data = json.load(f)
                
                self.taught_transitions = []
                self.taught_batches = data.get('batches', [])
                
                for batch in self.taught_batches:
                    batch_type = batch.get('batch_type', 'steady')
                    trigger_action = batch.get('trigger_action')
                    
                    for frame in batch.get('frames', []):
                        transition = {
                            'state': frame.get('state', {}),
                            'action': frame.get('action'),
                            'recent_actions': frame.get('recent_actions', []),
                            'frame_offset': frame.get('frame_offset', 0),
                            'batch_type': batch_type,
                            'trigger_action': trigger_action
                        }
                        self.taught_transitions.append(transition)
                
                self.taught_metadata = data.get('metadata', {})
                
                print(f"  📚 Loaded taught transitions:")
                print(f"     Batches: {len(self.taught_batches)}")
                print(f"     Frames: {len(self.taught_transitions)}")
                print(f"     Action changes: {self.taught_metadata.get('action_changes', 0)}")
                print(f"     Maps visited: {self.taught_metadata.get('maps_visited', [])}")
            else:
                self.taught_transitions = []
                self.taught_batches = []
                self.taught_metadata = {}
                print(f"  No taught transitions file found at {filepath}")
        except Exception as e:
            print(f"  Error loading taught transitions: {e}")
            self.taught_transitions = []
            self.taught_batches = []
            self.taught_metadata = {}
    
    def extract_partial_context(self, context_state, raw_position=None):
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_map = int(context_state[2])
        
        movement_blocked = self.get_position_stagnation() > 3
        
        near_transition = False
        memory = self.get_current_map_memory(current_map)
        for t in memory.get('transitions', []):
            t_pos = tuple(t['position']) if isinstance(t['position'], list) else t['position']
            if abs(raw_x - t_pos[0]) + abs(raw_y - t_pos[1]) <= 2:
                near_transition = True
                break
        
        tile_probed = not self.should_interact_at_tile(raw_x, raw_y, current_map)
        
        return {
            'in_battle': context_state[3] > 0.5,
            'in_menu': context_state[4] > 0.5,
            'movement_blocked': movement_blocked,
            'near_transition': near_transition,
            'tile_probed': tile_probed
        }
    
    def compute_markov_similarity(self, context_state, raw_position=None, taught_frames=None):
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        skip_map_check = taught_frames is not None
        
        if not frames:
            return 0.0, None, -1
        
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_map = int(context_state[2])
        current_dir = int(context_state[5])
        in_battle = context_state[3] > 0.5
        in_menu = context_state[4] > 0.5
        
        current_actions = list(self.action_history)
        current_partial = self.extract_partial_context(context_state, raw_position)
        
        best_score = 0.0
        best_action = None
        best_idx = -1
        
        for idx, transition in enumerate(frames):
            t_state = transition.get('state', {})
            t_action = transition.get('action')
            t_recent = transition.get('recent_actions', [])
            batch_type = transition.get('batch_type', 'steady')
            
            if not t_action or t_action == "NONE":
                continue
            
            immediate_score = 0.0
            
            if not skip_map_check:
                if t_state.get('map_id') != current_map:
                    continue
            immediate_score += 0.25
            
            t_x = t_state.get('x', 0)
            t_y = t_state.get('y', 0)
            pos_dist = abs(raw_x - t_x) + abs(raw_y - t_y)
            
            if pos_dist == 0:
                immediate_score += MARKOV_POS_EXACT_BONUS
            elif pos_dist <= 2:
                immediate_score += MARKOV_POS_NEAR_BONUS
            elif pos_dist <= MARKOV_POS_MAX_DIST:
                immediate_score += MARKOV_POS_FAR_BONUS
            else:
                continue
            
            if t_state.get('direction') == current_dir:
                immediate_score += 0.2
            
            t_in_battle = t_state.get('in_battle', 0) == 1
            t_in_menu = t_state.get('in_menu', 0) == 1
            
            if t_in_battle == in_battle:
                immediate_score += 0.1
            if t_in_menu == in_menu:
                immediate_score += 0.1
            
            sequential_score = 0.0
            
            if t_recent and current_actions:
                if len(current_actions) >= 8 and len(t_recent) >= 8:
                    if list(current_actions)[-8:] == t_recent[-8:]:
                        sequential_score = MARKOV_SEQ_FULL_WEIGHT
                
                if sequential_score < MARKOV_SEQ_MEDIUM_WEIGHT:
                    if len(current_actions) >= 5 and len(t_recent) >= 5:
                        if list(current_actions)[-5:] == t_recent[-5:]:
                            sequential_score = MARKOV_SEQ_MEDIUM_WEIGHT
                
                if sequential_score < MARKOV_SEQ_SHORT_WEIGHT:
                    if len(current_actions) >= 3 and len(t_recent) >= 3:
                        if list(current_actions)[-3:] == t_recent[-3:]:
                            sequential_score = MARKOV_SEQ_SHORT_WEIGHT
            
            partial_score = 0.0
            partial_matches = 0
            partial_total = 2
            
            if t_in_battle == current_partial['in_battle']:
                partial_matches += 1
            if t_in_menu == current_partial['in_menu']:
                partial_matches += 1
            
            partial_score = partial_matches / partial_total
            
            total_score = (
                MARKOV_IMMEDIATE_WEIGHT * immediate_score +
                MARKOV_SEQUENTIAL_WEIGHT * sequential_score +
                MARKOV_PARTIAL_WEIGHT * partial_score
            )
            
            if batch_type == "action_change":
                total_score *= 1.2
            
            if transition.get('frame_offset', 0) == 0:
                total_score *= 1.1
            
            if total_score > best_score:
                best_score = total_score
                best_action = t_action
                best_idx = idx
        
        return best_score, best_action, best_idx
    
    def get_markov_action(self, context_state, raw_position=None, taught_frames=None):
        if not self.markov_enabled:
            return False, None, 0.0
        
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        if not frames:
            return False, None, 0.0
        
        score, action, idx = self.compute_markov_similarity(
            context_state, raw_position, taught_frames=frames
        )
        
        self.last_markov_score = score
        
        if score >= MARKOV_FAMILIARITY_THRESHOLD:
            self.last_markov_action = action
            return True, action, score
        
        return False, None, score

    # =========================================================================
    # BATTLE MARKOV SYSTEM
    # =========================================================================
    
    def load_taught_battle_transitions(self, filepath=None):
        filepath = filepath or TAUGHT_BATTLE_TRANSITIONS_FILE
        try:
            if not Path(filepath).exists():
                self.battle_transitions = []
                self.battle_sequences = []
                self.battle_metadata = {}
                self.battle_loaded = False
                print(f"  ⚔️ No battle transitions file found at {filepath}")
                print(f"     Battle fallback: A-button only")
                return
            
            with open(filepath, 'r') as f:
                data = json.load(f)
            
            self.battle_transitions = data.get('flat_frames', [])
            self.battle_sequences = data.get('battle_sequences', [])
            self.battle_metadata = data.get('metadata', {})
            self.battle_loaded = True
            
            print(f"  ⚔️ Loaded battle transitions:")
            print(f"     Flat frames: {len(self.battle_transitions)}")
            print(f"     Battle sequences: {len(self.battle_sequences)}")
            print(f"     Total battle frames: {self.battle_metadata.get('total_battle_frames', 0)}")
            print(f"     Battles recorded: {self.battle_metadata.get('battles_recorded', 0)}")
            print(f"     Avg battle length: {self.battle_metadata.get('avg_battle_length', 0)}")
            outcomes = self.battle_metadata.get('outcomes', {})
            if outcomes:
                print(f"     Outcomes: {outcomes}")
            common_seqs = self.battle_metadata.get('most_common_sequences', [])
            if common_seqs:
                for seq in common_seqs[:3]:
                    print(f"     Common seq: {seq.get('sequence', [])} x{seq.get('count', 0)} ({seq.get('context', '')})")
        
        except Exception as e:
            print(f"  ⚠️ Error loading battle transitions: {e}")
            self.battle_transitions = []
            self.battle_sequences = []
            self.battle_metadata = {}
            self.battle_loaded = False
    
    def compute_battle_markov_similarity(self, context_state, palette_state=None):
        if not self.battle_transitions:
            return 0.0, None, -1
        
        current_actions = list(self.battle_action_history)
        in_menu = context_state[4] > 0.5
        
        best_score = 0.0
        best_action = None
        best_idx = -1
        
        for idx, frame in enumerate(self.battle_transitions):
            t_action = frame.get('action')
            t_recent = frame.get('recent_actions', [])
            t_state = frame.get('state', {})
            batch_type = frame.get('batch_type', 'steady')
            
            if not t_action or t_action == "NONE":
                continue
            
            seq_score = 0.0
            
            if t_recent and current_actions:
                if len(current_actions) >= 8 and len(t_recent) >= 8:
                    if list(current_actions)[-8:] == t_recent[-8:]:
                        seq_score = 1.0
                
                if seq_score < 0.6:
                    if len(current_actions) >= 5 and len(t_recent) >= 5:
                        if list(current_actions)[-5:] == t_recent[-5:]:
                            seq_score = 0.6
                
                if seq_score < 0.3:
                    if len(current_actions) >= 3 and len(t_recent) >= 3:
                        if list(current_actions)[-3:] == t_recent[-3:]:
                            seq_score = 0.3
            
            palette_score = 0.0
            
            if palette_state is not None and 'palette_snapshot' in frame:
                t_palette = np.array(frame['palette_snapshot'], dtype=float)
                if len(t_palette) > 0 and len(palette_state) > 0:
                    min_dim = min(len(t_palette), len(palette_state))
                    diff = np.linalg.norm(t_palette[:min_dim] - palette_state[:min_dim])
                    palette_score = 1.0 / (1.0 + diff * 0.01)
            else:
                palette_score = 0.5
            
            menu_score = 0.0
            t_in_menu = t_state.get('in_menu', 0) == 1
            if t_in_menu == in_menu:
                menu_score = 1.0
            
            total_score = (
                BATTLE_MARKOV_ACTION_SEQ_WEIGHT * seq_score +
                BATTLE_MARKOV_PALETTE_WEIGHT * palette_score +
                BATTLE_MARKOV_MENU_STATE_WEIGHT * menu_score
            )
            
            if batch_type == "action_change":
                total_score *= 1.15
            
            if frame.get('frame_offset', 0) == 0:
                total_score *= 1.1
            
            if total_score > best_score:
                best_score = total_score
                best_action = t_action
                best_idx = idx
        
        return best_score, best_action, best_idx
    
    def get_battle_markov_action(self, context_state, palette_state=None):
        if not self.battle_loaded or not self.battle_transitions:
            return False, None, 0.0
        
        score, action, idx = self.compute_battle_markov_similarity(
            context_state, palette_state
        )
        
        self.last_battle_markov_score = score
        
        threshold = BATTLE_MARKOV_THRESHOLD_LOW
        if len(self.battle_transitions) > 200:
            threshold = BATTLE_MARKOV_THRESHOLD_HIGH
        
        if score >= threshold and action:
            self.last_battle_markov_action = action
            return True, action, score
        
        return False, None, score

    # =========================================================================
    # BAG MARKOV SYSTEM
    # =========================================================================

    def load_taught_bag_transitions(self, filepath=None):
        filepath = filepath or TAUGHT_BAG_TRANSITIONS_FILE
        try:
            if not Path(filepath).exists():
                self.bag_transitions = []
                self.bag_metadata = {}
                self.bag_loaded = False
                print(f"  🎒 No bag transitions file found at {filepath}")
                print(f"     Bag fallback: B-button exit only")
                return

            with open(filepath, 'r') as f:
                data = json.load(f)

            self.bag_transitions = data.get('bag_frames', [])
            self.bag_metadata = data.get('metadata', {})
            self.bag_loaded = True

            print(f"  🎒 Loaded bag transitions:")
            print(f"     Bag frames: {len(self.bag_transitions)}")
            print(f"     Sessions recorded: {self.bag_metadata.get('bag_sessions_recorded', 0)}")
            items_used = self.bag_metadata.get('items_used', [])
            if items_used:
                print(f"     Items used in demos: {items_used}")
            pockets = self.bag_metadata.get('pockets_visited', [])
            if pockets:
                pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
                pk_strs = [pocket_names.get(p, f"?{p}") for p in pockets]
                print(f"     Pockets visited: {', '.join(pk_strs)}")

        except Exception as e:
            print(f"  ⚠️ Error loading bag transitions: {e}")
            self.bag_transitions = []
            self.bag_metadata = {}
            self.bag_loaded = False

    def compute_bag_markov_similarity(self):
        if not self.bag_transitions:
            return 0.0, None, -1

        bgd = self.bag_data
        md = self.menu_data
        gs = self.game_state_raw
        in_battle = self.battle_data.get('battle_cursor', -1) != -1

        current_pocket = bgd.get('pocket', -1)
        current_cursor = bgd.get('cursor', -1)
        current_item_id = self.get_item_at_cursor()
        current_mc = md.get('mc', -1)
        current_mm = md.get('mm', -1)

        hp_ratios = self.get_party_hp_ratios()
        lowest_hp = self.get_lowest_hp_ratio()
        has_status = self.has_status_condition_in_party()
        party_count = self.party_data.get('count', 0)

        current_actions = list(self.bag_action_history)

        best_score = 0.0
        best_action = None
        best_idx = -1

        for idx, frame in enumerate(self.bag_transitions):
            t_action = frame.get('action')
            t_state = frame.get('state', {})
            t_recent = frame.get('recent_actions', [])
            t_party = frame.get('party_context', {})
            batch_type = frame.get('batch_type', 'steady')

            if not t_action or t_action == "NONE":
                continue

            menu_score = 0.0
            menu_matches = 0
            menu_total = 5

            t_pocket = t_state.get('pocket', -1)
            if t_pocket == current_pocket and current_pocket >= 0:
                menu_matches += 1

            t_cursor = t_state.get('cursor', -1)
            if t_cursor >= 0 and current_cursor >= 0:
                cursor_dist = abs(t_cursor - current_cursor)
                if cursor_dist == 0:
                    menu_matches += 1
                elif cursor_dist == 1:
                    menu_matches += 0.5

            t_item_id = t_state.get('item_id', -1)
            if t_item_id > 0 and t_item_id == current_item_id:
                menu_matches += 1

            t_mc = t_state.get('mc', -1)
            if t_mc >= 0 and t_mc == current_mc:
                menu_matches += 1

            t_in_battle = t_state.get('in_battle', False)
            if t_in_battle == in_battle:
                menu_matches += 1

            menu_score = menu_matches / menu_total

            if menu_matches < 1.0:
                continue

            seq_score = 0.0

            if t_recent and current_actions:
                if len(current_actions) >= 6 and len(t_recent) >= 6:
                    if list(current_actions)[-6:] == t_recent[-6:]:
                        seq_score = 1.0

                if seq_score < 0.6:
                    if len(current_actions) >= 4 and len(t_recent) >= 4:
                        if list(current_actions)[-4:] == t_recent[-4:]:
                            seq_score = 0.6

                if seq_score < 0.3:
                    if len(current_actions) >= 2 and len(t_recent) >= 2:
                        if list(current_actions)[-2:] == t_recent[-2:]:
                            seq_score = 0.3

            party_score = 0.0
            party_matches = 0
            party_total = 3

            t_lowest_hp = t_party.get('lowest_hp_ratio', 1.0)
            if t_lowest_hp < 1.0 or lowest_hp < 1.0:
                hp_diff = abs(t_lowest_hp - lowest_hp)
                if hp_diff < 0.15:
                    party_matches += 1
                elif hp_diff < 0.3:
                    party_matches += 0.5
                if t_lowest_hp < 0.3 and lowest_hp < 0.3:
                    party_matches += 0.5
            else:
                party_matches += 1

            t_has_status = t_party.get('has_status', False)
            if t_has_status == has_status:
                party_matches += 1

            t_party_count = t_party.get('party_count', 0)
            if t_party_count > 0 and t_party_count == party_count:
                party_matches += 1
            elif t_party_count > 0:
                party_matches += 0.5

            party_score = party_matches / party_total

            total_score = (
                BAG_MARKOV_MENU_STATE_WEIGHT * menu_score +
                BAG_MARKOV_ACTION_SEQ_WEIGHT * seq_score +
                BAG_MARKOV_PARTY_CONTEXT_WEIGHT * party_score
            )

            if batch_type == "action_change":
                total_score *= 1.2

            if frame.get('frame_offset', 0) == 0:
                total_score *= 1.1

            if total_score > best_score:
                best_score = total_score
                best_action = t_action
                best_idx = idx

        return best_score, best_action, best_idx

    def get_bag_markov_action(self):
        if not self.bag_loaded or not self.bag_transitions:
            return False, None, 0.0

        score, action, idx = self.compute_bag_markov_similarity()

        self.last_bag_markov_score = score

        if score >= BAG_MARKOV_THRESHOLD and action:
            self.last_bag_markov_action = action
            return True, action, score

        return False, None, score

    # =========================================================================
    # START MENU MARKOV SYSTEM (v17.2)
    # =========================================================================

    def load_taught_start_menu_transitions(self, filepath=None):
        """
        Load start menu demonstration frames from taught_start_menu_transitions.json.

        Expected format:
        {
            "start_menu_frames": [
                {
                    "action": "DOWN",
                    "recent_actions": ["START", "DOWN", "DOWN", ...],
                    "state": {
                        "gs": 1,
                        "mc": 2,
                        "mm": 6,
                        "pc": -1,
                        "sc": -1
                    },
                    "context": {
                        "target": "bag",
                        "target_mc": 2,
                        "party_hp_lowest": 0.4,
                        "has_status": false,
                        "in_battle": false,
                        "map_id": 3
                    },
                    "batch_type": "action_change",
                    "frame_offset": 0
                },
                ...
            ],
            "metadata": {
                "total_frames": 80,
                "sessions_recorded": 10,
                "targets_navigated": {"bag": 7, "pokemon": 2, "save": 1},
                "avg_session_length": 8
            }
        }
        """
        filepath = filepath or TAUGHT_START_MENU_TRANSITIONS_FILE
        try:
            if not Path(filepath).exists():
                self.start_menu_transitions = []
                self.start_menu_metadata = {}
                self.start_menu_loaded = False
                print(f"  📋 No start menu transitions file found at {filepath}")
                print(f"     Start menu fallback: hardcoded navigation")
                return

            with open(filepath, 'r') as f:
                data = json.load(f)

            self.start_menu_transitions = data.get('start_menu_frames', [])
            self.start_menu_metadata = data.get('metadata', {})
            self.start_menu_loaded = True

            print(f"  📋 Loaded start menu transitions:")
            print(f"     Frames: {len(self.start_menu_transitions)}")
            print(f"     Sessions recorded: {self.start_menu_metadata.get('sessions_recorded', 0)}")
            targets = self.start_menu_metadata.get('targets_navigated', {})
            if targets:
                print(f"     Targets: {', '.join(f'{k}:{v}' for k, v in targets.items())}")
            avg_len = self.start_menu_metadata.get('avg_session_length', 0)
            if avg_len > 0:
                print(f"     Avg session: {avg_len} frames")

        except Exception as e:
            print(f"  ⚠️ Error loading start menu transitions: {e}")
            self.start_menu_transitions = []
            self.start_menu_metadata = {}
            self.start_menu_loaded = False

    def compute_start_menu_markov_similarity(self):
        """
        Start menu-specific Markov matching against taught frames.

        Primary signal: MENU STATE (45% weight)
        - Current mc (menu cursor position)
        - Target mc (what we're trying to navigate to)
        - mm (menu max — confirms we're in start menu)
        - pc/sc (should be -1 for start menu, not party)

        Secondary signal: ACTION SEQUENCE (35% weight)
        - Recent action history matches taught sequences
        - Start menu navigation is short (typically 2-6 actions)

        Tertiary signal: CONTEXT (20% weight)
        - Why we opened the menu (preparation target matches)
        - Party HP state (healing urgency)
        - Whether we're approaching from overworld or battle
        """
        if not self.start_menu_transitions:
            return 0.0, None, -1

        md = self.menu_data
        current_mc = md.get('mc', -1)
        current_mm = md.get('mm', -1)
        current_pc = md.get('pc', -1)
        target_mc = self.start_menu_target_mc

        lowest_hp = self.get_lowest_hp_ratio()
        has_status = self.has_status_condition_in_party()
        in_battle = self.battle_data.get('battle_cursor', -1) != -1

        current_actions = list(self.start_menu_action_history)

        best_score = 0.0
        best_action = None
        best_idx = -1

        for idx, frame in enumerate(self.start_menu_transitions):
            t_action = frame.get('action')
            t_state = frame.get('state', {})
            t_recent = frame.get('recent_actions', [])
            t_context = frame.get('context', {})
            batch_type = frame.get('batch_type', 'steady')

            if not t_action or t_action == "NONE":
                continue

            # === PRIMARY: Menu state matching (45% weight) ===
            menu_score = 0.0
            menu_matches = 0
            menu_total = 4  # mc, target_mc, mm, pc

            # Menu cursor match
            t_mc = t_state.get('mc', -1)
            if t_mc >= 0 and current_mc >= 0:
                if t_mc == current_mc:
                    menu_matches += 1
                elif abs(t_mc - current_mc) == 1:
                    menu_matches += 0.5

            # Target match — strongest signal
            # If taught frame was navigating to same target as us
            t_target_mc = t_context.get('target_mc', -1)
            if t_target_mc >= 0 and target_mc >= 0:
                if t_target_mc == target_mc:
                    menu_matches += 1
                # Bonus: taught frame's mc is between current and target
                # (frame was mid-navigation toward same goal)
                if t_mc >= 0 and current_mc >= 0 and target_mc >= 0:
                    if min(current_mc, target_mc) <= t_mc <= max(current_mc, target_mc):
                        menu_matches += 0.3
            elif target_mc < 0:
                # No specific target — accept any frame
                menu_matches += 0.5

            # Menu max match (confirms start menu context)
            t_mm = t_state.get('mm', -1)
            if t_mm >= 4 and current_mm >= 4:
                menu_matches += 1

            # Party cursor should be inactive in start menu
            t_pc = t_state.get('pc', -1)
            if t_pc < 0 and current_pc < 0:
                menu_matches += 1
            elif t_pc >= 0 and current_pc >= 0:
                menu_matches += 0.5  # Both in party submenu somehow

            menu_score = menu_matches / menu_total

            # Skip completely irrelevant frames
            if menu_matches < 1.0:
                continue

            # === SECONDARY: Action sequence matching (35% weight) ===
            seq_score = 0.0

            if t_recent and current_actions:
                # Start menu navigation is short — shorter match windows
                if len(current_actions) >= 5 and len(t_recent) >= 5:
                    if list(current_actions)[-5:] == t_recent[-5:]:
                        seq_score = 1.0

                if seq_score < 0.6:
                    if len(current_actions) >= 3 and len(t_recent) >= 3:
                        if list(current_actions)[-3:] == t_recent[-3:]:
                            seq_score = 0.6

                if seq_score < 0.3:
                    if len(current_actions) >= 2 and len(t_recent) >= 2:
                        if list(current_actions)[-2:] == t_recent[-2:]:
                            seq_score = 0.3

            # === TERTIARY: Context matching (20% weight) ===
            context_score = 0.0
            context_matches = 0
            context_total = 3

            # Target name match
            t_target = t_context.get('target', '')
            if self.start_menu_context == "preparation" and t_target == "bag":
                context_matches += 1
            elif self.start_menu_context == t_target:
                context_matches += 1
            elif self.start_menu_context == "unknown":
                context_matches += 0.5

            # HP urgency similarity
            t_hp = t_context.get('party_hp_lowest', 1.0)
            hp_diff = abs(t_hp - lowest_hp)
            if hp_diff < 0.2:
                context_matches += 1
            elif hp_diff < 0.4:
                context_matches += 0.5

            # Battle context
            t_in_battle = t_context.get('in_battle', False)
            if t_in_battle == in_battle:
                context_matches += 1

            context_score = context_matches / context_total

            # === COMBINE ===
            total_score = (
                START_MENU_MARKOV_MENU_STATE_WEIGHT * menu_score +
                START_MENU_MARKOV_ACTION_SEQ_WEIGHT * seq_score +
                START_MENU_MARKOV_CONTEXT_WEIGHT * context_score
            )

            # Boost action_change frames
            if batch_type == "action_change":
                total_score *= 1.2

            # Boost frame_offset 0
            if frame.get('frame_offset', 0) == 0:
                total_score *= 1.1

            if total_score > best_score:
                best_score = total_score
                best_action = t_action
                best_idx = idx

        return best_score, best_action, best_idx

    def get_start_menu_markov_action(self):
        """
        Get start menu action from Markov matching.

        Returns: (matched, action, score)
        """
        if not self.start_menu_loaded or not self.start_menu_transitions:
            return False, None, 0.0

        score, action, idx = self.compute_start_menu_markov_similarity()

        self.last_start_menu_markov_score = score

        if score >= START_MENU_MARKOV_THRESHOLD and action:
            self.last_start_menu_markov_action = action
            return True, action, score

        return False, None, score


# ============================================================================
# CELL 3 PART 3: Action Confirmation, Exploration Memory, Tile Probing
# ============================================================================
# NO CHANGES from original Cell 3 Part 1/2 — just reorganized boundary
# ============================================================================

    # =========================================================================
    # ACTION EXECUTION CONFIRMATION
    # =========================================================================
    
    def set_pending_action(self, action_name):
        self.pending_action = action_name
        self.pending_action_frames = 0
    
    def confirm_action_executed(self, context_state, prev_context_state):
        if self.pending_action is None:
            return True
        self.pending_action_frames += 1
        action_executed = False
        if prev_context_state is not None:
            if self.pending_action in ["UP", "DOWN", "LEFT", "RIGHT"]:
                pos_changed = (context_state[0] != prev_context_state[0] or 
                              context_state[1] != prev_context_state[1])
                dir_changed = context_state[5] != prev_context_state[5]
                action_executed = pos_changed or dir_changed
            elif self.pending_action in ["A", "B", "Start", "Select"]:
                menu_changed = abs(context_state[4] - prev_context_state[4]) > 0.1
                battle_changed = context_state[3] != prev_context_state[3]
                map_changed = context_state[2] != prev_context_state[2]
                action_executed = menu_changed or battle_changed or map_changed
        if action_executed or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES:
            self.last_confirmed_action = self.pending_action
            self.pending_action = None
            self.pending_action_frames = 0
            return True
        return False
    
    def should_send_new_action(self):
        return self.pending_action is None or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES

    # =========================================================================
    # EXPLORATION MEMORY PERSISTENCE
    # =========================================================================
    
    def load_exploration_memory(self):
        try:
            if self.EXPLORATION_MEMORY_FILE.exists():
                with open(self.EXPLORATION_MEMORY_FILE, 'r') as f:
                    data = json.load(f)
                    self.exploration_memory = {}
                    for map_key, map_data in data.items():
                        map_id = int(map_key.replace('map_', ''))
                        self.exploration_memory[map_id] = self._deserialize_map_memory(map_data)
                print(f"  Loaded exploration memory: {len(self.exploration_memory)} maps")
            else:
                self.exploration_memory = {}
        except Exception as e:
            print(f"  Error loading exploration memory: {e}")
            self.exploration_memory = {}

    def _deserialize_map_memory(self, map_data):
        memory = {
            'visited_tiles': set(tuple(t) for t in map_data.get('visited_tiles', [])),
            'obstructions': set(tuple(t) for t in map_data.get('obstructions', [])),
            'interactable_objects': map_data.get('interactable_objects', []),
            'last_visited_timestep': map_data.get('last_visited_timestep', 0),
            'transitions': map_data.get('transitions', []),
            'temp_debt': map_data.get('temp_debt', 0.0),
            'tile_interactions': {}
        }
        for tile_key, tile_data in map_data.get('tile_interactions', {}).items():
            memory['tile_interactions'][tile_key] = {
                'directions_tried': set(tile_data.get('directions_tried', [])),
                'direction_attempts': {int(k): v for k, v in tile_data.get('direction_attempts', {}).items()},
                'direction_successes': {int(k): v for k, v in tile_data.get('direction_successes', {}).items()},
                'exhausted': tile_data.get('exhausted', False)
            }
        return memory

    def save_exploration_memory(self):
        try:
            data = {f'map_{mid}': self._serialize_map_memory(md) for mid, md in self.exploration_memory.items()}
            with open(self.EXPLORATION_MEMORY_FILE, 'w') as f:
                json.dump(data, f, indent=2)
        except Exception as e:
            print(f"  Error saving exploration memory: {e}")

    def _serialize_map_memory(self, map_data):
        serialized_ti = {}
        for tile_key, td in map_data.get('tile_interactions', {}).items():
            serialized_ti[tile_key] = {
                'directions_tried': list(td.get('directions_tried', set())),
                'direction_attempts': {str(k): v for k, v in td.get('direction_attempts', {}).items()},
                'direction_successes': {str(k): v for k, v in td.get('direction_successes', {}).items()},
                'exhausted': td.get('exhausted', False)
            }
        return {
            'visited_tiles': list(map_data['visited_tiles']),
            'obstructions': list(map_data['obstructions']),
            'interactable_objects': map_data['interactable_objects'],
            'last_visited_timestep': map_data['last_visited_timestep'],
            'transitions': map_data.get('transitions', []),
            'temp_debt': map_data.get('temp_debt', 0.0),
            'tile_interactions': serialized_ti
        }

    def get_current_map_memory(self, map_id):
        if map_id not in self.exploration_memory:
            self.exploration_memory[map_id] = {
                'visited_tiles': set(), 'obstructions': set(), 'interactable_objects': [],
                'last_visited_timestep': self.timestep, 'transitions': [], 'temp_debt': 0.0,
                'tile_interactions': {}
            }
        return self.exploration_memory[map_id]

    def record_visited_tile(self, x, y, map_id):
        memory = self.get_current_map_memory(map_id)
        memory['visited_tiles'].add((int(x), int(y)))
        memory['last_visited_timestep'] = self.timestep

    def record_obstruction(self, x, y, map_id, direction):
        dx, dy = self.DIRECTION_DELTAS_INT.get(direction, (0, 0))
        memory = self.get_current_map_memory(map_id)
        memory['obstructions'].add((int(x + dx), int(y + dy)))

    # =========================================================================
    # TILE-BASED INTERACTION PROBING
    # =========================================================================
    
    def get_tile_interaction_key(self, x, y):
        return f"{int(x)}_{int(y)}"
    
    def get_tile_interaction_state(self, x, y, map_id):
        memory = self.get_current_map_memory(map_id)
        tile_key = self.get_tile_interaction_key(x, y)
        if tile_key not in memory['tile_interactions']:
            memory['tile_interactions'][tile_key] = {
                'directions_tried': set(),
                'direction_attempts': {0: 0, 1: 0, 2: 0, 3: 0},
                'direction_successes': {0: 0, 1: 0, 2: 0, 3: 0},
                'exhausted': False
            }
        return memory['tile_interactions'][tile_key]
    
    def should_interact_at_tile(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        if tile_state['exhausted']:
            return False
        if len(tile_state['directions_tried']) < 4:
            return True
        for d in range(4):
            attempts = tile_state['direction_attempts'].get(d, 0)
            successes = tile_state['direction_successes'].get(d, 0)
            if attempts > 0 and successes / attempts >= self.MIN_SUCCESS_RATE_THRESHOLD:
                return True
        return False
    
    def get_untried_directions(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        return [d for d in range(4) if d not in tile_state['directions_tried']]
    
    def get_best_interaction_direction(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        untried = self.get_untried_directions(x, y, map_id)
        if untried:
            return untried[0]
        best_dir, best_rate = None, 0.0
        for d in range(4):
            attempts = tile_state['direction_attempts'].get(d, 0)
            if attempts > 0:
                rate = tile_state['direction_successes'].get(d, 0) / attempts
                if rate > best_rate:
                    best_rate, best_dir = rate, d
        return best_dir
    
    def get_best_probe_action(self, raw_x, raw_y, current_map, current_dir):
        cache_key = (raw_x, raw_y, current_map, current_dir)
        
        if self._probe_cache_position == cache_key:
            return self._cached_probe_action, self._cached_probe_dir
        
        if not self.should_interact_at_tile(raw_x, raw_y, current_map):
            result = (None, None)
        else:
            untried = self.get_untried_directions(raw_x, raw_y, current_map)
            if not untried:
                best_dir = self.get_best_interaction_direction(raw_x, raw_y, current_map)
                if best_dir is not None:
                    result = ('A', current_dir) if current_dir == best_dir else (self.INT_TO_ACTION[best_dir], best_dir)
                else:
                    result = (None, None)
            elif current_dir in untried:
                result = ('A', current_dir)
            else:
                target_dir = untried[0]
                result = (self.INT_TO_ACTION[target_dir], target_dir)
        
        self._probe_cache_position = cache_key
        self._cached_probe_action, self._cached_probe_dir = result
        return result
    
    def record_tile_interaction_attempt(self, x, y, map_id, direction, success):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        tile_state['directions_tried'].add(direction)
        tile_state['direction_attempts'][direction] = tile_state['direction_attempts'].get(direction, 0) + 1
        if success:
            tile_state['direction_successes'][direction] = tile_state['direction_successes'].get(direction, 0) + 1
            memory = self.get_current_map_memory(map_id)
            dir_name = self.DIRECTION_NAMES.get(direction, str(direction))
            interactable = [int(x), int(y), dir_name]
            if interactable not in memory['interactable_objects']:
                memory['interactable_objects'].append(interactable)
                print(f"  🎯 INTERACTABLE FOUND: ({x}, {y}) facing {dir_name}")
        self._check_tile_exhaustion(x, y, map_id)
    
    def _check_tile_exhaustion(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        if len(tile_state['directions_tried']) < 4:
            return
        if not any(tile_state['direction_successes'].get(d, 0) > 0 for d in range(4)):
            tile_state['exhausted'] = True
            print(f"  ✓ Tile ({x}, {y}) exhausted - no interactions found")
    
    def get_direction_success_rate(self, x, y, map_id, direction):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        attempts = tile_state['direction_attempts'].get(direction, 0)
        if attempts == 0:
            return None
        return tile_state['direction_successes'].get(direction, 0) / attempts
    
    def start_interaction_verification(self, x, y, map_id, direction):
        self.pending_interaction_verify = {'x': x, 'y': y, 'map_id': map_id, 'direction': direction}
        self.interaction_verify_countdown = self.INTERACTION_VERIFY_FRAMES
    
    def check_interaction_verification(self, context_state, prev_context_state):
        if self.pending_interaction_verify is None:
            return
        self.interaction_verify_countdown -= 1
        success = False
        if prev_context_state is not None:
            in_overworld = prev_context_state[3] <= 0.5 and prev_context_state[4] <= 0.5
            if in_overworld:
                menu_changed = abs(context_state[4] - prev_context_state[4]) > 0.1
                battle_started = context_state[3] > 0.5 and prev_context_state[3] <= 0.5
                map_changed = int(context_state[2]) != int(prev_context_state[2])
                success = menu_changed or battle_started or map_changed
        if success or self.interaction_verify_countdown <= 0:
            info = self.pending_interaction_verify
            self.record_tile_interaction_attempt(info['x'], info['y'], info['map_id'], info['direction'], success)
            self.pending_interaction_verify = None


# ============================================================================
# CELL 3 PART 4: Transitions, Debt, Navigation (with Cross-Map)
# ============================================================================
# CHANGES from v17.2:
# 1. REMOVED: update_menu_trap_tracking() — moved to Part 1B (v17.3)
#    for dialogue exemption co-location
# 2. REMOVED: reset_menu_trap_boost() — moved to Part 1B (v17.3)
# 3. All other methods UNCHANGED
# ============================================================================

    # =========================================================================
    # TRANSITION SYSTEM
    # =========================================================================
    
    def record_transition(self, from_pos, from_map, to_map, direction, action_type):
        memory = self.get_current_map_memory(from_map)
        for t in memory['transitions']:
            if t['position'] == from_pos and t['direction'] == direction:
                t['use_count'] += 1
                t['last_used'] = self.timestep
                return
        memory['transitions'].append({
            'position': from_pos, 'direction': direction, 'action': action_type,
            'destination_map': to_map, 'use_count': 1, 'last_used': self.timestep
        })
        self._map_graph_dirty = True
        print(f"  🚪 TRANSITION FOUND: Map {from_map} ({from_pos}) → Map {to_map}")

    def get_transition_attraction(self, current_map):
        memory = self.get_current_map_memory(current_map)
        transitions = memory.get('transitions', [])
        if not transitions:
            return 0.0, None
        current_debt = self.map_novelty_debt.get(current_map, 0.0)
        current_temp_debt = self.get_temp_debt(current_map)
        current_coverage = self.get_exploration_coverage(current_map)
        best_attraction, best_transition = 0.0, None
        for t in transitions:
            if self.is_transition_banned(current_map, t['position'], t['direction']):
                continue
            dest_map = t['destination_map']
            dest_debt = self.map_novelty_debt.get(dest_map, 0.0)
            dest_temp_debt = self.get_temp_debt(dest_map)
            dest_coverage = self.get_exploration_coverage(dest_map)
            debt_diff = (current_debt + current_temp_debt * 2.0) - (dest_debt + dest_temp_debt * 2.0)
            coverage_diff = current_coverage - dest_coverage
            attraction = debt_diff * 0.5 + coverage_diff * 0.5
            if t['use_count'] < 3:
                attraction *= 1.5
            if attraction > best_attraction:
                best_attraction, best_transition = attraction, t
        return best_attraction * self.TRANSITION_ATTRACTION_WEIGHT, best_transition

    # =========================================================================
    # TRANSITION BAN SYSTEM
    # =========================================================================
    
    def create_transition_ban(self, map_id, tile_pos, direction_back):
        self.transition_bans[map_id] = {
            'banned_tile': tile_pos, 'banned_direction': direction_back,
            'vicinity_radius': self.BAN_VICINITY_RADIUS, 'vicinity_active': False,
            'created_at': self.timestep
        }
        print(f"  🚫 TRANSITION BAN: Map {map_id} at {tile_pos} facing {self.DIRECTION_NAMES.get(direction_back, '?')}")
    
    def is_transition_banned(self, map_id, position, direction):
        if map_id not in self.transition_bans:
            return False
        ban = self.transition_bans[map_id]
        banned_tile = tuple(ban['banned_tile']) if isinstance(ban['banned_tile'], list) else ban['banned_tile']
        position = tuple(position) if isinstance(position, list) else position
        if position == banned_tile and direction == ban['banned_direction']:
            return True
        if ban['vicinity_active']:
            dist = abs(position[0] - banned_tile[0]) + abs(position[1] - banned_tile[1])
            if dist <= ban['vicinity_radius'] and direction == ban['banned_direction']:
                return True
        return False
    
    def is_position_banned(self, map_id, x, y, direction):
        return self.is_transition_banned(map_id, (x, y), direction)
    
    def update_transition_ban(self, map_id, current_pos):
        if map_id not in self.transition_bans:
            return
        ban = self.transition_bans[map_id]
        banned_tile = tuple(ban['banned_tile']) if isinstance(ban['banned_tile'], list) else ban['banned_tile']
        if not ban['vicinity_active'] and abs(current_pos[0] - banned_tile[0]) + abs(current_pos[1] - banned_tile[1]) >= 3:
            ban['vicinity_active'] = True
            print(f"  🚫 VICINITY BAN ACTIVE: Map {map_id}")
    
    def check_ban_lift_conditions(self, map_id):
        if map_id not in self.transition_bans:
            return
        ban = self.transition_bans[map_id]
        should_lift, reason = False, ""
        memory = self.get_current_map_memory(map_id)
        non_banned = [t for t in memory.get('transitions', []) if not self.is_transition_banned(map_id, t['position'], t['direction'])]
        if non_banned:
            should_lift, reason = True, "alternative transition found"
        elif self.get_exploration_coverage(map_id) >= self.BAN_COVERAGE_LIFT_THRESHOLD:
            should_lift, reason = True, f"coverage reached"
        elif self.timestep - ban['created_at'] >= self.BAN_TIMEOUT_STEPS:
            should_lift, reason = True, "timeout"
        if should_lift:
            del self.transition_bans[map_id]
            print(f"  ✅ BAN LIFTED: Map {map_id} - {reason}")

    # =========================================================================
    # DEBT SYSTEMS
    # =========================================================================
    
    def get_temp_debt(self, map_id):
        memory = self.get_current_map_memory(map_id)
        raw_debt = memory.get('temp_debt', 0.0)
        if map_id != self.current_map_id:
            steps_away = self.timestep - memory.get('last_visited_timestep', 0)
            return max(0.0, raw_debt - steps_away * self.TEMP_DEBT_DECAY)
        return raw_debt

    def accumulate_temp_debt(self, map_id):
        memory = self.get_current_map_memory(map_id)
        memory['temp_debt'] = min(self.TEMP_DEBT_MAX, memory.get('temp_debt', 0.0) + self.TEMP_DEBT_ACCUMULATION)

    def decay_all_debts(self):
        for map_id in list(self.map_novelty_debt.keys()):
            if map_id != self.current_map_id:
                self.map_novelty_debt[map_id] *= (1.0 - self.DEBT_DECAY_RATE)
                if self.map_novelty_debt[map_id] < 0.1:
                    del self.map_novelty_debt[map_id]
        
        current_loc = None
        if self.current_map_id is not None and len(self.last_positions) > 0:
            pos = self.last_positions[-1]
            current_loc = self.get_location_key(pos[0], pos[1], self.current_map_id)
        
        for loc in list(self.location_novelty.keys()):
            if loc != current_loc:
                self.location_novelty[loc] *= (1.0 - self.DEBT_DECAY_RATE)
                if self.location_novelty[loc] < 0.1:
                    del self.location_novelty[loc]

    def get_exploration_coverage(self, map_id):
        memory = self.get_current_map_memory(map_id)
        visited = len(memory['visited_tiles'])
        obstructions = len(memory['obstructions'])
        if visited == 0 or visited + obstructions < 10:
            return 0.0
        return visited / (visited + obstructions)

    def detect_obstruction(self, prev_context, context_state, raw_position, prev_raw_position):
        if prev_context is None or prev_raw_position is None:
            return False
        if self.last_action not in ['UP', 'DOWN', 'LEFT', 'RIGHT']:
            return False
        if raw_position == prev_raw_position:
            self.record_obstruction(raw_position[0], raw_position[1], int(context_state[2]), int(context_state[5]))
            return True
        return False

    # =========================================================================
    # NOTE: update_menu_trap_tracking() and reset_menu_trap_boost()
    # have been moved to Cell 3 Part 1B (v17.3) for co-location with
    # the dialogue state system that exempts them.
    # =========================================================================

    # =========================================================================
    # STANDARD METHODS (no duplicates — defined in Part 1A)
    # =========================================================================

    def get_location_key(self, x, y, map_id, bin_size=5):
        return (int(map_id), int(x // bin_size) * bin_size, int(y // bin_size) * bin_size)

    def is_near_map_edge(self, x, y):
        return x < 10 or x > 245 or y < 10 or y > 245

    def record_action_execution(self, action_name):
        if action_name:
            self.action_execution_count[action_name] = self.action_execution_count.get(action_name, 0) + 1

    def get_position_stagnation(self):
        if len(self.last_positions) < 2:
            return 0
        current_pos = self.last_positions[-1]
        return sum(1 for pos in reversed(list(self.last_positions)[:-1]) if pos == current_pos)

    def get_group_weight(self, group):
        return sum(a.utility for a in self.actions() if a.group == group)

    # =========================================================================
    # MAP CONNECTIVITY GRAPH
    # =========================================================================

    def build_map_graph(self):
        if not self._map_graph_dirty:
            return self._map_graph
        
        graph = {}
        
        for map_id, memory in self.exploration_memory.items():
            edges = []
            for t in memory.get('transitions', []):
                dest = t.get('destination_map')
                if dest is not None:
                    edges.append((dest, t))
            if edges:
                graph[map_id] = edges
        
        self._map_graph = graph
        self._map_graph_dirty = False
        
        return graph

    def find_map_path(self, from_map, to_map):
        if from_map == to_map:
            return [from_map]
        
        graph = self.build_map_graph()
        
        if from_map not in graph:
            return []
        
        from collections import deque as bfs_deque
        queue = bfs_deque([(from_map, [from_map])])
        visited = {from_map}
        
        while queue:
            current, path = queue.popleft()
            
            for dest_map, _ in graph.get(current, []):
                if dest_map == to_map:
                    return path + [dest_map]
                
                if dest_map not in visited:
                    visited.add(dest_map)
                    queue.append((dest_map, path + [dest_map]))
        
        return []

    def get_transition_to_map(self, from_map, to_map):
        memory = self.get_current_map_memory(from_map)
        
        best_transition = None
        best_use_count = -1
        
        for t in memory.get('transitions', []):
            if t.get('destination_map') == to_map:
                pos = tuple(t['position']) if isinstance(t['position'], list) else t['position']
                if self.is_transition_banned(from_map, pos, t['direction']):
                    continue
                
                use_count = t.get('use_count', 0)
                if use_count > best_use_count:
                    best_use_count = use_count
                    best_transition = t
        
        return best_transition

    def get_cross_map_status(self):
        if not self.nav_map_chain:
            return {'active': False}
        
        return {
            'active': True,
            'chain': self.nav_map_chain,
            'chain_index': self.nav_chain_index,
            'chain_length': len(self.nav_map_chain),
            'current_map': self.nav_map_chain[self.nav_chain_index] if self.nav_chain_index < len(self.nav_map_chain) else None,
            'target_map': self.nav_map_chain[-1] if self.nav_map_chain else None,
            'final_target': self.nav_cross_map_target,
            'paused': self.nav_paused,
            'paused_reason': self.nav_paused_reason
        }

    # =========================================================================
    # NAVIGATION SYSTEM - A* Pathfinding + Cross-Map + Taught Targets
    # =========================================================================
    
    def load_taught_nav_targets(self, filepath=None):
        filepath = filepath or TAUGHT_NAV_TARGETS_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    data = json.load(f)
                
                self.taught_nav_targets = {}
                for map_key, targets in data.get('targets_by_map', {}).items():
                    map_id = int(map_key)
                    self.taught_nav_targets[map_id] = targets
                
                self.taught_nav_global_order = data.get('global_order', [])
                self.taught_nav_loaded = True
                
                total = sum(len(t) for t in self.taught_nav_targets.values())
                maps = list(self.taught_nav_targets.keys())
                print(f"  🎯 Loaded taught nav targets:")
                print(f"     Total targets: {total}")
                print(f"     Maps with targets: {maps}")
                print(f"     Global order entries: {len(self.taught_nav_global_order)}")
            else:
                self.taught_nav_targets = {}
                self.taught_nav_global_order = []
                self.taught_nav_loaded = False
                print(f"  No taught nav targets found at {filepath}")
        except Exception as e:
            print(f"  Error loading taught nav targets: {e}")
            self.taught_nav_targets = {}
            self.taught_nav_global_order = []
            self.taught_nav_loaded = False

    def _astar(self, start, goal, map_id):
        import heapq
        
        memory = self.get_current_map_memory(map_id)
        visited_tiles = memory['visited_tiles']
        obstructions = memory['obstructions']
        
        start = (int(start[0]), int(start[1]))
        goal = (int(goal[0]), int(goal[1]))
        
        if start not in visited_tiles:
            return []
        
        if goal not in visited_tiles:
            best_adj = None
            best_dist = float('inf')
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                adj = (goal[0] + dx, goal[1] + dy)
                if adj in visited_tiles and adj not in obstructions:
                    d = abs(adj[0] - start[0]) + abs(adj[1] - start[1])
                    if d < best_dist:
                        best_dist = d
                        best_adj = adj
            if best_adj is None:
                return []
            goal = best_adj
        
        if start == goal:
            return [start]
        
        open_set = [(abs(goal[0] - start[0]) + abs(goal[1] - start[1]), 0, start)]
        came_from = {}
        g_score = {start: 0}
        closed = set()
        
        while open_set:
            f, g, current = heapq.heappop(open_set)
            
            if current == goal:
                path = [current]
                while current in came_from:
                    current = came_from[current]
                    path.append(current)
                path.reverse()
                return path
            
            if current in closed:
                continue
            closed.add(current)
            
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                neighbor = (current[0] + dx, current[1] + dy)
                if neighbor in closed or neighbor not in visited_tiles or neighbor in obstructions:
                    continue
                new_g = g + 1
                if new_g < g_score.get(neighbor, float('inf')):
                    g_score[neighbor] = new_g
                    h = abs(goal[0] - neighbor[0]) + abs(goal[1] - neighbor[1])
                    came_from[neighbor] = current
                    heapq.heappush(open_set, (new_g + h, new_g, neighbor))
        
        return []

    def _get_frontier_tiles(self, map_id):
        memory = self.get_current_map_memory(map_id)
        visited = memory['visited_tiles']
        obstructions = memory['obstructions']
        
        frontier = set()
        for vx, vy in visited:
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                neighbor = (vx + dx, vy + dy)
                if neighbor not in visited and neighbor not in obstructions:
                    if 0 <= neighbor[0] <= 255 and 0 <= neighbor[1] <= 255:
                        frontier.add(neighbor)
        return list(frontier)

    def _score_nav_target(self, target, current_pos, map_id):
        memory = self.get_current_map_memory(map_id)
        visited = memory['visited_tiles']
        obstructions = memory['obstructions']
        tx, ty = target
        
        unvisited_neighbors = 0
        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            n = (tx + dx, ty + dy)
            if n not in visited and n not in obstructions:
                if 0 <= n[0] <= 255 and 0 <= n[1] <= 255:
                    unvisited_neighbors += 1
        
        score = unvisited_neighbors * 2.0
        dist = abs(current_pos[0] - tx) + abs(current_pos[1] - ty)
        score -= dist * 0.05
        if target in self.nav_struck_targets:
            score -= 100.0
        return score

    def _get_taught_targets_for_map(self, map_id, current_pos):
        if not self.taught_nav_loaded:
            return []
        
        map_targets = self.taught_nav_targets.get(map_id, [])
        if not map_targets:
            return []
        
        candidates = []
        for t in map_targets:
            order = t.get('order', 0)
            if order in self.nav_visited_targets:
                continue
            pos = tuple(t['position'])
            if pos in self.nav_struck_targets:
                continue
            dist = abs(current_pos[0] - pos[0]) + abs(current_pos[1] - pos[1])
            candidates.append((pos, t, dist))
        
        candidates.sort(key=lambda x: x[2])
        return [(pos, t) for pos, t, dist in candidates]

    def _get_next_taught_target_any_map(self, current_pos, current_map):
        if not self.taught_nav_loaded or not self.taught_nav_global_order:
            return None, None, None
        
        for entry in self.taught_nav_global_order:
            order = entry.get('order', -1)
            if order in self.nav_visited_targets:
                continue
            
            target_map = entry.get('map_id')
            pos = tuple(entry.get('position', [0, 0]))
            
            if pos in self.nav_struck_targets:
                continue
            
            target_data = entry
            for t in self.taught_nav_targets.get(target_map, []):
                if t.get('order') == order:
                    target_data = t
                    break
            
            return pos, target_data, target_map
        
        return None, None, None

    def build_nav_target_list(self, current_pos, map_id):
        targets = []
        
        taught_targets = self._get_taught_targets_for_map(map_id, current_pos)
        
        if taught_targets:
            for pos, t_data in taught_targets:
                dist = abs(current_pos[0] - pos[0]) + abs(current_pos[1] - pos[1])
                score = t_data.get('forward_progress_score', 0.5) * 10.0
                score -= dist * 0.02
                progress_type = t_data.get('progress_type', 'unknown')
                targets.append((pos, score, f"taught_{progress_type}", map_id))
            
            targets.sort(key=lambda x: x[1], reverse=True)
            return targets
        
        cross_pos, cross_data, cross_map = self._get_next_taught_target_any_map(current_pos, map_id)
        
        if cross_pos is not None and cross_map is not None and cross_map != map_id:
            map_path = self.find_map_path(map_id, cross_map)
            
            if map_path and len(map_path) > 1:
                next_map = map_path[1]
                transition = self.get_transition_to_map(map_id, next_map)
                
                if transition:
                    t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
                    score = 15.0
                    score -= len(map_path) * 0.5
                    targets.append((t_pos, score, f"cross_map_to_{cross_map}", cross_map))
                    
                    self._pending_cross_map = {
                        'final_target': cross_pos,
                        'final_map': cross_map,
                        'target_data': cross_data,
                        'map_chain': map_path,
                        'transition': transition
                    }
                    
                    targets.sort(key=lambda x: x[1], reverse=True)
                    return targets
            
            elif not map_path:
                targets.append(((0, 0), 5.0, f"cross_map_need_{cross_map}", cross_map))
                self._pending_cross_map = {
                    'final_target': cross_pos,
                    'final_map': cross_map,
                    'target_data': cross_data,
                    'map_chain': [],
                    'transition': None
                }
                return targets
        
        frontier = self._get_frontier_tiles(map_id)
        for ft in frontier:
            score = self._score_nav_target(ft, current_pos, map_id)
            targets.append((ft, score, 'frontier', map_id))
        
        memory = self.get_current_map_memory(map_id)
        for t in memory.get('transitions', []):
            t_pos = tuple(t['position']) if isinstance(t['position'], list) else t['position']
            if self.is_transition_banned(map_id, t_pos, t['direction']):
                continue
            score = self._score_nav_target(t_pos, current_pos, map_id)
            dest_coverage = self.get_exploration_coverage(t['destination_map'])
            score += 3.0 * (1.0 - dest_coverage)
            if t_pos not in self.nav_struck_targets:
                targets.append((t_pos, score, 'transition', map_id))
        
        targets.sort(key=lambda x: x[1], reverse=True)
        return targets

    def start_navigation(self, current_pos, map_id):
        self._pending_cross_map = None
        self.nav_target_list = self.build_nav_target_list(current_pos, map_id)
        
        if not self.nav_target_list:
            return False
        
        if self._pending_cross_map:
            cross = self._pending_cross_map
            self._pending_cross_map = None
            
            if cross['map_chain'] and cross['transition']:
                self.nav_map_chain = cross['map_chain']
                self.nav_chain_index = 0
                self.nav_cross_map_target = cross['final_target']
                self.nav_cross_map_target_data = cross['target_data']
                self.nav_paused = False
                
                t_pos = tuple(cross['transition']['position']) if isinstance(cross['transition']['position'], list) else cross['transition']['position']
                path = self._astar(current_pos, t_pos, map_id)
                
                if path and len(path) > 1:
                    self.nav_active = True
                    self.nav_path = path
                    self.nav_path_index = 1
                    self.nav_target = t_pos
                    self.nav_steps_taken = 0
                    self.nav_stagnation_count = 0
                    self.nav_last_position = current_pos
                    self.nav_curiosity_countdown = 0
                    
                    chain_str = ' → '.join(str(m) for m in self.nav_map_chain)
                    print(f"  🧭🌍 CROSS-MAP NAV START: {chain_str}")
                    print(f"     Final target: ({cross['final_target'][0]}, {cross['final_target'][1]}) on map {cross['final_map']}")
                    print(f"     Immediate: → transition at ({t_pos[0]}, {t_pos[1]}) → map {self.nav_map_chain[1]}")
                    self.nav_cross_map_refresh_countdown = self.NAV_CROSS_MAP_REFRESH_INTERVAL
                    return True
                else:
                    self._clear_cross_map_state()
                    
            elif not cross['map_chain']:
                self.nav_map_chain = []
                self.nav_chain_index = 0
                self.nav_cross_map_target = cross['final_target']
                self.nav_cross_map_target_data = cross['target_data']
                self.nav_paused = True
                self.nav_paused_reason = f"no path to map {cross['final_map']}"
                self.nav_paused_target_map = cross['final_map']
                self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
                self.nav_active = True
                
                print(f"  🧭⏸️ CROSS-MAP NAV PAUSED: no path to map {cross['final_map']}")
                print(f"     Exploring to find transitions...")
                return True
        
        self.nav_target_index = 0
        return self._navigate_to_next_target(current_pos, map_id)

    def advance_map_chain(self, new_map_id, current_pos):
        if not self.nav_map_chain:
            return False
        
        new_index = None
        for i, chain_map in enumerate(self.nav_map_chain):
            if chain_map == new_map_id:
                new_index = i
                break
        
        if new_index is None:
            print(f"  🧭🌍 CROSS-MAP: landed on unexpected map {new_map_id}, aborting chain")
            self._clear_cross_map_state()
            return False
        
        self.nav_chain_index = new_index
        
        if new_map_id == self.nav_map_chain[-1]:
            if self.nav_cross_map_target:
                target_pos = self.nav_cross_map_target
                path = self._astar(current_pos, target_pos, new_map_id)
                
                if path and len(path) > 1:
                    self.nav_path = path
                    self.nav_path_index = 1
                    self.nav_target = target_pos
                    self.nav_steps_taken = 0
                    self.nav_stagnation_count = 0
                    self.nav_last_position = current_pos
                    
                    print(f"  🧭🌍 CROSS-MAP FINAL: arrived at map {new_map_id}, "
                          f"pathfinding to ({target_pos[0]}, {target_pos[1]})")
                    return True
                else:
                    print(f"  🧭🌍 CROSS-MAP: can't reach target on final map {new_map_id}")
                    self._clear_cross_map_state()
                    return False
            else:
                self._clear_cross_map_state()
                return False
        
        next_map = self.nav_map_chain[new_index + 1]
        transition = self.get_transition_to_map(new_map_id, next_map)
        
        if transition:
            t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
            path = self._astar(current_pos, t_pos, new_map_id)
            
            if path and len(path) > 1:
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = t_pos
                self.nav_steps_taken = 0
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                
                remaining = len(self.nav_map_chain) - new_index - 1
                print(f"  🧭🌍 CROSS-MAP STEP: map {new_map_id} → transition to map {next_map} "
                      f"({remaining} maps remaining)")
                return True
            else:
                print(f"  🧭🌍 CROSS-MAP: can't reach transition on map {new_map_id}")
                self.nav_paused = True
                self.nav_paused_reason = f"can't reach transition to map {next_map}"
                self.nav_paused_target_map = next_map
                self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
                return True
        else:
            print(f"  🧭🌍 CROSS-MAP: no transition from map {new_map_id} to map {next_map}")
            self.nav_paused = True
            self.nav_paused_reason = f"no transition to map {next_map}"
            self.nav_paused_target_map = next_map
            self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
            return True

    def check_nav_pause_resume(self, current_pos, map_id):
        if not self.nav_paused:
            return False
        
        self.nav_pause_check_countdown -= 1
        if self.nav_pause_check_countdown > 0:
            return False
        
        self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
        
        target_map = self.nav_paused_target_map
        if target_map is None:
            return False
        
        self._map_graph_dirty = True
        
        final_map = self.nav_map_chain[-1] if self.nav_map_chain else target_map
        if self.nav_cross_map_target_data:
            final_map_from_data = None
            for entry in self.taught_nav_global_order:
                if tuple(entry.get('position', [])) == self.nav_cross_map_target:
                    final_map_from_data = entry.get('map_id')
                    break
            if final_map_from_data:
                final_map = final_map_from_data
        
        new_path = self.find_map_path(map_id, final_map)
        
        if new_path and len(new_path) > 1:
            self.nav_map_chain = new_path
            self.nav_chain_index = 0
            self.nav_paused = False
            self.nav_paused_reason = ""
            self.nav_paused_target_map = None
            
            next_map = new_path[1]
            transition = self.get_transition_to_map(map_id, next_map)
            
            if transition:
                t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
                path = self._astar(current_pos, t_pos, map_id)
                
                if path and len(path) > 1:
                    self.nav_path = path
                    self.nav_path_index = 1
                    self.nav_target = t_pos
                    self.nav_steps_taken = 0
                    self.nav_stagnation_count = 0
                    self.nav_last_position = current_pos
                    
                    chain_str = ' → '.join(str(m) for m in new_path)
                    print(f"  🧭▶️ CROSS-MAP NAV RESUMED: {chain_str}")
                    return True
            
            self.nav_paused = True
            self.nav_paused_reason = f"can't reach transition to map {next_map}"
        
        return False

    def refresh_cross_map_navigation(self, current_pos, current_map):
        if not self.nav_cross_map_target:
            return False
        
        final_target = self.nav_cross_map_target
        
        final_map = self.nav_map_chain[-1] if self.nav_map_chain else None
        if final_map is None:
            return False
        
        if current_map == final_map:
            path = self._astar(current_pos, final_target, current_map)
            if path and len(path) > 1:
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = final_target
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                self.nav_paused = False
                print(f"  🧭🔄 CROSS-MAP REFRESH: on final map {current_map}, re-pathing to target")
                return True
            else:
                print(f"  🧭🔄 CROSS-MAP REFRESH: on final map but can't reach target")
                return False
        
        self._map_graph_dirty = True
        new_chain = self.find_map_path(current_map, final_map)
        
        if not new_chain:
            self.nav_map_chain = [current_map]
            self.nav_chain_index = 0
            self.nav_paused = True
            self.nav_paused_reason = f"refresh: no path from map {current_map} to map {final_map}"
            self.nav_paused_target_map = final_map
            self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
            print(f"  🧭🔄 CROSS-MAP REFRESH: no path from {current_map} to {final_map}, pausing")
            return True
        
        old_chain = self.nav_map_chain
        self.nav_map_chain = new_chain
        self.nav_chain_index = 0
        
        next_map = new_chain[1]
        transition = self.get_transition_to_map(current_map, next_map)
        
        if transition:
            t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
            path = self._astar(current_pos, t_pos, current_map)
            
            if path and len(path) > 1:
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = t_pos
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                self.nav_paused = False
                
                old_str = ' → '.join(str(m) for m in old_chain)
                new_str = ' → '.join(str(m) for m in new_chain)
                if old_chain != new_chain:
                    print(f"  🧭🔄 CROSS-MAP REFRESH: chain updated {old_str} → {new_str}")
                else:
                    print(f"  🧭🔄 CROSS-MAP REFRESH: re-pathed on map {current_map}")
                return True
            else:
                self.nav_paused = True
                self.nav_paused_reason = f"refresh: can't reach transition to map {next_map}"
                self.nav_paused_target_map = next_map
                self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
                print(f"  🧭🔄 CROSS-MAP REFRESH: can't reach transition to {next_map}, pausing")
                return True
        else:
            self.nav_paused = True
            self.nav_paused_reason = f"refresh: no transition to map {next_map}"
            self.nav_paused_target_map = next_map
            self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
            print(f"  🧭🔄 CROSS-MAP REFRESH: no transition to {next_map}, pausing")
            return True

    def _clear_cross_map_state(self):
        self.nav_map_chain = []
        self.nav_chain_index = 0
        self.nav_cross_map_target = None
        self.nav_cross_map_target_data = None
        self.nav_paused = False
        self.nav_paused_reason = ""
        self.nav_paused_target_map = None
        self.nav_pause_check_countdown = 0
        self.nav_cross_map_refresh_countdown = 0

    def _navigate_to_next_target(self, current_pos, map_id):
        while self.nav_target_index < len(self.nav_target_list):
            entry = self.nav_target_list[self.nav_target_index]
            target = entry[0]
            score = entry[1]
            target_type = entry[2]
            target_map = entry[3] if len(entry) > 3 else map_id
            
            if target in self.nav_struck_targets:
                self.nav_target_index += 1
                continue
            
            if target_type.startswith('cross_map_need'):
                self.nav_target_index += 1
                continue
            
            path = self._astar(current_pos, target, map_id)
            
            if path and len(path) > 1:
                self.nav_active = True
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = target
                self.nav_steps_taken = 0
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                self.nav_curiosity_countdown = 0
                
                print(f"  🧭 NAV START: → ({target[0]}, {target[1]}) [{target_type}] "
                      f"score={score:.1f} path={len(path)} steps")
                return True
            
            self.nav_target_index += 1
        
        self.abort_navigation("no valid targets")
        return False

    def get_nav_action(self, current_pos):
        if not self.nav_active or not self.nav_path:
            return None
        
        if self.nav_paused:
            return None
        
        if self.nav_path_index < len(self.nav_path):
            next_tile = self.nav_path[self.nav_path_index]
            
            if current_pos == next_tile:
                self.nav_path_index += 1
                if self.nav_path_index >= len(self.nav_path):
                    return None
                next_tile = self.nav_path[self.nav_path_index]
            
            dx = next_tile[0] - current_pos[0]
            dy = next_tile[1] - current_pos[1]
            
            if dx > 0: return "RIGHT"
            elif dx < 0: return "LEFT"
            elif dy > 0: return "DOWN"
            elif dy < 0: return "UP"
        
        return None

    def update_nav_state(self, current_pos, map_id):
        if not self.nav_active:
            return False
        
        if self.nav_paused:
            self.check_nav_pause_resume(current_pos, map_id)
            return True
        
        self.nav_steps_taken += 1
        
        if self.nav_map_chain:
            self.nav_cross_map_refresh_countdown -= 1
            if self.nav_cross_map_refresh_countdown <= 0:
                self.nav_cross_map_refresh_countdown = self.NAV_CROSS_MAP_REFRESH_INTERVAL
                self.refresh_cross_map_navigation(current_pos, map_id)
        
        if self.nav_steps_taken >= self.NAV_MAX_STEPS:
            self.abort_navigation("max steps reached")
            return False
        
        if current_pos == self.nav_last_position:
            self.nav_stagnation_count += 1
            if self.nav_stagnation_count >= self.NAV_STAGNATION_LIMIT:
                self.abort_navigation("stuck during pathfinding")
                return False
        else:
            self.nav_stagnation_count = 0
        self.nav_last_position = current_pos
        
        if self.nav_target:
            dist_to_target = abs(current_pos[0] - self.nav_target[0]) + abs(current_pos[1] - self.nav_target[1])
            if dist_to_target <= 1:
                if self.nav_map_chain and self.nav_chain_index < len(self.nav_map_chain) - 1:
                    print(f"  🧭🌍 Approaching transition at ({self.nav_target[0]}, {self.nav_target[1]})")
                    return True
                
                if self.nav_curiosity_countdown == 0:
                    self.nav_curiosity_countdown = self.NAV_CURIOSITY_WINDOW
                    self._mark_taught_target_visited(self.nav_target)
                    print(f"  🧭 NAV ARRIVED: ({self.nav_target[0]}, {self.nav_target[1]}) "
                          f"— curiosity window {self.NAV_CURIOSITY_WINDOW} steps")
                    return False
        
        return True

    def _mark_taught_target_visited(self, position):
        if not self.taught_nav_loaded:
            return
        pos_tuple = tuple(position)
        maps_to_check = [self.current_map_id] if self.current_map_id is not None else []
        maps_to_check += [m for m in self.taught_nav_targets.keys() if m != self.current_map_id]
        
        for check_map in maps_to_check:
            for t in self.taught_nav_targets.get(check_map, []):
                t_pos = tuple(t['position'])
                if t_pos == pos_tuple:
                    order = t.get('order', -1)
                    if order >= 0:
                        self.nav_visited_targets.add(order)
                        print(f"  🧭 TARGET VISITED: order #{order} at ({pos_tuple[0]}, {pos_tuple[1]}) on map {check_map}")
                    return

    def complete_nav_target(self, found_novelty):
        if found_novelty:
            print(f"  🧭 NAV SUCCESS: Novelty found at ({self.nav_target[0]}, {self.nav_target[1]})")
            self.abort_navigation("novelty found")
            return
        
        if self.nav_target:
            self.nav_struck_targets.add(self.nav_target)
            print(f"  🧭 NAV STRIKE: ({self.nav_target[0]}, {self.nav_target[1]}) — no novelty, trying next")
        
        if self.nav_map_chain and self.nav_chain_index >= len(self.nav_map_chain) - 1:
            self._clear_cross_map_state()
        
        self.nav_target_index += 1
        current_pos = self.nav_last_position or (0, 0)
        map_id = self.current_map_id
        
        if not self._navigate_to_next_target(current_pos, map_id):
            self.abort_navigation("all targets exhausted")

    def abort_navigation(self, reason=""):
        if self.nav_active:
            cross_info = ""
            if self.nav_map_chain:
                cross_info = f" [cross-map chain: {' → '.join(str(m) for m in self.nav_map_chain)}]"
            print(f"  🧭 NAV END: {reason} (took {self.nav_steps_taken} steps){cross_info}")
        
        self.nav_active = False
        self.nav_path = []
        self.nav_path_index = 0
        self.nav_target = None
        self.nav_steps_taken = 0
        self.nav_stagnation_count = 0
        self.nav_curiosity_countdown = 0
        self._clear_cross_map_state()

    def update_known_area_counter(self, raw_x, raw_y, map_id):
        memory = self.get_current_map_memory(map_id)
        pos = (int(raw_x), int(raw_y))
        if pos in memory['visited_tiles']:
            self.known_area_counter += 1
        else:
            self.known_area_counter = 0

    def should_start_navigation(self):
        if self.nav_active:
            return False
        if self.known_area_counter < self.KNOWN_AREA_TRIGGER:
            return False
        return True

    def is_nav_active(self):
        return self.nav_active

    def is_nav_paused(self):
        return self.nav_paused

    def is_in_nav_curiosity_window(self):
        return self.nav_curiosity_countdown > 0

    def tick_nav_curiosity_window(self):
        if self.nav_curiosity_countdown > 0:
            self.nav_curiosity_countdown -= 1
            if self.nav_curiosity_countdown <= 0:
                return True
        return False

    def get_nav_targets_status(self):
        if not self.taught_nav_loaded:
            return {'loaded': False, 'total': 0, 'visited': 0, 'remaining': 0}
        total = sum(len(t) for t in self.taught_nav_targets.values())
        visited = len(self.nav_visited_targets)
        return {'loaded': True, 'total': total, 'visited': visited, 'remaining': total - visited}


# ============================================================================
# CELL 3 PART 5: Stagnation, Blend Triggers, Mode Swap, Repetition/Pattern
# ============================================================================
# NO CHANGES — reorganized from original Cell 3 Part 3
# ============================================================================

    # =========================================================================
    # BLEND TIER DETECTION
    # =========================================================================
    
    def get_blend_tier(self):
        """
        Determine blend tier based on current stagnation metrics.
        Returns 0 (no blend), 1 (light), 2 (medium), or 3 (hard).
        Higher tier = more taught model influence.
        """
        # Tier 3 (hard): extreme stagnation
        t3 = self.BLEND_TIER_TRIGGERS[3]
        if (self.detected_pattern and self.pattern_repeat_count >= t3['pattern_repeats']):
            return 3
        if self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * t3['state_stagnation_mult']:
            return 3
        
        # Tier 2 (medium): significant stagnation
        t2 = self.BLEND_TIER_TRIGGERS[2]
        if (self.detected_pattern and self.pattern_repeat_count >= t2['pattern_repeats']):
            return 2
        if self.get_position_stagnation() >= t2['pos_stagnation']:
            return 2
        if self.consecutive_action_count >= t2['consecutive']:
            return 2
        
        # Tier 1 (light): early stagnation
        t1 = self.BLEND_TIER_TRIGGERS[1]
        if (self.detected_pattern and self.pattern_repeat_count >= t1['pattern_repeats']):
            return 1
        if self.get_position_stagnation() >= t1['pos_stagnation']:
            return 1
        if self.consecutive_action_count >= t1['consecutive']:
            return 1
        
        return 0
    
    def try_blend_if_needed(self):
        """
        Check if blend should trigger. Called from action selection.
        Returns True if a blend was performed.
        """
        if not self.taught_reference['loaded']:
            return False
        
        tier = self.get_blend_tier()
        
        if tier == 0:
            return False
        
        # Only blend if tier escalated or cooldown passed
        if tier <= self.blend_tier and (self.timestep - self.last_blend_timestep) < self.BLEND_COOLDOWN:
            return False
        
        self.blend_from_taught(tier)
        return True

    # =========================================================================
    # MODE SWAP & STAGNATION  
    # =========================================================================
    
    def get_context_state_hash(self, context_state):
        return (round(context_state[0], 2), round(context_state[1], 2), int(context_state[2]),
                int(context_state[3]), round(context_state[4], 2), int(context_state[5]))

    def check_state_stagnation(self, context_state):
        current_hash = self.get_context_state_hash(context_state)
        if current_hash == self.last_context_state_hash:
            self.state_stagnation_count += 1
            if self.state_stagnation_count == 1 and self.last_action:
                self.stagnation_initiator_action = self.last_action
        else:
            self.state_stagnation_count = 0
            self.stagnation_initiator_action = None
        self.last_context_state_hash = current_hash
        return self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD

    def check_position_stagnation(self):
        return self.get_position_stagnation()

    def should_force_random(self):
        """
        Returns True if the agent is badly stuck and needs forced randomization.
        Also triggers blend from taught model before randomizing.
        """
        force = False
        
        if self.get_position_stagnation() >= 8:
            force = True
        if self.consecutive_action_count >= 15:
            force = True
        if self.detected_pattern and self.pattern_repeat_count >= 4:
            force = True
        if self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * 2:
            force = True
        
        if force:
            # Attempt blend before randomizing — blend fixes priorities,
            # random breaks the immediate loop
            self.try_blend_if_needed()
        
        return force

    def get_forced_random_action_name(self):
        """Pick a random action that ISN'T the currently repeated one or in the pattern."""
        candidates = ["UP", "DOWN", "LEFT", "RIGHT", "A", "B"]
        
        if self.current_repeated_action and self.current_repeated_action in candidates:
            candidates.remove(self.current_repeated_action)
        
        if self.detected_pattern:
            for a in self.detected_pattern:
                if a in candidates:
                    candidates.remove(a)
        
        if not candidates:
            candidates = ["UP", "DOWN", "LEFT", "RIGHT"]
            if self.current_repeated_action in candidates:
                candidates.remove(self.current_repeated_action)
        
        if not candidates:
            candidates = ["UP", "DOWN", "LEFT", "RIGHT"]
        
        return random.choice(candidates)

    def check_direction_change_progress(self, context_state):
        current_dir = int(context_state[5])
        if self.last_direction_for_progress is None:
            self.last_direction_for_progress = current_dir
            return False
        changed = current_dir != self.last_direction_for_progress
        self.last_direction_for_progress = current_dir
        return changed

    def apply_stagnation_initiator_penalty(self):
        if self.stagnation_initiator_action is None:
            return
        for a in self.actions():
            if a.action == self.stagnation_initiator_action:
                old_util = a.utility
                floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                a.utility = max(floor, a.utility * 0.5)
                print(f"  📍 STAGNATION PENALTY: {self.stagnation_initiator_action} {old_util:.3f} → {a.utility:.3f}")
                break
        self.stagnation_initiator_action = None

    def check_productive_change(self, context_state):
        current_map = int(context_state[2])
        current_battle = context_state[3] > 0.5
        current_pos = (context_state[0], context_state[1])
        productive, reason = False, ""
        
        if self.last_map_id is not None and current_map != self.last_map_id:
            productive, reason = True, "map change"
        if self.last_battle_state is not None and current_battle != self.last_battle_state:
            productive, reason = True, "battle change"
        if self.position_at_mode_swap is not None:
            dist = np.sqrt((current_pos[0] - self.position_at_mode_swap[0])**2 + 
                          (current_pos[1] - self.position_at_mode_swap[1])**2)
            if dist > 0.03:
                productive, reason = True, f"moved {dist*255:.1f} tiles"
        
        if self.direction_change_counts_as_progress and self.check_direction_change_progress(context_state):
            self.state_stagnation_count = max(0, self.state_stagnation_count - 5)
        
        self.last_map_id = current_map
        self.last_battle_state = current_battle
        return productive, reason

    def on_productive_change(self, reason):
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.swap_chain_count = 0
        self.state_stagnation_count = 0
        self.stagnation_initiator_action = None
        self.unproductive_swap_count = 0
        
        # Reset blend tier on productive progress
        if self.blend_tier > 0:
            print(f"  ✅ Blend tier reset: {self.blend_tier} → 0 ({reason})")
            self.blend_tier = 0
        
        # Reset known area counter — productive change means fresh territory
        self.known_area_counter = 0

    def on_mode_swap(self, from_mode, to_mode):
        self.swap_chain_count += 1
        self.frames_in_current_mode = 0
        self.unproductive_swap_count += 1
        if self.unproductive_swap_count >= self.UNPRODUCTIVE_SWAP_THRESHOLD:
            self._reset_highest_to_third(to_mode)
            self.unproductive_swap_count = 0
        if to_mode == "interact":
            self.interact_to_move_threshold = min(self.MAX_THRESHOLD, self.interact_to_move_threshold + self.THRESHOLD_INCREMENT)
        else:
            self.move_to_interact_threshold = min(self.MAX_THRESHOLD, self.move_to_interact_threshold + self.THRESHOLD_INCREMENT)

    def _reset_highest_to_third(self, mode):
        if mode in ["battle", "both"]:
            return
        group = "move" if mode == "move" else "interact"
        group_actions = [a for a in self.actions() if a.group == group]
        if len(group_actions) < 3:
            return
        sorted_actions = sorted(group_actions, key=lambda a: a.utility, reverse=True)
        floor = self.INTERACT_UTILITY_FLOOR if group == "interact" else self.MOVE_UTILITY_FLOOR
        sorted_actions[0].utility = max(sorted_actions[2].utility * 0.9, floor)

    def should_use_both_mode(self):
        return (self.state_stagnation_count > self.BOTH_MODE_STAGNATION_THRESHOLD or 
                self.unproductive_swap_count > self.BOTH_MODE_SWAP_THRESHOLD)

    def determine_control_mode(self, context_state, raw_position=None):
        if context_state[3] > 0.5:
            return "battle"
        
        self.frames_in_current_mode += 1
        position_stagnation = self.get_position_stagnation()
        
        productive, reason = self.check_productive_change(context_state)
        if productive:
            self.on_productive_change(reason)
        
        if self.should_use_both_mode():
            return "both"
        
        if self.check_state_stagnation(context_state):
            self.apply_stagnation_initiator_penalty()
            new_mode = "interact" if self.control_mode == "move" else "move"
            self.control_mode = new_mode
            self.position_at_mode_swap = (context_state[0], context_state[1])
            self.on_mode_swap(self.control_mode, new_mode)
            self.state_stagnation_count = 0
            return self.control_mode
        
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_map = int(context_state[2])
        
        tile_needs_probing = self.should_interact_at_tile(raw_x, raw_y, current_map)
        untried_directions = self.get_untried_directions(raw_x, raw_y, current_map)
        
        if tile_needs_probing and untried_directions and self.control_mode == "move" and self.frames_in_current_mode >= 3:
            self.control_mode = "interact"
            self.position_at_mode_swap = (context_state[0], context_state[1])
            self.frames_in_current_mode = 0
            return self.control_mode
        
        if self.control_mode == "move" and position_stagnation >= self.move_to_interact_threshold:
            self.control_mode = "interact"
            self.position_at_mode_swap = (context_state[0], context_state[1])
            self.on_mode_swap("move", "interact")
        elif self.control_mode == "interact":
            if (not tile_needs_probing or not untried_directions) and self.frames_in_current_mode >= 5:
                self.control_mode = "move"
                self.position_at_mode_swap = (context_state[0], context_state[1])
                self.frames_in_current_mode = 0
            elif self.frames_in_current_mode >= self.interact_to_move_threshold:
                self.control_mode = "move"
                self.position_at_mode_swap = (context_state[0], context_state[1])
                self.on_mode_swap("interact", "move")
        
        return self.control_mode

    # =========================================================================
    # EXPLORATION TRACKING
    # =========================================================================
    
    def update_exploration_tracking(self, context_state, prev_context_state, raw_position=None, prev_raw_position=None):
        current_map = int(context_state[2])
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_pos = (raw_x, raw_y)
        
        if self.current_map_id is not None and current_map != self.current_map_id:
            prev_map = self.current_map_id
            if prev_context_state is not None and prev_raw_position is not None:
                self.record_transition(prev_raw_position, prev_map, current_map,
                    int(prev_context_state[5]), 'interact' if self.last_action == 'A' else 'walk')
            if prev_raw_position is not None:
                entry_dir = int(context_state[5]) if prev_context_state is not None else 0
                self.create_transition_ban(current_map, current_pos, (entry_dir + 2) % 4)
            self.on_map_change(current_map)
        
        self.current_map_id = current_map
        self.record_visited_tile(raw_x, raw_y, current_map)
        self.accumulate_temp_debt(current_map)
        self.update_transition_ban(current_map, current_pos)
        self.check_ban_lift_conditions(current_map)
        
        if prev_context_state is not None and prev_raw_position is not None:
            self.detect_obstruction(prev_context_state, context_state, raw_position, prev_raw_position)
        
        self.check_interaction_verification(context_state, prev_context_state)
        self.last_direction = int(context_state[5])
        
        if self.timestep % 300 == 0:
            self.decay_all_debts()

    def on_map_change(self, new_map):
        self.save_exploration_memory()
        self.control_mode = "move"
        self.frames_in_current_mode = 0
        
        # Abort navigation on map change
        if self.nav_active:
            self.abort_navigation("map changed")
        self.known_area_counter = 0
        self.nav_struck_targets.clear()
        
        memory = self.get_current_map_memory(new_map)
        tile_interactions = memory.get('tile_interactions', {})
        print(f"  🗺️ MAP CHANGE → {new_map}: {len(memory['visited_tiles'])} visited, {len(memory['obstructions'])} obs")
        print(f"     Tiles probed: {len(tile_interactions)}, exhausted: {sum(1 for t in tile_interactions.values() if t.get('exhausted', False))}")

    # =========================================================================
    # REPETITION & PATTERN HANDLING
    # =========================================================================
    
    def track_consecutive_action(self, action_name):
        if action_name == self.current_repeated_action:
            self.consecutive_action_count += 1
        else:
            self.current_repeated_action = action_name
            self.consecutive_action_count = 1

    def get_learning_multiplier(self, action_name):
        if action_name != self.current_repeated_action or self.consecutive_action_count < self.LEARNING_SLOWDOWN_START:
            return 1.0
        progress = min(1.0, (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) / 
                       (self.LEARNING_SLOWDOWN_MAX - self.LEARNING_SLOWDOWN_START))
        return max(0.05, 1.0 - 0.95 * progress)

    def get_nth_highest_utility(self, group, n=3):
        utilities = sorted([a.utility for a in self.actions() if a.group == group], reverse=True)
        if len(utilities) < n:
            return self.INTERACT_UTILITY_FLOOR if group == "interact" else self.MOVE_UTILITY_FLOOR
        return utilities[n-1]

    def detect_pattern(self):
        if len(self.action_history) < 6:
            return None, 0
        recent = list(self.action_history)[-self.PATTERN_CHECK_WINDOW:]
        for pattern_len in range(1, self.PATTERN_MAX_LENGTH + 1):
            if len(recent) < pattern_len * self.PATTERN_MIN_REPEATS:
                continue
            candidate = tuple(recent[-pattern_len:])
            repeat_count, idx = 0, len(recent) - pattern_len
            while idx >= 0 and tuple(recent[idx:idx + pattern_len]) == candidate:
                repeat_count += 1
                idx -= pattern_len
            if repeat_count >= self.PATTERN_MIN_REPEATS:
                return candidate, repeat_count
        return None, 0

    def apply_pattern_penalty(self):
        pattern, repeat_count = self.detect_pattern()
        if pattern is None:
            self.detected_pattern, self.pattern_repeat_count = None, 0
            return
        self.detected_pattern, self.pattern_repeat_count = pattern, repeat_count
        
        penalty_factor = max(0.3, 1.0 - repeat_count * 0.15)
        
        for action_name in set(pattern):
            for a in self.actions():
                if a.action == action_name:
                    floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                    a.utility = max(floor, a.utility * penalty_factor)
                    break

    def apply_repetition_penalty(self):
        if self.current_repeated_action is None:
            return
        for a in self.actions():
            if a.action == self.current_repeated_action:
                floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                if self.consecutive_action_count >= self.HARD_RESET_THRESHOLD:
                    a.utility = floor
                    self.consecutive_action_count = 0
                    print(f"  🔨 HARD RESET: {a.action} → {floor:.3f}")
                elif self.consecutive_action_count >= self.PENALTY_THRESHOLD:
                    a.utility = max(a.utility * 0.5, floor)
                break

# ============================================================================
# CELL 3 PART 6: Entity Spawning/Clustering, Learning, Save/Load
# ============================================================================
# CHANGES from v17.3:
# 1. UPDATED: learn() — runs pipeline forward passes and backward credit
#    assignment for active context pipelines (battle, overworld, bag, party)
# 2. UPDATED: learn_battle_chain() — runs battle pipeline forward+backward
# 3. UPDATED: learn_bag_chain() — runs bag pipeline forward+backward
# 4. UPDATED: learn_party_chain() — runs party pipeline forward+backward
# 5. NEW: learn_pipeline() — generic pipeline forward+backward+spawn
# 6. UPDATED: save_model_checkpoint() — includes pipeline state, revenge
# 7. UPDATED: load_taught_model() — restores pipeline state, revenge
# 8. UPDATED: initialize_from_taught_model() — restores pipeline state
# 9. UPDATED: cleanup_memory() — includes pipeline pool pruning
# 10. All entity pruning methods UNCHANGED
# ============================================================================

    # =========================================================================
    # ENTITY SPAWNING (Legacy wrappers + chain-aware)
    # =========================================================================

    def spawn_entity_from_novelty(self, learning_state, context_state, raw_position=None):
        """Legacy wrapper — spawns into overworld chain."""
        self.spawn_entity_for_chain("overworld", learning_state, context_state, raw_position)

    def check_entity_capacity(self):
        """Legacy wrapper — checks overworld chain."""
        self.check_chain_entity_capacity("overworld")

    def cluster_entities(self):
        """Legacy wrapper — clusters overworld chain."""
        self.cluster_chain_entities("overworld")

    # =========================================================================
    # INNATE ENTITY SPAWNING
    # =========================================================================

    def spawn_innate_entities(self, learning_state):
        """Legacy wrapper — spawns overworld innate entities."""
        self.spawn_innate_overworld_entities(learning_state)

    # =========================================================================
    # ENTITY PRUNING
    # =========================================================================

    def prune_low_utility_entities(self, chain, min_familiarity=5.0,
                                    max_fit_score=0.05, min_activations=30):
        innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition",
                        "battle_hp_crisis", "battle_enemy_weak", "battle_species_match",
                        "battle_status", "battle_trainer"}

        chain_entities = self.entities(chain=chain)
        prunable = []

        for e in chain_entities:
            if e.entity_type in innate_types:
                continue
            if len(e.cluster_activations) < min_activations:
                continue
            if e.familiarity < min_familiarity:
                continue
            if e.activation_fit_score <= max_fit_score:
                prunable.append(e)

        if not prunable:
            return 0

        non_innate = [e for e in chain_entities if e.entity_type not in innate_types]
        max_prune = max(1, len(non_innate) // 2)
        prunable.sort(key=lambda e: e.utility)
        to_prune = prunable[:max_prune]

        prune_ids = {id(e) for e in to_prune}
        self.perceptrons = [p for p in self.perceptrons if id(p) not in prune_ids]
        self._cache_valid = False

        pruned = len(to_prune)
        if pruned > 0:
            print(f"  🧹 [{chain}] Pruned {pruned} low-utility entities "
                  f"(fam≥{min_familiarity:.1f} fit≤{max_fit_score:.2f})")
            for e in to_prune:
                print(f"     {e.entity_type}: fam={e.familiarity:.1f} "
                      f"fit={e.activation_fit_score:.3f} util={e.utility:.3f}")
        return pruned

    def periodic_entity_pruning(self):
        total_pruned = 0
        for chain in ['overworld', 'battle', 'party', 'bag', 'shared']:
            n_entities = self.get_chain_entity_count(chain)
            capacity = self.get_chain_entity_capacity(chain)
            if n_entities >= capacity * 0.7:
                pruned = self.prune_low_utility_entities(chain)
                total_pruned += pruned
        return total_pruned

    # =========================================================================
    # PIPELINE POOL PRUNING (NEW)
    # =========================================================================

    def prune_pipeline_pools(self):
        """
        Prune low-quality perceptrons from pipeline pools.
        Pages them to residual instead of deleting.
        """
        total_pruned = 0
        for pid, pipeline in self.pipelines.items():
            for i, pool in enumerate(pipeline.pools):
                pool_perceptrons = [p for p in self.perceptrons if p.pool_id == pool.pool_id]
                if len(pool_perceptrons) < pool.max_perceptrons * 0.7:
                    continue

                prunable = []
                for p in pool_perceptrons:
                    if len(p.cluster_activations) < 20:
                        continue  # too young
                    if p.familiarity >= 3.0 and p.activation_fit_score <= 0.05:
                        prunable.append(p)

                if not prunable:
                    continue

                max_prune = max(1, len(pool_perceptrons) // 3)
                prunable.sort(key=lambda p: p.utility)
                to_prune = prunable[:max_prune]

                for p in to_prune:
                    pool.page_to_residual(p)

                prune_ids = {id(p) for p in to_prune}
                self.perceptrons = [p for p in self.perceptrons if id(p) not in prune_ids]
                self._cache_valid = False
                total_pruned += len(to_prune)

                if to_prune:
                    print(f"  🧹 [{pid}.{pool.name}] Pruned {len(to_prune)} → residual "
                          f"(pool: {len(pool_perceptrons) - len(to_prune)} remain, "
                          f"residual: {len(pool.residual)})")

        return total_pruned

    # =========================================================================
    # MEMORY CLEANUP
    # =========================================================================

    def cleanup_memory(self):
        # Trim location memory
        if len(self.location_memory) > 500:
            sorted_locs = sorted(self.location_memory.items(), key=lambda x: x[1], reverse=True)
            self.location_memory = dict(sorted_locs[:400])
            print(f"  🧹 Location memory trimmed to 400")

        if len(self.location_novelty) > 500:
            sorted_nov = sorted(self.location_novelty.items(), key=lambda x: x[1], reverse=True)
            self.location_novelty = dict(sorted_nov[:400])

        # Trim exhausted tile interactions
        for map_id, memory in self.exploration_memory.items():
            ti = memory.get('tile_interactions', {})
            if len(ti) > 200:
                to_remove = []
                for tile_key, td in ti.items():
                    if td.get('exhausted', False):
                        if not any(td.get('direction_successes', {}).get(d, 0) > 0 for d in range(4)):
                            to_remove.append(tile_key)
                for tk in to_remove[:50]:
                    del ti[tk]
                if to_remove:
                    print(f"  🧹 Map {map_id}: removed {min(len(to_remove), 50)} exhausted tiles")

        # Legacy entity pruning
        self.periodic_entity_pruning()

        # Pipeline pool pruning (NEW)
        self.prune_pipeline_pools()

        # Knowledge pruning
        self.prune_stale_move_knowledge()
        self.prune_stale_item_knowledge()

        # Type clustering
        if self.battles_since_last_clustering >= self.CLUSTERING_BATTLE_INTERVAL:
            self.run_type_clustering()

        # Revenge readiness check
        self.check_revenge_readiness()

    # =========================================================================
    # LEARNING — CORE HELPERS
    # =========================================================================

    def enforce_utility_floors(self):
        for a in self.actions():
            floor = self.MOVE_UTILITY_FLOOR if a.group == "move" else self.INTERACT_UTILITY_FLOOR
            a.utility = max(a.utility, floor)

    def get_spawn_threshold_adaptive(self, error_type='combined', percentile=50):
        history = {'numeric': self.numeric_error_history, 'visual': self.visual_error_history}.get(
            error_type, self.error_history)
        return max(0.001, np.percentile(history, percentile)) if len(history) >= 100 else 0.0005

    def stagnation_level(self, window=10):
        if len(self.prev_learning_states) < window:
            return 0.0
        recent = list(self.prev_learning_states)[-window:]
        diffs = []
        for i in range(1, len(recent)):
            a, b = recent[i], recent[i-1]
            min_dim = min(len(a), len(b))
            diffs.append(np.linalg.norm(a[:min_dim] - b[:min_dim]))
        return 1.0 - np.tanh(np.mean(diffs) * 2.0)

    def predict_future_error(self, state, action, context_state, raw_position=None):
        entity_novelty = np.mean([e.predict(state) * e.utility
                                  for e in self.entities(chain="overworld")]) if self.entities(chain="overworld") else 0.5
        shared_ents = self.entities(chain="shared")
        if shared_ents:
            shared_signal = np.mean([e.predict(state) * e.utility for e in shared_ents])
            entity_novelty = entity_novelty * 0.7 + shared_signal * 0.3

        # Blend with overworld pipeline signal if available
        ow_pipeline = self.overworld_pipeline
        if ow_pipeline.active:
            # Use cached output from last forward pass
            ow_output = ow_pipeline.pools[-2].get_cached_output()  # execution layer
            if np.any(ow_output != 0):
                pipeline_signal = np.mean(np.abs(ow_output))
                entity_novelty = entity_novelty * 0.5 + pipeline_signal * 0.5

        combined = entity_novelty * 0.7 + action.utility * 0.3

        current_map = int(context_state[2])
        loc = self.get_location_key(*(raw_position if raw_position else
                                       (context_state[0]*255, context_state[1]*255)), current_map)

        map_debt = min(self.map_novelty_debt.get(current_map, 0.0), self.MAX_MAP_DEBT)
        loc_debt = min(self.location_novelty.get(loc, 0.0), self.MAX_LOCATION_DEBT)
        total_debt = map_debt + self.get_temp_debt(current_map) + loc_debt * 0.5
        combined *= 1.0 / (1.0 + total_debt * 5.0)

        if action.action == self.current_repeated_action and self.consecutive_action_count > self.LEARNING_SLOWDOWN_START:
            combined *= 1.0 / (1.0 + (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) * 0.15)
        if self.detected_pattern and action.action in self.detected_pattern:
            combined *= 1.0 / (1.0 + self.pattern_repeat_count * 0.2)

        return combined + np.random.randn() * 0.05

    def compute_multi_modal_error(self, state, next_state):
        diffs = [abs(next_state[i] - state[i]) for i in range(min(8, len(state), len(next_state)))]
        weights = [0.5, 0.5, 10.0, 5.0, 3.0, 2.0, 1.5, 0.3]
        weighted = sum(d * w for d, w in zip(diffs, weights)) + np.linalg.norm(next_state[8:] - state[8:]) * 2.0
        numeric = sum(diffs)
        visual = np.linalg.norm(next_state[8:] - state[8:])
        return weighted, numeric, visual

    # =========================================================================
    # PIPELINE LEARNING (NEW) — Generic forward + backward + spawn
    # =========================================================================

    def learn_pipeline(self, pipeline_id, raw_input, prev_raw_input=None,
                       error_scale=1.0, game_state_data=None):
        """
        Generic pipeline learning step:
        1. Forward pass through all layers
        2. Compute error from input change (if prev available)
        3. Backward credit assignment
        4. Spawn into pools that need it

        Args:
            pipeline_id: which pipeline
            raw_input: current raw state for this pipeline's context
            prev_raw_input: previous raw state (None = first frame, skip learning)
            error_scale: multiplier on error signal
            game_state_data: dict for spawn context keying

        Returns:
            (pipeline_output, error, pipeline_active)
        """
        pipeline = self.pipelines.get(pipeline_id)
        if pipeline is None:
            return np.zeros(Pool.DEFAULT_OUTPUT_WIDTH), 0.0, False

        # Forward pass
        output, active = pipeline.forward(raw_input, self.perceptrons)

        if prev_raw_input is None:
            return output, 0.0, active

        # Compute error from state change
        min_dim = min(len(raw_input), len(prev_raw_input))
        error = np.linalg.norm(raw_input[:min_dim] - prev_raw_input[:min_dim]) * error_scale

        # Backward credit assignment
        pipeline.backward(error, self.perceptrons)

        # Spawn into pools that need it
        # Check each layer's pool for spawn conditions
        for i, pool in enumerate(pipeline.pools):
            if pool.needs_spawn(error):
                n_current = pool.get_perceptron_count(self.perceptrons)
                if n_current < pool.max_perceptrons:
                    # Get the input that this layer received during forward pass
                    layer_input = pipeline._layer_inputs[i] if i < len(pipeline._layer_inputs) else raw_input
                    self.spawn_into_pipeline_pool(
                        pipeline_id, i, layer_input,
                        game_state_data=game_state_data,
                    )

        return output, error, active

    # =========================================================================
    # CHAIN-GENERIC LEARNING (existing — kept as fallback)
    # =========================================================================

    def learn_chain(self, chain, learning_state, next_learning_state, error_scale=1.0):
        if learning_state.shape != next_learning_state.shape:
            max_dim = max(len(learning_state), len(next_learning_state))
            learning_state = np.pad(learning_state, (0, max(0, max_dim - len(learning_state))))
            next_learning_state = np.pad(next_learning_state, (0, max(0, max_dim - len(next_learning_state))))

        diff = next_learning_state - learning_state
        error = np.linalg.norm(diff) * error_scale

        chain_history = self.get_chain_error_history(chain)
        chain_history.append(error)

        chain_actions = self.actions(chain=chain)
        chain_entities = self.entities(chain=chain)
        shared_actions = self.actions(chain="shared")
        shared_entities = self.entities(chain="shared")
        all_chain_perceptrons = chain_actions + chain_entities + shared_actions + shared_entities

        for p in all_chain_perceptrons:
            p.update(learning_state, error, stagnation=0.0)

        spawn_threshold = self.get_chain_spawn_threshold(chain, percentile=75)
        if error > spawn_threshold and len(chain_history) >= 50:
            n_ents = self.get_chain_entity_count(chain)
            cap = self.get_chain_entity_capacity(chain)
            if n_ents < cap * 1.5:
                self.spawn_entity_for_chain(chain, learning_state)

        if chain == "battle":
            self.prev_battle_learning_states.append(learning_state)
        elif chain == "party":
            self.prev_party_learning_states.append(learning_state)
        elif chain == "bag":
            self.prev_bag_learning_states.append(learning_state)

        return error

    def learn_battle_chain(self, battle_data, party_data, turn_count):
        """
        Battle chain learning — runs both pipeline AND legacy chain.
        Pipeline gradually takes over as it gains authority.
        """
        current_state = build_learning_state_battle(battle_data, party_data, turn_count)

        if not self.innate_entities_spawned_battle:
            self.spawn_innate_battle_entities(current_state)

        prev_state = self.prev_battle_learning_states[-1] if len(self.prev_battle_learning_states) > 0 else None

        # === Pipeline learning (NEW) ===
        game_data = {
            'enemy_species': battle_data.get('enemy_species', -1),
            'player_species': battle_data.get('player_species', -1),
        }
        pipeline_output, pipeline_error, pipeline_active = self.learn_pipeline(
            "battle", current_state, prev_state,
            error_scale=1.0, game_state_data=game_data
        )

        # === Legacy chain learning (fallback — always runs) ===
        chain_error = 0.0
        if prev_state is not None:
            chain_error = self.learn_chain("battle", prev_state, current_state)
        else:
            self.prev_battle_learning_states.append(current_state)

        # Return blended error (pipeline takes authority as it learns)
        authority = self.battle_pipeline.get_total_authority()
        if authority > 0 and pipeline_error > 0:
            return pipeline_error * authority + chain_error * (1.0 - authority)
        return chain_error

    def learn_party_chain(self, party_data, active_slot=-1):
        """
        Party chain learning — pipeline + legacy.
        """
        current_state = build_learning_state_party(party_data, active_slot)
        prev_state = self.prev_party_learning_states[-1] if len(self.prev_party_learning_states) > 0 else None

        # Pipeline learning
        pipeline_output, pipeline_error, pipeline_active = self.learn_pipeline(
            "party", current_state, prev_state,
            error_scale=0.5
        )

        # Legacy chain learning
        chain_error = 0.0
        if prev_state is not None:
            chain_error = self.learn_chain("party", prev_state, current_state, error_scale=0.5)
        else:
            self.prev_party_learning_states.append(current_state)

        authority = self.party_pipeline.get_total_authority()
        if authority > 0 and pipeline_error > 0:
            return pipeline_error * authority + chain_error * (1.0 - authority)
        return chain_error

    def learn_bag_chain(self, bag_data, party_data, menu_data, in_battle=False):
        """
        Bag chain learning — pipeline + legacy.
        """
        current_state = build_learning_state_bag(bag_data, party_data, menu_data, in_battle)
        prev_state = self.prev_bag_learning_states[-1] if len(self.prev_bag_learning_states) > 0 else None

        # Pipeline learning
        game_data = {'pocket': bag_data.get('pocket', -1)}
        pipeline_output, pipeline_error, pipeline_active = self.learn_pipeline(
            "bag", current_state, prev_state,
            error_scale=0.5, game_state_data=game_data
        )

        # Legacy chain learning
        chain_error = 0.0
        if prev_state is not None:
            chain_error = self.learn_chain("bag", prev_state, current_state, error_scale=0.5)
        else:
            self.prev_bag_learning_states.append(current_state)

        authority = self.bag_pipeline.get_total_authority()
        if authority > 0 and pipeline_error > 0:
            return pipeline_error * authority + chain_error * (1.0 - authority)
        return chain_error

    # =========================================================================
    # MAIN LEARN — OVERWORLD + DISPATCH TO ACTIVE CHAINS
    # =========================================================================

    def learn(self, learning_state, next_learning_state, context_state, next_context_state,
              dead=False, raw_position=None, next_raw_position=None):
        if learning_state.shape != next_learning_state.shape:
            max_dim = max(len(learning_state), len(next_learning_state))
            learning_state = np.pad(learning_state, (0, max(0, max_dim - len(learning_state))))
            next_learning_state = np.pad(next_learning_state, (0, max(0, max_dim - len(next_learning_state))))

        if not self.innate_entities_spawned_overworld:
            self.spawn_innate_overworld_entities(learning_state)

        prev_context = self.prev_context_states[-1] if self.prev_context_states else None
        prev_raw = getattr(self, '_last_raw_position', None)
        self.update_exploration_tracking(context_state, prev_context, raw_position, prev_raw)
        self._last_raw_position = raw_position

        # Track previous game state for start menu detection
        self._prev_game_state_raw = self.game_state_raw

        weighted_error, numeric_error, visual_error = self.compute_multi_modal_error(
            learning_state, next_learning_state)
        self.error_history.append(weighted_error)
        self.numeric_error_history.append(numeric_error)
        self.visual_error_history.append(visual_error)

        # === OVERWORLD PIPELINE LEARNING (NEW) ===
        # Run overworld pipeline when NOT in battle and NOT in menu
        if context_state[3] <= 0.5 and context_state[4] <= 0.5:
            prev_ow = self.prev_learning_states[-1] if len(self.prev_learning_states) > 0 else None
            game_data = {'map_id': int(context_state[2])}
            ow_output, ow_error, ow_active = self.learn_pipeline(
                "overworld", learning_state, prev_ow,
                error_scale=1.0, game_state_data=game_data
            )

        # Overworld entity spawning (legacy)
        spawn_threshold = self.get_spawn_threshold_adaptive('combined', percentile=75)
        if weighted_error > spawn_threshold and len(self.error_history) >= 100:
            menu_active = context_state[4] > 0.5
            battle_active = context_state[3] > 0.5
            if not menu_active and not battle_active:
                self.spawn_entity_for_chain("overworld", learning_state, context_state, raw_position)

        current_map = int(context_state[2])
        loc = self.get_location_key(*(raw_position if raw_position else
                                       (context_state[0]*255, context_state[1]*255)), current_map)

        self.visited_maps[current_map] = self.visited_maps.get(current_map, 0) + 1
        self.location_memory[loc] = self.location_memory.get(loc, 0) + 1

        if self.visited_maps[current_map] > 10:
            self.map_novelty_debt[current_map] = min(self.MAX_MAP_DEBT,
                self.map_novelty_debt.get(current_map, 0.0) + 0.05 * (self.visited_maps[current_map] - 10))
        if self.location_memory[loc] > 15:
            self.location_novelty[loc] = min(self.MAX_LOCATION_DEBT,
                self.location_novelty.get(loc, 0.0) + 0.1 * (self.location_memory[loc] - 15))

        if self.visited_maps[current_map] > 30:
            weighted_error *= 0.5
        if self.location_memory[loc] > 25:
            weighted_error *= 0.7

        stagnation = self.stagnation_level()
        learning_mult = self.get_learning_multiplier(self.last_action) if self.last_action else 1.0
        if self.detected_pattern and self.last_action in self.detected_pattern:
            learning_mult *= 0.5

        # Update overworld + shared perceptrons (legacy flat system)
        overworld_perceptrons = self.actions(chain="overworld") + self.entities(chain="overworld")
        shared_perceptrons = self.actions(chain="shared") + self.entities(chain="shared")

        # Filter out pool-assigned perceptrons — they get updates via pipeline backward
        legacy_ow = [p for p in overworld_perceptrons if p.pool_id is None]
        legacy_shared = [p for p in shared_perceptrons if p.pool_id is None]

        for p in legacy_ow + legacy_shared:
            mult = learning_mult if (p.kind == "action" and p.action == self.last_action) else 1.0
            if p.kind == "action" and self.detected_pattern and p.action in self.detected_pattern:
                mult *= 0.5
            p.update(learning_state, weighted_error * mult, stagnation=stagnation)

        # Dampen Start/Select
        for a in self.actions():
            if a.action in ['Start', 'Select'] and a.weights is not None:
                a.weights *= 0.999

        self.apply_repetition_penalty()
        self.apply_pattern_penalty()
        self.enforce_utility_floors()

        # Movement boost
        if prev_context is not None and np.linalg.norm(context_state[:2] - prev_context[:2]) > 0.001:
            if self.last_action and self.consecutive_action_count < self.PENALTY_THRESHOLD:
                menu_active = context_state[4] > 0.5
                battle_active = context_state[3] > 0.5
                if not menu_active and not battle_active:
                    boost = 1.08
                    if self.nav_active:
                        boost = 1.0 + (0.08 * self.NAV_LEARNING_DAMPENING)
                    elif raw_position and self.is_near_map_edge(*raw_position):
                        boost = 1.15
                    for a in self.actions(chain="overworld") + self.actions(chain="shared"):
                        if a.action == self.last_action:
                            a.utility = min(a.utility * boost, 2.0)
                            break

        # === DISPATCH TO ACTIVE CONTEXT CHAINS ===

        if context_state[3] > 0.5 and self.has_battle_data():
            self.learn_battle_chain(self.battle_data, self.party_data, self.turn_count)

        if self.party_menu_active:
            active_slot = self.get_active_slot_index()
            self.learn_party_chain(self.party_data, active_slot)

        if self.bag_thread_active:
            in_battle = context_state[3] > 0.5
            self.learn_bag_chain(self.bag_data, self.party_data, self.menu_data, in_battle)

        # === REVENGE READINESS CHECK (periodic) ===
        if self.timestep % 100 == 0:
            self.check_revenge_readiness()

        # === PERIODIC CLEANUP (every 1000 steps) ===
        if self.timestep % 1000 == 0 and self.timestep > 0:
            self.cleanup_memory()

        # === PERIODIC SAVES ===
        if self.timestep % self.SAVE_INTERVAL == 0:
            self.save_exploration_memory()
            if self.roster_dirty: self.save_roster()
            if self.move_knowledge_dirty: self.save_move_knowledge()
            if self.item_knowledge_dirty: self.save_item_knowledge()
            if self.type_clusters_dirty: self.save_type_clusters()

        self.action_history.append(self.last_action)

    # =========================================================================
    # STATE LOGGING
    # =========================================================================

    def log_state(self, learning_state, context_state):
        self.prev_learning_states.append(learning_state)
        self.prev_context_states.append(context_state)

    def update_position(self, x, y):
        self.last_positions.append((int(x), int(y)))

    def get_tile_interaction_stats(self, map_id):
        memory = self.get_current_map_memory(map_id)
        ti = memory.get('tile_interactions', {})
        return {
            'probed': len(ti),
            'exhausted': sum(1 for t in ti.values() if t.get('exhausted', False)),
            'with_success': sum(1 for t in ti.values()
                                if any(t.get('direction_successes', {}).get(d, 0) > 0 for d in range(4)))
        }

    # =========================================================================
    # SAVE / LOAD / BOOTSTRAP
    # =========================================================================

    def load_taught_model(self, filepath):
        try:
            with open(filepath, 'r') as f:
                model = json.load(f)

            if "perceptrons" not in model:
                print(f"  ⚠️ Model empty, starting fresh")
                return 0

            for saved_action in model["perceptrons"]["actions"]:
                for a in self.actions():
                    if a.action == saved_action["action"]:
                        a.utility = saved_action["utility"]
                        a.learning_rate = saved_action.get("learning_rate", 0.01)
                        a.familiarity = saved_action.get("familiarity", 0.0)
                        a.chain = saved_action.get("chain", "shared")
                        if saved_action.get("weights_nonzero"):
                            dim = saved_action.get("weights_shape", 1376)
                            a.weights = np.zeros(dim)
                            for idx, val in saved_action["weights_nonzero"]:
                                if idx < dim: a.weights[idx] = val
                        a.set_activation_state(saved_action.get("activation_state"))
                        a.set_pool_state(saved_action.get("pool_state"))
                        break
                    if a.action in ['Start', 'Select'] and a.weights is not None:
                        a.weights = np.zeros(len(a.weights))
                        a.utility = 0.05

            for saved_entity in model["perceptrons"].get("entities", []):
                matched = False
                for e in self.entities():
                    if e.entity_type == saved_entity["entity_type"]:
                        e.utility = saved_entity.get("utility", 1.0)
                        e.familiarity = saved_entity.get("familiarity", 0.0)
                        e.chain = saved_entity.get("chain", "shared")
                        if saved_entity.get("weights_nonzero"):
                            dim = saved_entity.get("weights_shape", 1376)
                            e.weights = np.zeros(dim)
                            for idx, val in saved_entity["weights_nonzero"]:
                                if idx < dim: e.weights[idx] = val
                        e.set_activation_state(saved_entity.get("activation_state"))
                        e.set_pool_state(saved_entity.get("pool_state"))
                        matched = True
                        break

                if not matched and saved_entity.get("weights_nonzero"):
                    chain = saved_entity.get("chain", "shared")
                    entity = Perceptron("entity", entity_type=saved_entity["entity_type"], chain=chain)
                    dim = saved_entity.get("weights_shape", 1376)
                    entity.weights = np.zeros(dim)
                    for idx, val in saved_entity["weights_nonzero"]:
                        if idx < dim: entity.weights[idx] = val
                    entity.utility = saved_entity.get("utility", 1.0)
                    entity.familiarity = saved_entity.get("familiarity", 0.0)
                    entity.set_activation_state(saved_entity.get("activation_state"))
                    entity.set_pool_state(saved_entity.get("pool_state"))
                    self.add(entity)

            if "debt_tracking" in model:
                debt = model["debt_tracking"]
                self.map_novelty_debt = {int(k): v for k, v in debt.get("map_novelty_debt", {}).items()}
                self.visited_maps = {int(k): v for k, v in debt.get("visited_maps", {}).items()}
                for k, v in debt.get("location_novelty", {}).items():
                    self.location_novelty[eval(k)] = v

            if "roster" in model:
                self.roster = {int(k): v for k, v in model["roster"].items()}
                for v in self.roster.values():
                    if isinstance(v.get('fingerprint'), list): v['fingerprint'] = tuple(v['fingerprint'])
                print(f"  📋 Roster restored: {len(self.roster)} slots")

            if "move_knowledge" in model:
                mk_data = model["move_knowledge"]
                self.move_knowledge = {}
                for mk, mv in mk_data.get('player_moves', {}).items():
                    mv['vs_species'] = {int(sk): sv for sk, sv in mv.get('vs_species', {}).items()}
                    self.move_knowledge[int(mk)] = mv
                self.enemy_move_knowledge = {int(k): v for k, v in mk_data.get('enemy_moves', {}).items()}
                print(f"  📖 Moves restored: {len(self.move_knowledge)}")

            if "item_knowledge" in model:
                self.item_knowledge = {int(k): v for k, v in model["item_knowledge"].items()}
                print(f"  🎒 Items restored: {len(self.item_knowledge)}")

            if "map_battle_stats" in model:
                self.load_map_battle_stats(model["map_battle_stats"])

            if "battle_tracking" in model:
                self.battle_low_hp_exits = model["battle_tracking"].get("battle_low_hp_exits", 0)

            if "chain_stats" in model:
                cs = model["chain_stats"]
                self.entity_spawn_counts = cs.get("entity_spawn_counts", self.entity_spawn_counts)
                self.entity_merge_counts = cs.get("entity_merge_counts", self.entity_merge_counts)
                for chain, cap in cs.get("entity_capacities", {}).items():
                    self.ENTITY_CAPACITY[chain] = cap

            # === LOAD TYPE CLUSTERS ===
            if "type_clusters" in model:
                tc = model["type_clusters"]
                self.move_type_clusters = {int(k): v for k, v in tc.get('move_type_clusters', {}).items()}
                self.species_type_clusters = {int(k): v for k, v in tc.get('species_type_clusters', {}).items()}
                self.cluster_effectiveness = tc.get('cluster_effectiveness', {})
                self.move_to_cluster = {int(k): int(v) for k, v in tc.get('move_to_cluster', {}).items()}
                self.species_to_cluster = {int(k): int(v) for k, v in tc.get('species_to_cluster', {}).items()}
                self.clustering_run_count = tc.get('clustering_run_count', 0)
                print(f"  🧬 Type clusters restored: {len(self.move_type_clusters)}mc "
                      f"{len(self.species_type_clusters)}sc {len(self.cluster_effectiveness)}eff")

            # === LOAD PIPELINE STATE (NEW) ===
            if "pipelines" in model:
                for pid, p_state in model["pipelines"].items():
                    pipeline = self.pipelines.get(pid)
                    if pipeline:
                        pipeline.load_save_state(p_state)
                print(f"  🔗 Pipeline state restored")

            # === LOAD REVENGE TARGETS (NEW) ===
            if "revenge_targets" in model:
                self.revenge_targets = model["revenge_targets"]
                active_revenge = sum(1 for t in self.revenge_targets.values()
                                    if t['status'] in ('grinding', 'ready'))
                print(f"  ⚔️🔥 Revenge targets restored: {len(self.revenge_targets)} "
                      f"({active_revenge} active)")

            loaded_ts = model.get("timestep", 0)
            self.timestep = loaded_ts

            chain_stats = self.get_chain_stats()
            if chain_stats:
                parts = [f"{c}:{s['actions']}a+{s['entities']}e" for c, s in chain_stats.items()]
                print(f"  🧬 Chains: {' | '.join(parts)}")

            # Pipeline summary
            p_summary = self.get_pipeline_summary()
            if p_summary != 'all empty':
                print(f"  🔗 Pipelines: {p_summary}")

            activation_counts = {}
            for p in self.perceptrons:
                an = p.active_activation
                activation_counts[an] = activation_counts.get(an, 0) + 1
            if len(activation_counts) > 1:
                parts = [f"{k}:{v}" for k, v in sorted(activation_counts.items(), key=lambda x: x[1], reverse=True)]
                print(f"  🧬 Activations: {' '.join(parts)}")

            return loaded_ts

        except Exception as e:
            print(f"  ⚠️ Error loading model: {e}, starting fresh")
            return 0

    def initialize_from_taught_model(self, filepath):
        if not Path(filepath).exists():
            print(f"  No taught model at {filepath}")
            return 0

        try:
            with open(filepath, 'r') as f:
                model = json.load(f)

            if "perceptrons" not in model:
                print(f"  ⚠️ Taught model empty or invalid")
                return 0

            taught_timestep = model.get("timestep", 0)

            # === 1. LOAD ACTION PERCEPTRONS ===
            actions_loaded = 0
            for saved_action in model["perceptrons"].get("actions", []):
                for a in self.actions():
                    if a.action == saved_action["action"]:
                        a.utility = saved_action["utility"]
                        a.learning_rate = saved_action.get("learning_rate", 0.01)
                        a.familiarity = saved_action.get("familiarity", 0.0)
                        a.chain = saved_action.get("chain", "shared")
                        if saved_action.get("weights_nonzero"):
                            dim = saved_action.get("weights_shape", 1376)
                            a.weights = np.zeros(dim)
                            for idx, val in saved_action["weights_nonzero"]:
                                if idx < dim: a.weights[idx] = val
                        a.set_activation_state(saved_action.get("activation_state"))
                        a.set_pool_state(saved_action.get("pool_state"))
                        actions_loaded += 1
                        break

            # === 2. LOAD ENTITY PERCEPTRONS ===
            entities_loaded = 0
            innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition",
                            "battle_hp_crisis", "battle_enemy_weak", "battle_species_match",
                            "battle_status", "battle_trainer"}

            for saved_entity in model["perceptrons"].get("entities", []):
                et = saved_entity.get("entity_type", "unknown")
                chain = saved_entity.get("chain", "shared")

                if et in innate_types:
                    existing = None
                    for e in self.entities():
                        if e.entity_type == et:
                            existing = e; break
                    if existing:
                        existing.utility = saved_entity.get("utility", 1.0)
                        existing.familiarity = saved_entity.get("familiarity", 0.0)
                        existing.learning_rate = saved_entity.get("learning_rate", 0.01)
                        existing.chain = chain
                        if saved_entity.get("weights_nonzero"):
                            dim = saved_entity.get("weights_shape", 1376)
                            existing.ensure_weights(dim)
                            existing.weights = np.zeros(dim)
                            for idx, val in saved_entity["weights_nonzero"]:
                                if idx < dim: existing.weights[idx] = val
                        existing.set_activation_state(saved_entity.get("activation_state"))
                        existing.set_pool_state(saved_entity.get("pool_state"))
                        entities_loaded += 1
                        continue

                if saved_entity.get("weights_nonzero"):
                    entity = Perceptron("entity", entity_type=et, chain=chain)
                    dim = saved_entity.get("weights_shape", 1376)
                    entity.weights = np.zeros(dim)
                    for idx, val in saved_entity["weights_nonzero"]:
                        if idx < dim: entity.weights[idx] = val
                    entity.utility = saved_entity.get("utility", 1.0)
                    entity.familiarity = saved_entity.get("familiarity", 0.0)
                    entity.learning_rate = saved_entity.get("learning_rate", 0.01)
                    entity.set_activation_state(saved_entity.get("activation_state"))
                    entity.set_pool_state(saved_entity.get("pool_state"))
                    self.add(entity)
                    entities_loaded += 1

            # === 3. SET INNATE FLAGS ===
            ow_innate = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition"}
            bat_innate = {"battle_hp_crisis", "battle_enemy_weak", "battle_species_match",
                          "battle_status", "battle_trainer"}
            restored_types = {se.get("entity_type") for se in model["perceptrons"].get("entities", [])}
            if ow_innate.issubset(restored_types):
                self.innate_entities_spawned_overworld = True
                self.innate_entities_spawned = True
            if bat_innate.issubset(restored_types):
                self.innate_entities_spawned_battle = True

            # === 4. LOAD CHAIN STATS ===
            cs = model.get("chain_stats", {})
            if cs:
                self.entity_spawn_counts = cs.get("entity_spawn_counts", self.entity_spawn_counts)
                self.entity_merge_counts = cs.get("entity_merge_counts", self.entity_merge_counts)
                for chain, cap in cs.get("entity_capacities", {}).items():
                    self.ENTITY_CAPACITY[chain] = cap

            # === 5. LOAD MAP BATTLE STATS ===
            if "map_battle_stats" in model:
                self.load_map_battle_stats(model["map_battle_stats"])

            # === 6. LOAD TYPE CLUSTERS ===
            if "type_clusters" in model:
                tc = model["type_clusters"]
                self.move_type_clusters = {int(k): v for k, v in tc.get('move_type_clusters', {}).items()}
                self.species_type_clusters = {int(k): v for k, v in tc.get('species_type_clusters', {}).items()}
                self.cluster_effectiveness = tc.get('cluster_effectiveness', {})
                self.move_to_cluster = {int(k): int(v) for k, v in tc.get('move_to_cluster', {}).items()}
                self.species_to_cluster = {int(k): int(v) for k, v in tc.get('species_to_cluster', {}).items()}
                self.clustering_run_count = tc.get('clustering_run_count', 0)
                print(f"  🧬 Type clusters bootstrapped: {len(self.move_type_clusters)}mc "
                      f"{len(self.species_type_clusters)}sc")

            # === 7. LOAD PIPELINE STATE (NEW) ===
            if "pipelines" in model:
                for pid, p_state in model["pipelines"].items():
                    pipeline = self.pipelines.get(pid)
                    if pipeline:
                        pipeline.load_save_state(p_state)

            # === 8. AI TIMESTEP STARTS AT 0 ===
            self.timestep = 0

            # === LOG ===
            chain_stats = self.get_chain_stats()
            chain_parts = [f"{c}:{s['actions']}a+{s['entities']}e" for c, s in chain_stats.items()]
            act_counts = {}
            for p in self.perceptrons:
                act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
            act_parts = [f"{k}:{v}" for k, v in sorted(act_counts.items(), key=lambda x: x[1], reverse=True)]

            print(f"  🎓 BOOTSTRAPPED FROM TAUGHT MODEL:")
            print(f"     Source: {filepath}")
            print(f"     Human played: {taught_timestep} steps")
            print(f"     Actions loaded: {actions_loaded}")
            print(f"     Entities loaded: {entities_loaded}")
            print(f"     Chains: {' | '.join(chain_parts)}")
            print(f"     Pipelines: {self.get_pipeline_summary()}")
            print(f"     Activations: {' '.join(act_parts)}")
            print(f"     Utilities: {', '.join(f'{a.action}:{a.utility:.3f}' for a in self.actions())}")
            print(f"     AI timestep: 0 (fresh start with human's patterns)")

            return taught_timestep

        except Exception as e:
            print(f"  ⚠️ Error bootstrapping from taught model: {e}")
            return 0

    def merge_taught_exploration(self, taught_filepath):
        if not Path(taught_filepath).exists():
            print(f"  No taught exploration at {taught_filepath}")
            return
        with open(taught_filepath, 'r') as f:
            taught_data = json.load(f)
        ta, ia = 0, 0
        for map_key, taught_map in taught_data.items():
            map_id = int(map_key.replace('map_', ''))
            ai_map = self.get_current_map_memory(map_id)
            for tt in taught_map.get('transitions', []):
                tp, td = tuple(tt['position']), tt['direction']
                if not any(tuple(e['position']) == tp and e['direction'] == td for e in ai_map['transitions']):
                    ai_map['transitions'].append(tt); ta += 1
            for ti in taught_map.get('interactable_objects', []):
                if ti not in ai_map['interactable_objects']:
                    ai_map['interactable_objects'].append(ti); ia += 1
        print(f"  Merged: {ta} transitions, {ia} interactables")

    def save_model_checkpoint(self, filepath):
        model = {
            "timestep": self.timestep,
            "perceptrons": {"actions": [], "entities": []},
            "debt_tracking": {
                "map_novelty_debt": {str(k): v for k, v in self.map_novelty_debt.items()},
                "location_novelty": {str(k): v for k, v in self.location_novelty.items()},
                "visited_maps": {str(k): v for k, v in self.visited_maps.items()}
            },
            "control_mode": self.control_mode,
            "markov_stats": {
                "markov_action_count": self.markov_action_count,
                "curiosity_action_count": self.curiosity_action_count
            },
            "blend_stats": {"blend_count": self.blend_count, "last_blend_tier": self.blend_tier},
            "battle_stats": {
                "battle_action_count": self.battle_action_count,
                "battle_markov_action_count": self.battle_markov_action_count,
                "current_battle_id": self.current_battle_id
            },
            "bag_stats": {
                "bag_thread_total_actions": self.bag_thread_total_actions,
                "bag_thread_markov_actions": self.bag_thread_markov_actions
            },
            "prep_stats": {
                "prep_total_count": self.prep_total_count,
                "prep_success_count": self.prep_success_count
            },
            "start_menu_stats": {
                "start_menu_total_actions": self.start_menu_total_actions,
                "start_menu_markov_actions": self.start_menu_markov_actions
            },
            "chain_stats": {
                "entity_spawn_counts": self.entity_spawn_counts,
                "entity_merge_counts": self.entity_merge_counts,
                "entity_capacities": dict(self.ENTITY_CAPACITY),
            },
            "roster": {},
            "move_knowledge": {"player_moves": {}, "enemy_moves": {}},
            "item_knowledge": {},
            "map_battle_stats": {},
            "battle_tracking": {"battle_low_hp_exits": self.battle_low_hp_exits},
            "type_clusters": {
                "move_type_clusters": {str(k): v for k, v in self.move_type_clusters.items()},
                "species_type_clusters": {str(k): v for k, v in self.species_type_clusters.items()},
                "cluster_effectiveness": self.cluster_effectiveness,
                "move_to_cluster": {str(k): v for k, v in self.move_to_cluster.items()},
                "species_to_cluster": {str(k): v for k, v in self.species_to_cluster.items()},
                "clustering_run_count": self.clustering_run_count,
            },
            # === PIPELINE STATE (NEW) ===
            "pipelines": {pid: p.get_save_state() for pid, p in self.pipelines.items()},
            # === REVENGE TARGETS (NEW) ===
            "revenge_targets": self.revenge_targets,
        }

        # Roster
        for k, v in self.roster.items():
            e = v.copy()
            if isinstance(e.get('fingerprint'), tuple): e['fingerprint'] = list(e['fingerprint'])
            model["roster"][str(k)] = e

        # Move knowledge
        for mk, mv in self.move_knowledge.items():
            c = mv.copy(); c['vs_species'] = {str(sk): sv for sk, sv in mv.get('vs_species', {}).items()}
            model["move_knowledge"]["player_moves"][str(mk)] = c
        for ek, ev in self.enemy_move_knowledge.items():
            model["move_knowledge"]["enemy_moves"][str(ek)] = ev

        # Item knowledge
        model["item_knowledge"] = {str(k): v for k, v in self.item_knowledge.items()}

        # Map battle stats
        model["map_battle_stats"] = self.get_map_battle_stats_for_save()

        # Perceptrons with chain + activation + pool state
        for a in self.actions():
            model["perceptrons"]["actions"].append({
                "action": a.action, "group": a.group, "chain": a.chain,
                "utility": float(a.utility),
                "weights_shape": len(a.weights) if a.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(a.weights)
                                     if abs(v) > 1e-10] if a.weights is not None else [],
                "learning_rate": float(a.learning_rate),
                "familiarity": float(a.familiarity),
                "activation_state": a.get_activation_state(),
                "pool_state": a.get_pool_state(),
            })

        for e in self.entities():
            model["perceptrons"]["entities"].append({
                "entity_type": e.entity_type, "chain": e.chain,
                "utility": float(e.utility),
                "weights_shape": len(e.weights) if e.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(e.weights)
                                     if abs(v) > 1e-10] if e.weights is not None else [],
                "familiarity": float(e.familiarity),
                "activation_state": e.get_activation_state(),
                "pool_state": e.get_pool_state(),
            })

        with open(filepath, 'w') as f:
            json.dump(model, f, indent=2)

        # Save auxiliary files
        if self.roster_dirty: self.save_roster()
        if self.move_knowledge_dirty: self.save_move_knowledge()
        if self.item_knowledge_dirty: self.save_item_knowledge()
        if self.type_clusters_dirty: self.save_type_clusters()

        # Save residual file (NEW)
        self.save_residual_file()

In [ ]:
# ============================================================================
# CELL 4: Action Selection - Party Menu + Dialogue + Start Menu + Bag +
#          Prep + Battle + Overworld
# ============================================================================
# CHANGES from v17.3:
# 1. UPDATED: battle_action() — queries battle pipeline execution layer
#    when pipeline has authority, blends with existing logic
# 2. UPDATED: bag_action() — queries bag pipeline when available
# 3. UPDATED: anticipatory_action() overworld section — queries overworld
#    pipeline execution layer, blends with existing scoring
# 4. NEW: _pipeline_action_override() — helper to interpret pipeline
#    execution layer output as action preference
# 5. All dialogue, party menu, start menu, prep handlers UNCHANGED
# ============================================================================

import random

GBA_ACTIONS = ["Up", "Down", "Left", "Right", "A", "B", "Start", "Select"]
ACTION_DELTAS = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}
DIRECTION_TO_ACTION = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
ACTION_TO_DIRECTION = {"DOWN": 0, "UP": 1, "LEFT": 2, "RIGHT": 3}

BATTLE_CURSOR_FIGHT = 0
BATTLE_CURSOR_BAG = 1
BATTLE_CURSOR_POKEMON = 2
BATTLE_CURSOR_RUN = 3

# Action name to pipeline output dimension mapping
# Pipeline execution layers output 8 dims; first 8 map to actions:
# [0]=UP, [1]=DOWN, [2]=LEFT, [3]=RIGHT, [4]=A, [5]=B, [6]=Start, [7]=Select
PIPELINE_ACTION_MAP = ["UP", "DOWN", "LEFT", "RIGHT", "A", "B", "Start", "Select"]


def manhattan_distance(pos1, pos2):
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])


def _get_action_perceptron(actions_list, action_name):
    for a in actions_list:
        if a.action == action_name: return a
    return None


def _record_battle_action(brain, a):
    brain.record_action_execution(a.action)
    brain.track_consecutive_action(a.action)
    brain.battle_action_history.append(a.action)
    brain.set_party_menu_last_action(a.action)
    return a

def _record_party_menu_action(brain, a):
    brain.record_action_execution(a.action)
    brain.track_consecutive_action(a.action)
    brain.party_menu_action_count += 1
    brain.set_party_menu_last_action(a.action)
    if brain.party_menu_context in ("battle_forced", "battle_voluntary"):
        brain.battle_action_history.append(a.action)
        brain.battle_action_count += 1
    return a

def _record_bag_action(brain, a):
    brain.record_action_execution(a.action)
    brain.track_consecutive_action(a.action)
    brain.bag_thread_action_count += 1
    brain.bag_thread_total_actions += 1
    brain.bag_action_history.append(a.action)
    brain.set_bag_thread_last_action(a.action)
    if brain.bag_thread_context == "battle":
        brain.battle_action_history.append(a.action)
        brain.battle_action_count += 1
    return a

def _record_preparation_action(brain, a):
    brain.record_action_execution(a.action)
    brain.track_consecutive_action(a.action)
    return a

def _record_start_menu_action(brain, a):
    brain.record_action_execution(a.action)
    brain.track_consecutive_action(a.action)
    brain.start_menu_action_count += 1
    brain.start_menu_total_actions += 1
    brain.start_menu_action_history.append(a.action)
    brain.set_start_menu_last_action(a.action)
    return a

def _record_dialogue_action(brain, a, is_choice=False):
    brain.record_action_execution(a.action)
    brain.dialogue_last_action = a.action

    if is_choice:
        brain.track_consecutive_action(a.action)
        brain.dialogue_choice_action_count += 1
    else:
        brain.dialogue_skip_action_count += 1

    return a


# ============================================================================
# PIPELINE ACTION INTERPRETATION (NEW)
# ============================================================================

def _pipeline_action_override(brain, pipeline_id, layer_name, raw_input,
                               actions_list, min_authority=0.15):
    """
    Query a pipeline's execution layer for action preference.

    Interprets the pool output as action scores:
    dims 0-7 map to UP, DOWN, LEFT, RIGHT, A, B, Start, Select

    Returns: (action_perceptron, authority) or (None, 0.0) if pipeline
    doesn't have sufficient authority or is empty.
    """
    pipeline = brain.pipelines.get(pipeline_id)
    if pipeline is None:
        return None, 0.0

    # Find the execution layer by name
    exec_pool = None
    exec_idx = -1
    for i, pool in enumerate(pipeline.pools):
        if pool.name == layer_name:
            exec_pool = pool
            exec_idx = i
            break

    if exec_pool is None:
        return None, 0.0

    # Check authority
    if exec_pool.authority < min_authority:
        return None, 0.0

    # Run forward pass up to execution layer
    current_input = raw_input
    for i in range(exec_idx + 1):
        pool = pipeline.pools[i]
        current_input = pool.compute_output(current_input, brain.perceptrons)

    # Interpret output as action scores
    output = current_input
    best_action_name = None
    best_score = -float('inf')

    for dim_idx, action_name in enumerate(PIPELINE_ACTION_MAP):
        if dim_idx < len(output):
            score = float(output[dim_idx])
            if score > best_score:
                best_score = score
                best_action_name = action_name

    if best_action_name is None:
        return None, 0.0

    a = _get_action_perceptron(actions_list, best_action_name)
    return a, exec_pool.authority


# ============================================================================
# GRID + CURSOR NAVIGATION
# ============================================================================

def _navigate_2x2(current, target):
    if current == target: return "A"
    cr, cc = current // 2, current % 2
    tr, tc = target // 2, target % 2
    if cr < tr: return "DOWN"
    elif cr > tr: return "UP"
    elif cc < tc: return "RIGHT"
    elif cc > tc: return "LEFT"
    return "A"

def _navigate_party_cursor(current, target):
    if current == target: return "A"
    return "DOWN" if current < target else "UP"

def _navigate_vertical_cursor(current, target):
    if current == target: return "A"
    return "DOWN" if current < target else "UP"


# ============================================================================
# TEXT DIALOGUE ACTION — Pure text advancement (no utility tracking)
# ============================================================================

def text_dialogue_action(brain, context_state):
    actions_list = brain.actions()

    if brain.dialogue_last_action == "A":
        action_name = "B" if random.random() < 0.3 else "A"
    else:
        action_name = "A"

    a = _get_action_perceptron(actions_list, action_name)
    if a:
        return _record_dialogue_action(brain, a, is_choice=False)

    a = _get_action_perceptron(actions_list, "A")
    return _record_dialogue_action(brain, a, is_choice=False) if a else actions_list[0]


# ============================================================================
# DIALOGUE CHOICE ACTION — YES/NO prompts (real decision, normal tracking)
# ============================================================================

def dialogue_choice_action(brain, context_state):
    actions_list = brain.actions()
    mc = brain.menu_data.get('mc', -1)
    mm = brain.menu_data.get('mm', -1)

    if brain.markov_enabled:
        matched, action_name, score = brain.get_markov_action(context_state)
        if matched and action_name and action_name in ("A", "B", "UP", "DOWN"):
            a = _get_action_perceptron(actions_list, action_name)
            if a:
                brain.markov_action_count += 1
                return _record_dialogue_action(brain, a, is_choice=True)

    a = _get_action_perceptron(actions_list, "A")
    if a:
        return _record_dialogue_action(brain, a, is_choice=True)

    return actions_list[0]


# ============================================================================
# PARTY MENU ACTION
# ============================================================================

def party_menu_action(brain, context_state):
    actions_list = brain.actions()
    context = brain.party_menu_context
    pc = brain.battle_data.get('party_cursor', -1)

    if context == "battle_forced":
        target = brain.party_menu_target_slot
        if target < 0:
            target = brain.get_best_switch_slot()
            brain.party_menu_target_slot = target
            brain.forced_switch_target_slot = target
            if target >= 0: print(f"  🔄 FORCED SWITCH → slot {target}")
            else:
                a = _get_action_perceptron(actions_list, "B")
                return _record_party_menu_action(brain, a) if a else actions_list[0]
        if target >= 0 and 0 <= pc <= 5:
            nav = _navigate_party_cursor(pc, target)
            if nav == "A":
                brain.forced_switch_pending = False
                brain.forced_switch_target_slot = -1
            a = _get_action_perceptron(actions_list, nav)
            return _record_party_menu_action(brain, a) if a else actions_list[0]
        a = _get_action_perceptron(actions_list, "A")
        return _record_party_menu_action(brain, a) if a else actions_list[0]

    a = _get_action_perceptron(actions_list, "B")
    return _record_party_menu_action(brain, a) if a else actions_list[0]


# ============================================================================
# START MENU ACTION — Markov + Targeted Navigation + Fallback B
# ============================================================================

def start_menu_action(brain, context_state):
    actions_list = brain.actions()
    md = brain.menu_data
    mc = md.get('mc', -1)
    target_mc = brain.start_menu_target_mc

    matched, action_name, score = brain.get_start_menu_markov_action()
    if matched and action_name:
        if action_name in ("START", "SELECT"):
            action_name = "A" if mc == target_mc else "B"
        brain.start_menu_markov_actions += 1
        brain.last_start_menu_markov_action = action_name
        a = _get_action_perceptron(actions_list, action_name)
        if a:
            return _record_start_menu_action(brain, a)

    if target_mc >= 0 and 0 <= mc <= 6:
        if mc == target_mc:
            a = _get_action_perceptron(actions_list, "A")
            if a:
                return _record_start_menu_action(brain, a)
        else:
            nav = _navigate_vertical_cursor(mc, target_mc)
            a = _get_action_perceptron(actions_list, nav)
            if a:
                return _record_start_menu_action(brain, a)

    if target_mc < 0:
        a = _get_action_perceptron(actions_list, "B")
        if a:
            return _record_start_menu_action(brain, a)

    a = _get_action_perceptron(actions_list, "B")
    return _record_start_menu_action(brain, a) if a else actions_list[0]


# ============================================================================
# BAG ACTION — Pipeline + Markov + chain entity signal + fallback B
# ============================================================================

def bag_action(brain, context_state):
    actions_list = brain.actions()
    md = brain.menu_data

    prev_mc = brain.prev_menu_data.get('mc', -1)
    prev_action = brain.bag_thread_last_action
    if prev_action == "A" and prev_mc == 0:
        prev_items = brain.prev_bag_data.get('items', [])
        prev_cursor = brain.prev_bag_data.get('cursor', -1)
        item_id = prev_items[prev_cursor].get('id', -1) if 0 <= prev_cursor < len(prev_items) else -1
        if item_id > 0 and brain.pending_item_observation is None:
            brain.start_item_observation(item_id, target_slot=md.get('pc', -1))

    # === BAG PIPELINE QUERY (NEW) ===
    # If bag pipeline has learned, use its execution layer
    bag_state = build_learning_state_bag(brain.bag_data, brain.party_data,
                                          brain.menu_data,
                                          context_state[3] > 0.5)
    pipeline_action, pipeline_auth = _pipeline_action_override(
        brain, "bag", "execution", bag_state, actions_list, min_authority=0.2
    )
    if pipeline_action is not None:
        # Pipeline suggests an action — use it with probability proportional to authority
        if random.random() < pipeline_auth:
            return _record_bag_action(brain, pipeline_action)

    # === MARKOV FALLBACK ===
    matched, action_name, score = brain.get_bag_markov_action()
    if matched and action_name:
        if action_name in ("START", "SELECT"): action_name = "B"
        brain.bag_thread_markov_actions += 1
        brain.last_bag_markov_action = action_name
        a = _get_action_perceptron(actions_list, action_name)
        if a: return _record_bag_action(brain, a)

    if brain.get_lowest_hp_ratio() < 0.5:
        bag_signal = brain.get_chain_entity_signal("bag", bag_state)
        healing = brain.get_healing_items()
        if healing and bag_signal > 0.3:
            pass

    a = _get_action_perceptron(actions_list, "B")
    return _record_bag_action(brain, a) if a else actions_list[0]


# ============================================================================
# PREPARATION ACTION
# ============================================================================

def preparation_action(brain, context_state):
    actions_list = brain.actions()
    action_name = brain.get_preparation_action()

    if action_name is None:
        if brain.is_start_menu_active() and brain.start_menu_context == "preparation":
            return start_menu_action(brain, context_state)
        a = _get_action_perceptron(actions_list, "B")
        return _record_preparation_action(brain, a) if a else actions_list[0]

    a = _get_action_perceptron(actions_list, action_name)
    return _record_preparation_action(brain, a) if a else actions_list[0]


# ============================================================================
# BATTLE ACTION — Pipeline + Smart Moves + Battle Chain Entity Signal
# ============================================================================

def battle_action(brain, context_state, palette_state=None):
    actions_list = brain.actions()
    brain.battle_frame_count += 1
    brain.battle_action_count += 1

    bd = brain.battle_data
    has_data = brain.has_battle_data()

    if has_data:
        menu_state = brain.infer_battle_menu_state()
        brain.battle_menu_state = menu_state
        bc, mc = bd['battle_cursor'], bd['move_cursor']
        want_run = brain.should_run()

        # === BATTLE PIPELINE QUERY (NEW) ===
        # When pipeline has authority AND we're not in a clear cursor state,
        # let pipeline suggest the action
        battle_state = build_learning_state_battle(bd, brain.party_data, brain.turn_count)
        pipeline_action, pipeline_auth = _pipeline_action_override(
            brain, "battle", "execution", battle_state, actions_list, min_authority=0.2
        )

        if menu_state == "main_menu" and 0 <= bc <= 3:
            brain.battle_cursor_action_count += 1

            # Pipeline can influence main menu decision
            if pipeline_action is not None and pipeline_auth > 0.3:
                # Pipeline suggests something — check if it's a valid cursor nav
                if pipeline_action.action in ("UP", "DOWN", "LEFT", "RIGHT", "A"):
                    # High authority: use pipeline
                    if random.random() < pipeline_auth * 0.5:
                        return _record_battle_action(brain, pipeline_action)

            # Existing logic: navigate to target
            target = BATTLE_CURSOR_RUN if want_run else BATTLE_CURSOR_FIGHT
            nav = _navigate_2x2(bc, target)
            a = _get_action_perceptron(actions_list, nav)
            if a: return _record_battle_action(brain, a)

        if menu_state == "move_select" and 0 <= mc <= 3:
            brain.battle_cursor_action_count += 1
            target_slot = 0

            enemy_species = bd.get('enemy_species', -1)
            if enemy_species > 0:
                ranked = brain.get_best_move_for_enemy(enemy_species)
                if ranked and ranked[0][2] > 1.5:
                    target_slot = ranked[0][1]
                elif ranked:
                    battle_signal = brain.get_chain_entity_signal("battle", battle_state)

                    if battle_signal > 0.4 and len(ranked) > 1:
                        if random.random() < 0.3:
                            target_slot = ranked[1][1]
                        else:
                            target_slot = ranked[0][1]
                    else:
                        target_slot = ranked[0][1]
                else:
                    available = brain.get_moves_with_pp()
                    if available: target_slot = available[0][0]
            else:
                available = brain.get_moves_with_pp()
                if available: target_slot = available[0][0]

            nav = _navigate_2x2(mc, target_slot)
            if nav == "A":
                brain.last_move_slot = target_slot
                brain.last_move_used = bd.get(['move0','move1','move2','move3'][target_slot], -1)
            a = _get_action_perceptron(actions_list, nav)
            if a: return _record_battle_action(brain, a)

        # === Pipeline fallback for non-cursor states (NEW) ===
        # When not in a recognized cursor state, pipeline might know what to do
        if pipeline_action is not None and pipeline_auth > 0.25:
            if random.random() < pipeline_auth:
                return _record_battle_action(brain, pipeline_action)

    # === MARKOV FALLBACK ===
    matched, action_name, score = brain.get_battle_markov_action(context_state, palette_state)
    if matched and action_name:
        if action_name in ("START", "SELECT"): action_name = "A"
        brain.battle_markov_action_count += 1
        brain.last_battle_markov_action = action_name
        a = _get_action_perceptron(actions_list, action_name)
        if a: return _record_battle_action(brain, a)

    a = _get_action_perceptron(actions_list, "A")
    return _record_battle_action(brain, a) if a else actions_list[0]


# ============================================================================
# CURIOSITY OVERRIDE (overworld chain entities + pipeline)
# ============================================================================

def _check_curiosity_override(brain, learning_state, context_state, raw_position, map_density):
    raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
    raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
    current_map = int(context_state[2])

    memory = brain.get_current_map_memory(current_map)
    if (raw_x, raw_y) not in memory['visited_tiles']: return True

    if brain.should_interact_at_tile(raw_x, raw_y, current_map):
        if brain.get_untried_directions(raw_x, raw_y, current_map): return True

    if context_state[3] <= 0.5 and context_state[4] <= 0.5:
        ow_signal = brain.get_chain_entity_signal("overworld", learning_state)
        shared_signal = brain.get_chain_entity_signal("shared", learning_state)
        entity_signal = ow_signal * 0.7 + shared_signal * 0.3

        # Blend with overworld pipeline signal if available (NEW)
        ow_pipeline = brain.overworld_pipeline
        if ow_pipeline.active:
            # Use frontier detection layer output as novelty signal
            frontier_pool = ow_pipeline.pools[2]  # L2: frontier_detection
            if frontier_pool.authority > 0.1:
                frontier_output = frontier_pool.get_cached_output()
                pipeline_novelty = np.mean(np.abs(frontier_output))
                entity_signal = entity_signal * 0.6 + pipeline_novelty * 0.4

        density = map_density or {'tier': 'medium'}
        threshold = {'sparse': 0.15, 'thin': 0.25, 'medium': 0.35, 'dense': 0.45}.get(
            density['tier'], 0.35)
        if entity_signal > threshold: return True

    return False


# ============================================================================
# MAIN ACTION SELECTION
# ============================================================================

def anticipatory_action(brain, learning_state, context_state,
                       exploration_weight=1.3, min_interact_prob=0.15,
                       raw_position=None, forced_explore_prob=0.18,
                       override_threshold=1.5, taught_frames=None,
                       map_density=None, palette_state=None):
    """
    Priority: PartyMenu → Dialogue → StartMenu → Bag → Battle → Prep → Overworld
    """
    actions_list = brain.actions()
    if not actions_list:
        return Perceptron("action", action="UP", group="move")

    # === 0. PARTY MENU ===
    if brain.is_party_menu_active():
        return party_menu_action(brain, context_state)

    # === 0.3. TEXT DIALOGUE (pure text or battle text — skip with A/B) ===
    if brain.is_dialogue_skip_state():
        return text_dialogue_action(brain, context_state)

    # === 0.4. DIALOGUE CHOICE (YES/NO prompt — real decision) ===
    if brain.is_dialogue_choice_state():
        return dialogue_choice_action(brain, context_state)

    # === 0.5. START MENU (when active independently, not via prep) ===
    if brain.is_start_menu_active() and not brain.is_preparation_active():
        return start_menu_action(brain, context_state)

    # === 1. BAG ===
    if brain.is_bag_thread_active():
        return bag_action(brain, context_state)

    # === 2. BATTLE ===
    if context_state[3] > 0.5:
        return battle_action(brain, context_state, palette_state)

    # === 3. PREPARATION ===
    if brain.is_preparation_active():
        brain.update_preparation_state(context_state)
        if brain.is_preparation_active():
            return preparation_action(brain, context_state)

    # === OVERWORLD ===
    density = map_density or {'taught_frames': 0, 'tier': 'sparse', 'coverage': 0.0, 'visited': 0}
    tier = density['tier']

    markov_threshold = {'sparse': 0.72, 'thin': 0.65, 'medium': 0.58, 'dense': 0.50}.get(
        tier, MARKOV_FAMILIARITY_THRESHOLD)
    adapted_explore_prob = {'sparse': 0.30, 'thin': 0.24, 'medium': 0.18, 'dense': 0.12}.get(
        tier, forced_explore_prob)
    adapted_exploration_weight = {'sparse': 1.8, 'thin': 1.5, 'medium': 1.3, 'dense': 1.1}.get(
        tier, exploration_weight)
    transition_weight_mult = {'sparse': 0.3, 'thin': 0.6, 'medium': 1.0, 'dense': 1.4}.get(tier, 1.0)

    raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
    raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
    current_map = int(context_state[2])
    current_dir = int(context_state[5])
    current_pos = (raw_x, raw_y)

    # === 4. PREP TRIGGER ===
    if context_state[3] <= 0.5 and context_state[4] <= 0.5:
        should, reason, target = brain.should_prepare(raw_x, raw_y, current_map)
        if should:
            brain.start_preparation(reason, target)
            if brain.is_preparation_active():
                return preparation_action(brain, context_state)

    # === 5. FORCED RANDOM ===
    brain.check_state_stagnation(context_state)
    if brain.should_force_random():
        if brain.is_nav_active(): brain.abort_navigation("forced random")
        forced_name = brain.get_forced_random_action_name()
        for a in actions_list:
            if a.action == forced_name:
                brain.curiosity_action_count += 1
                brain.record_action_execution(a.action)
                brain.track_consecutive_action(a.action)
                return a

    # === 6. NAVIGATION ===
    if context_state[3] <= 0.5 and context_state[4] <= 0.5:
        brain.update_known_area_counter(raw_x, raw_y, current_map)

    if not brain.is_nav_active() and brain.should_start_navigation():
        if context_state[3] <= 0.5 and context_state[4] <= 0.5:
            if brain.start_navigation(current_pos, current_map):
                brain.known_area_counter = 0

    if brain.is_nav_active():
        if brain.is_nav_paused():
            brain.update_nav_state(current_pos, current_map)
        else:
            if _check_curiosity_override(brain, learning_state, context_state, raw_position, map_density):
                if not brain.nav_map_chain:
                    brain.abort_navigation("curiosity override")
            else:
                if brain.update_nav_state(current_pos, current_map):
                    nav_action = brain.get_nav_action(current_pos)
                    if nav_action:
                        for a in actions_list:
                            if a.action == nav_action:
                                brain.curiosity_action_count += 1
                                brain.record_action_execution(a.action)
                                brain.track_consecutive_action(a.action)
                                return a
                    elif not brain.nav_paused:
                        brain.abort_navigation("path invalid")

    if brain.is_in_nav_curiosity_window():
        if brain.tick_nav_curiosity_window():
            memory = brain.get_current_map_memory(current_map)
            tile_probing = brain.should_interact_at_tile(raw_x, raw_y, current_map)
            brain.complete_nav_target(tile_probing or current_pos not in memory['visited_tiles'])

    # === 6.5. OVERWORLD PIPELINE QUERY (NEW) ===
    # If overworld pipeline has authority, let it influence action selection
    ow_pipeline_action, ow_pipeline_auth = _pipeline_action_override(
        brain, "overworld", "execution", learning_state, actions_list, min_authority=0.2
    )

    # === 7. MARKOV ===
    if brain.markov_enabled and taught_frames:
        score, action, idx = brain.compute_markov_similarity(context_state, raw_position, taught_frames=taught_frames)
        brain.last_markov_score = score
        if score >= markov_threshold and action:
            brain.last_markov_action = action
            for a in actions_list:
                if a.action == action:
                    brain.markov_action_count += 1
                    brain.record_action_execution(a.action)
                    brain.track_consecutive_action(a.action)
                    if a.action == 'A' and brain.should_interact_at_tile(raw_x, raw_y, current_map):
                        brain.start_interaction_verification(raw_x, raw_y, current_map, current_dir)
                    return a

    # === 8. CURIOSITY ===
    brain.curiosity_action_count += 1
    mode = brain.determine_control_mode(context_state, raw_position=raw_position)

    memory = brain.get_current_map_memory(current_map)
    visited_tiles = memory['visited_tiles']
    obstructions = memory['obstructions']
    tile_probing = brain.should_interact_at_tile(raw_x, raw_y, current_map)
    probe_action, probe_dir = brain.get_best_probe_action(raw_x, raw_y, current_map, current_dir)
    trans_attract, best_trans = brain.get_transition_attraction(current_map)
    coverage = brain.get_exploration_coverage(current_map)

    if random.random() < adapted_explore_prob:
        valid = [a for a in actions_list if a.action not in ('Start', 'Select')]
        chosen = random.choice(valid)
        brain.record_action_execution(chosen.action)
        brain.track_consecutive_action(chosen.action)
        if chosen.action == 'A' and tile_probing:
            brain.start_interaction_verification(raw_x, raw_y, current_map, current_dir)
        return chosen

    action_scores = {}
    for a in actions_list:
        if a.action in ('Start', 'Select'):
            action_scores[a.action] = (a, 0.0); continue

        predicted = brain.predict_future_error(learning_state, a, context_state, raw_position=raw_position)

        if a.group == "move":
            predicted *= adapted_exploration_weight
            dx, dy = ACTION_DELTAS.get(a.action, (0, 0))
            target_tile = (raw_x + dx, raw_y + dy)
            action_dir = ACTION_TO_DIRECTION.get(a.action, -1)

            if target_tile not in visited_tiles: predicted *= brain.UNVISITED_TILE_BONUS
            if target_tile in obstructions: predicted *= brain.OBSTRUCTION_PENALTY
            if brain.is_position_banned(current_map, raw_x, raw_y, action_dir): predicted *= 0.05

            if trans_attract > 0.3 and best_trans and coverage > 0.5:
                tp = tuple(best_trans['position']) if isinstance(best_trans['position'], list) else best_trans['position']
                if manhattan_distance(target_tile, tp) < manhattan_distance(current_pos, tp):
                    predicted *= (1.0 + trans_attract * transition_weight_mult)

            if probe_action == a.action and probe_dir is not None: predicted *= 2.0
            predicted *= (0.9 + random.random() * 0.2)

        elif a.group == "interact":
            predicted = max(predicted, min_interact_prob)
            if a.action == 'B': predicted *= brain.menu_trap_b_boost
            if a.action == 'A':
                if tile_probing and probe_action == 'A': predicted *= 3.0
                elif tile_probing: predicted *= 0.5
                else: predicted *= 0.3

        # === BLEND PIPELINE PREFERENCE (NEW) ===
        # If overworld pipeline has authority, boost the action it prefers
        if ow_pipeline_action is not None and a.action == ow_pipeline_action.action:
            pipeline_boost = 1.0 + ow_pipeline_auth * 2.0  # up to 3x at full authority
            predicted *= pipeline_boost

        action_scores[a.action] = (a, predicted)

    pref = "interact" if mode in ("battle", "interact") else "move"
    in_mode = [(a, s) for _, (a, s) in action_scores.items() if a.group == pref and s > 0]
    out_mode = [(a, s) for _, (a, s) in action_scores.items()
                if a.group != pref and s > 0 and a.action not in ('Start', 'Select')]

    best_in = max(in_mode, key=lambda x: x[1]) if in_mode else None
    best_out = max(out_mode, key=lambda x: x[1]) if out_mode else None

    if best_in and best_out:
        chosen = best_out[0] if best_out[1] > best_in[1] * override_threshold else best_in[0]
    elif best_in: chosen = best_in[0]
    elif best_out: chosen = best_out[0]
    else: chosen = max(actions_list, key=lambda a: a.utility)

    brain.record_action_execution(chosen.action)
    brain.track_consecutive_action(chosen.action)
    if chosen.action == 'A' and tile_probing:
        brain.start_interaction_verification(raw_x, raw_y, current_map, current_dir)
    return chosen

In [ ]:
# ============================================================================
# CELL 5: Cache System - MapCache, CacheManager, IOThread, SaveWorkerThread,
#          EventRecorderThread
# ============================================================================
# CHANGES from v17.2:
# 1. UPDATED: MapCache — added text_flag field
# 2. UPDATED: MapCache.update_state() — accepts text_flag parameter
# 3. UPDATED: MapCache.get_state() — returns text_flag as 11th value
# 4. UPDATED: IOThread.run() — unpacks 11 values from read_game_state()
# 5. All other classes UNCHANGED
# ============================================================================

import threading
import queue
import gc


class MapCache:
    """Thread-safe container for one map's data."""

    def __init__(self, map_id):
        self.map_id = map_id
        self.lock = threading.Lock()

        self.exploration_data = None
        self.taught_frames = []

        self.current_state = np.zeros(EXPECTED_STATE_DIM)
        self.palette = np.zeros(PALETTE_DIM)
        self.tiles = np.zeros(TILE_DIM)
        self.raw_position = (0, 0)
        self.dead = False
        self.battle_data = DEFAULT_BATTLE_DATA.copy()
        self.party_data = DEFAULT_PARTY_DATA.copy()
        self.game_state_raw = 0
        self.menu_data = DEFAULT_MENU_DATA.copy()
        self.bag_data = DEFAULT_BAG_DATA.copy()
        self.text_flag = 0  # NEW v17.3: text dialogue flag
        self.state_fresh = False
        self.state_version = 0

        self.pending_action_out = None

    def get_state(self):
        """
        Returns 11 values (v17.3: added text_flag as 11th).
        """
        with self.lock:
            return (
                self.current_state.copy(),
                self.palette.copy(),
                self.tiles.copy(),
                self.dead,
                self.raw_position,
                self.battle_data.copy(),
                self.party_data.copy(),
                self.game_state_raw,
                self.menu_data.copy(),
                self.bag_data.copy(),
                self.text_flag,
            )

    def update_state(self, context_state, palette, tiles, dead, raw_position,
                     battle_data=None, party_data=None,
                     game_state_raw=None, menu_data=None, bag_data=None,
                     text_flag=None):
        with self.lock:
            self.current_state = context_state
            self.palette = palette
            self.tiles = tiles
            self.dead = dead
            self.raw_position = raw_position
            if battle_data is not None:
                self.battle_data = battle_data
            if party_data is not None:
                self.party_data = party_data
            if game_state_raw is not None:
                self.game_state_raw = game_state_raw
            if menu_data is not None:
                self.menu_data = menu_data
            if bag_data is not None:
                self.bag_data = bag_data
            if text_flag is not None:
                self.text_flag = text_flag
            self.state_fresh = True
            self.state_version += 1

    def is_fresh(self):
        with self.lock:
            return self.state_fresh

    def mark_consumed(self):
        with self.lock:
            self.state_fresh = False

    def get_version(self):
        with self.lock:
            return self.state_version

    def set_pending_action(self, action_name):
        with self.lock:
            self.pending_action_out = action_name

    def get_pending_action(self):
        with self.lock:
            a = self.pending_action_out
            self.pending_action_out = None
            return a

    def get_taught_frames(self):
        return self.taught_frames


class CacheManager:
    """Manages all MapCaches. Pre-indexes at startup, handles map switching."""

    def __init__(self, brain):
        self.brain = brain
        self.caches = {}
        self.active_cache = None
        self.active_map_id = None
        self.lock = threading.Lock()

    def load_all(self, exploration_path=None, taught_path=None):
        exploration_path = exploration_path or EXPLORATION_MEMORY_FILE
        taught_path = taught_path or TAUGHT_TRANSITIONS_FILE

        for map_id, mem_data in self.brain.exploration_memory.items():
            cache = self._get_or_create(map_id)
            cache.exploration_data = mem_data

        taught_by_map = {}
        for t in self.brain.taught_transitions:
            t_map = t.get('state', {}).get('map_id')
            if t_map is not None:
                taught_by_map.setdefault(t_map, []).append(t)

        for map_id, frames in taught_by_map.items():
            cache = self._get_or_create(map_id)
            cache.taught_frames = frames

        total_maps = len(self.caches)
        total_taught = sum(len(c.taught_frames) for c in self.caches.values())
        print(f"  📦 CacheManager: {total_maps} maps cached, {total_taught} taught frames indexed")

    def _get_or_create(self, map_id):
        if map_id not in self.caches:
            self.caches[map_id] = MapCache(map_id)
        return self.caches[map_id]

    def get_active(self):
        return self.active_cache

    def detect_and_set_initial_map(self):
        (ctx, pal, til, dead, raw_pos, battle_data, party_data,
         game_state_raw, menu_data, bag_data, text_flag) = read_game_state()
        map_id = int(ctx[2])
        self._switch_to(map_id)
        self.active_cache.update_state(ctx, pal, til, dead, raw_pos,
                                       battle_data, party_data,
                                       game_state_raw, menu_data, bag_data,
                                       text_flag)
        print(f"  📦 Initial map: {map_id}")
        return map_id

    def switch_map(self, new_map_id):
        if new_map_id == self.active_map_id:
            return
        self._sync_from_brain()
        self._switch_to(new_map_id)
        self._sync_to_brain()

    def _switch_to(self, map_id):
        with self.lock:
            cache = self._get_or_create(map_id)
            self.active_cache = cache
            self.active_map_id = map_id

    def _sync_to_brain(self):
        cache = self.active_cache
        if cache and cache.exploration_data is not None:
            self.brain.exploration_memory[cache.map_id] = cache.exploration_data

    def _sync_from_brain(self):
        cache = self.active_cache
        if cache and cache.map_id in self.brain.exploration_memory:
            cache.exploration_data = self.brain.exploration_memory[cache.map_id]

    def sync_all_from_brain(self):
        for map_id, mem_data in self.brain.exploration_memory.items():
            cache = self._get_or_create(map_id)
            cache.exploration_data = mem_data

    def save_exploration_memory(self):
        self._sync_from_brain()
        self.brain.save_exploration_memory()

    def get_active_taught_frames(self):
        if self.active_cache:
            return self.active_cache.get_taught_frames()
        return []

    def get_map_density(self):
        if not self.active_cache:
            return {'taught_frames': 0, 'tier': 'sparse', 'coverage': 0.0, 'visited': 0}

        n_frames = len(self.active_cache.get_taught_frames())
        map_id = self.active_map_id

        coverage = self.brain.get_exploration_coverage(map_id) if map_id is not None else 0.0
        memory = self.brain.get_current_map_memory(map_id) if map_id is not None else {}
        visited = len(memory.get('visited_tiles', set()))

        if n_frames < 50:
            tier = 'sparse'
        elif n_frames < 200:
            tier = 'thin'
        elif n_frames < 1000:
            tier = 'medium'
        else:
            tier = 'dense'

        return {
            'taught_frames': n_frames,
            'tier': tier,
            'coverage': coverage,
            'visited': visited
        }


class IOThread(threading.Thread):
    """Background thread: reads game_state.json, writes action.json."""

    def __init__(self, cache_manager, interval=0.02, gc_interval=300):
        super().__init__(daemon=True)
        self.cm = cache_manager
        self.interval = interval
        self.gc_interval = gc_interval
        self.running = False
        self._iteration = 0

    def run(self):
        self.running = True
        print(f"  🔄 IOThread started (interval={self.interval*1000:.0f}ms)")

        while self.running:
            try:
                cache = self.cm.get_active()
                if cache is None:
                    time.sleep(self.interval)
                    continue

                # v17.3: read_game_state() now returns 11 values
                (ctx, pal, til, dead, raw_pos, battle_data, party_data,
                 game_state_raw, menu_data, bag_data, text_flag) = read_game_state()
                cache.update_state(ctx, pal, til, dead, raw_pos,
                                   battle_data, party_data,
                                   game_state_raw, menu_data, bag_data,
                                   text_flag)

                action = cache.get_pending_action()
                if action is not None:
                    write_action(action)

                self._iteration += 1
                if self._iteration % self.gc_interval == 0:
                    gc.collect()

            except Exception as e:
                print(f"  [IOThread ERROR] {e}")

            time.sleep(self.interval)

    def stop(self):
        self.running = False
        print("  🔄 IOThread stopped")


class SaveWorkerThread(threading.Thread):
    """Background thread for all brain state file I/O saves."""

    _SENTINEL = object()

    def __init__(self, maxsize=3):
        super().__init__(daemon=True)
        self.save_queue = queue.Queue(maxsize=maxsize)
        self.running = False
        self.saves_completed = 0
        self.saves_dropped = 0
        self.last_save_duration = 0.0
        self.total_save_time = 0.0
        self._lock = threading.Lock()

    def run(self):
        self.running = True
        print(f"  💾 SaveWorkerThread started (queue maxsize={self.save_queue.maxsize})")

        while self.running:
            try:
                try:
                    job = self.save_queue.get(timeout=1.0)
                except queue.Empty:
                    continue

                if job is self._SENTINEL:
                    self.save_queue.task_done()
                    break

                start_time = time.time()
                self._process_job(job)
                duration = time.time() - start_time

                with self._lock:
                    self.saves_completed += 1
                    self.last_save_duration = duration
                    self.total_save_time += duration

                self.save_queue.task_done()
                gc.collect()

            except Exception as e:
                print(f"  [SaveWorker ERROR] {e}")
                try:
                    self.save_queue.task_done()
                except ValueError:
                    pass

        self.running = False
        print(f"  💾 SaveWorkerThread stopped ({self.saves_completed} saves, "
              f"{self.saves_dropped} dropped, {self.total_save_time:.1f}s total)")

    def _process_job(self, job):
        job_type = job.get('type', 'unknown')
        brain = job.get('brain')
        cache_manager = job.get('cache_manager')
        filepath = job.get('filepath')
        timestep = job.get('timestep', -1)

        if brain is None:
            print(f"  [SaveWorker] No brain reference in job, skipping")
            return

        try:
            if job_type == 'checkpoint':
                if filepath:
                    brain.save_model_checkpoint(filepath)
                    print(f"  💾 Checkpoint saved (step {timestep}, bg)")

            elif job_type == 'exploration':
                if cache_manager:
                    cache_manager.save_exploration_memory()
                else:
                    brain.save_exploration_memory()

            elif job_type == 'roster':
                brain.save_roster()

            elif job_type == 'move_knowledge':
                brain.save_move_knowledge()

            elif job_type == 'item_knowledge':
                brain.save_item_knowledge()

            elif job_type == 'type_clusters':
                brain.save_type_clusters()

            elif job_type == 'all_knowledge':
                if cache_manager:
                    cache_manager.save_exploration_memory()
                else:
                    brain.save_exploration_memory()
                if brain.roster_dirty:
                    brain.save_roster()
                if brain.move_knowledge_dirty:
                    brain.save_move_knowledge()
                if brain.item_knowledge_dirty:
                    brain.save_item_knowledge()
                if brain.type_clusters_dirty:
                    brain.save_type_clusters()
                if filepath:
                    brain.save_model_checkpoint(filepath)
                print(f"  💾 Full save completed (step {timestep}, bg)")

            else:
                print(f"  [SaveWorker] Unknown job type: {job_type}")

        except Exception as e:
            print(f"  [SaveWorker] Error in {job_type}: {e}")

    def submit_job(self, job):
        try:
            self.save_queue.put_nowait(job)
            return True
        except queue.Full:
            try:
                dropped = self.save_queue.get_nowait()
                self.save_queue.task_done()
                with self._lock:
                    self.saves_dropped += 1
            except queue.Empty:
                pass

            try:
                self.save_queue.put_nowait(job)
                return True
            except queue.Full:
                with self._lock:
                    self.saves_dropped += 1
                return False

    def submit_checkpoint(self, brain, filepath, timestep, cache_manager=None):
        return self.submit_job({
            'type': 'checkpoint',
            'brain': brain,
            'filepath': filepath,
            'timestep': timestep,
            'cache_manager': cache_manager,
        })

    def submit_exploration(self, brain, cache_manager=None):
        return self.submit_job({
            'type': 'exploration',
            'brain': brain,
            'cache_manager': cache_manager,
        })

    def submit_all_knowledge(self, brain, filepath, timestep, cache_manager=None):
        return self.submit_job({
            'type': 'all_knowledge',
            'brain': brain,
            'filepath': filepath,
            'timestep': timestep,
            'cache_manager': cache_manager,
        })

    def submit_dirty_knowledge(self, brain):
        submitted = 0
        if brain.roster_dirty:
            self.submit_job({'type': 'roster', 'brain': brain})
            submitted += 1
        if brain.move_knowledge_dirty:
            self.submit_job({'type': 'move_knowledge', 'brain': brain})
            submitted += 1
        if brain.item_knowledge_dirty:
            self.submit_job({'type': 'item_knowledge', 'brain': brain})
            submitted += 1
        if brain.type_clusters_dirty:
            self.submit_job({'type': 'type_clusters', 'brain': brain})
            submitted += 1
        return submitted

    def get_stats(self):
        with self._lock:
            return {
                'saves_completed': self.saves_completed,
                'saves_dropped': self.saves_dropped,
                'last_save_duration': self.last_save_duration,
                'total_save_time': self.total_save_time,
                'queue_size': self.save_queue.qsize(),
                'running': self.running,
            }

    def stop(self):
        self.running = False
        try:
            while not self.save_queue.empty():
                try:
                    self.save_queue.get_nowait()
                    self.save_queue.task_done()
                except queue.Empty:
                    break
            self.save_queue.put_nowait(self._SENTINEL)
        except queue.Full:
            pass


class EventRecorderThread(threading.Thread):
    """
    Background thread for recording AI-side events to ai_event_timeline.json.
    """

    _SENTINEL = object()

    def __init__(self, filepath=None, flush_interval=30, max_queue_size=100,
                 max_events=5000):
        super().__init__(daemon=True)
        self.filepath = filepath or AI_EVENT_TIMELINE_FILE
        self.flush_interval = flush_interval
        self.max_events = max_events

        self.event_queue = queue.Queue(maxsize=max_queue_size)
        self.events = []
        self.running = False

        self.battle_count = 0
        self.bag_count = 0
        self.map_count = 0
        self.levelup_count = 0
        self.maps_seen = set()

        self._last_flush_time = 0.0
        self._lock = threading.Lock()

    def run(self):
        self.running = True
        self._last_flush_time = time.time()

        self._load_existing()

        print(f"  📝 EventRecorderThread started (flush every {self.flush_interval}s, "
              f"max {self.max_events} events, existing: {len(self.events)})")

        while self.running:
            try:
                drained = 0
                while True:
                    try:
                        event = self.event_queue.get_nowait()
                    except queue.Empty:
                        break

                    if event is self._SENTINEL:
                        self.event_queue.task_done()
                        self._flush_to_disk()
                        self.running = False
                        break

                    self._record_event(event)
                    self.event_queue.task_done()
                    drained += 1

                if not self.running:
                    break

                now = time.time()
                if now - self._last_flush_time >= self.flush_interval:
                    self._flush_to_disk()
                    self._last_flush_time = now

                time.sleep(0.5)

            except Exception as e:
                print(f"  [EventRecorder ERROR] {e}")
                time.sleep(1.0)

        self.running = False
        print(f"  📝 EventRecorderThread stopped ({len(self.events)} events recorded)")

    def _load_existing(self):
        try:
            if Path(self.filepath).exists():
                with open(self.filepath, 'r') as f:
                    data = json.load(f)
                self.events = data.get('events', [])
                summary = data.get('summary', {})
                self.battle_count = summary.get('battle_events', 0)
                self.bag_count = summary.get('bag_events', 0)
                self.map_count = summary.get('map_events', 0)
                self.levelup_count = summary.get('levelup_events', 0)
                self.maps_seen = set(summary.get('maps_visited', []))
        except Exception:
            self.events = []

    def _record_event(self, event):
        with self._lock:
            self.events.append(event)

            etype = event.get('type', '')
            if etype == 'battle_end':
                self.battle_count += 1
            elif etype == 'bag_session':
                self.bag_count += 1
            elif etype == 'map_transition':
                self.map_count += 1
            elif etype == 'level_up':
                self.levelup_count += 1

            map_id = event.get('map_id')
            if map_id is not None:
                self.maps_seen.add(map_id)

            if len(self.events) > self.max_events:
                trim_count = self.max_events // 5
                self.events = self.events[trim_count:]

    def _flush_to_disk(self):
        with self._lock:
            if not self.events:
                return

            try:
                first_ts = self.events[0].get('timestep', 0) if self.events else 0
                last_ts = self.events[-1].get('timestep', 0) if self.events else 0

                data = {
                    'events': self.events,
                    'summary': {
                        'total_events': len(self.events),
                        'battle_events': self.battle_count,
                        'bag_events': self.bag_count,
                        'map_events': self.map_count,
                        'levelup_events': self.levelup_count,
                        'first_timestep': first_ts,
                        'last_timestep': last_ts,
                        'maps_visited': sorted(self.maps_seen),
                    }
                }

                with open(self.filepath, 'w') as f:
                    json.dump(data, f, indent=2)

            except Exception as e:
                print(f"  [EventRecorder] Flush error: {e}")

    def get_queue(self):
        return self.event_queue

    def get_stats(self):
        with self._lock:
            return {
                'running': self.running,
                'total_events': len(self.events),
                'battles': self.battle_count,
                'bags': self.bag_count,
                'maps': self.map_count,
                'levelups': self.levelup_count,
                'maps_visited': len(self.maps_seen),
                'queue_size': self.event_queue.qsize(),
            }

    def stop(self):
        self.running = False
        try:
            self.event_queue.put_nowait(self._SENTINEL)
        except queue.Full:
            self._flush_to_disk()

In [ ]:
# ============================================================================
# CELL 6: Main Loop — Multi-Pool Pipeline Integration
# ============================================================================
# CHANGES from v17.3:
# 1. Load residual perceptrons file at startup
# 2. Pipeline status in startup banner
# 3. Revenge module status in startup banner + logging
# 4. Pipeline summary in 100-step and 500-step logging
# 5. Save residual file in periodic saves
# 6. Revenge status in milestone logging
# 7. All existing v17.3 functionality UNCHANGED
# ============================================================================

import tracemalloc

tracemalloc.start()

brain = Brain()

# === CREATE ACTION PERCEPTRONS (shared chain) ===
for b in ["UP", "DOWN", "LEFT", "RIGHT"]:
    brain.add(Perceptron("action", action=b, group="move", chain="shared"))
for b in ["A", "B", "Start", "Select"]:
    brain.add(Perceptron("action", action=b, group="interact", chain="shared"))

TAUGHT_MODEL_PATH = BASE_PATH / "taught_model_checkpoint.json"
TAUGHT_EXPLORATION_PATH = BASE_PATH / "taught_exploration_memory.json"

# === LOAD MODEL (3-way: resume → bootstrap → fresh) ===
bootstrapped_from_taught = False
taught_timestep = 0

if MODEL_CHECKPOINT_FILE.exists():
    loaded_ts = brain.load_taught_model(MODEL_CHECKPOINT_FILE)
    print(f"🤖 AI MODEL: Resumed from timestep {loaded_ts}")
    print(f"   Utils: {[f'{a.action}:{a.utility:.3f}' for a in brain.actions()]}")
else:
    if TAUGHT_MODEL_PATH.exists():
        taught_timestep = brain.initialize_from_taught_model(TAUGHT_MODEL_PATH)
        bootstrapped_from_taught = True
    else:
        print("🤖 AI MODEL: Starting fresh")

# === LOAD TAUGHT REFERENCE (always — for stagnation blending) ===
brain.load_taught_reference(TAUGHT_MODEL_PATH)

# === LOAD ALL DATA ===
brain.merge_taught_exploration(TAUGHT_EXPLORATION_PATH)
brain.load_taught_transitions(TAUGHT_TRANSITIONS_FILE)
brain.load_taught_battle_transitions(TAUGHT_BATTLE_TRANSITIONS_FILE)
brain.load_taught_bag_transitions(TAUGHT_BAG_TRANSITIONS_FILE)
brain.load_taught_start_menu_transitions(TAUGHT_START_MENU_TRANSITIONS_FILE)
brain.load_taught_nav_targets(TAUGHT_NAV_TARGETS_FILE)
brain.load_roster(ROSTER_FILE)
brain.load_move_knowledge(MOVE_KNOWLEDGE_FILE)
brain.load_item_knowledge(ITEM_KNOWLEDGE_FILE)
brain.load_event_timeline(EVENT_TIMELINE_FILE)

# === LOAD TYPE CHART DATA ===
brain.load_type_clusters(TYPE_CLUSTERS_FILE)
brain.load_ground_truth_types(TYPE_DATA_FILE)

# === LOAD RESIDUAL PERCEPTRONS (NEW) ===
brain.load_residual_file()

map_graph = brain.build_map_graph()
graph_edges = sum(len(v) for v in map_graph.values())
graph_maps = list(map_graph.keys())

cache_manager = CacheManager(brain)
cache_manager.load_all()
cache_manager.detect_and_set_initial_map()

# === START THREADS ===
io_thread = IOThread(cache_manager, interval=0.02, gc_interval=300)
io_thread.start()

save_worker = SaveWorkerThread(maxsize=3)
save_worker.start()

event_recorder = EventRecorderThread(
    filepath=AI_EVENT_TIMELINE_FILE,
    flush_interval=brain.EVENT_RECORDER_FLUSH_INTERVAL,
    max_queue_size=100,
    max_events=5000
)
event_recorder.start()

# Connect event queue to brain
brain.event_queue = event_recorder.get_queue()
brain.event_recorder_active = True

exploration_weight = 1.3
forced_explore_prob = 0.18
prev_context_state = None
prev_raw_position = None
last_processed_version = -1
battle_outcomes = {'win': 0, 'loss': 0, 'run': 0, 'catch': 0, 'unknown': 0}

_mem_baseline = tracemalloc.get_traced_memory()

print("="*70)
print("AI CONTROL — Multi-Pool Pipeline Network")
print("="*70)

# === BOOTSTRAP STATUS ===
print("BOOTSTRAP STATUS:")
if bootstrapped_from_taught:
    n_ent = len(brain.entities())
    cs = brain.get_chain_stats()
    cs_str = ' | '.join(f"{c}:{s['actions']}a+{s['entities']}e" for c, s in cs.items())
    print(f"  🎓 Bootstrapped from taught model ({taught_timestep} human steps)")
    print(f"  Perceptrons: {len(brain.perceptrons)} ({cs_str})")
    print(f"  AI starts at timestep 0 — will refine through play")
else:
    if MODEL_CHECKPOINT_FILE.exists():
        print(f"  Resumed own checkpoint (no bootstrap needed)")
    else:
        print(f"  No bootstrap (fresh start)")

print("="*70)
print("ARCHITECTURE:")
print(f"  Chains: overworld(visual/spatial) | battle(compact) | party | bag")
print(f"  Empirical activation discovery: {', '.join(ACTIVATION_LIBRARY.get_names())}")
chain_stats = brain.get_chain_stats()
if chain_stats:
    for c, s in chain_stats.items():
        print(f"    {c}: {s['actions']}actions {s['entities']}entities")
act_counts = {}
for p in brain.perceptrons:
    act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
if act_counts:
    print(f"  Activations: {' '.join(f'{k}:{v}' for k,v in sorted(act_counts.items(), key=lambda x:x[1], reverse=True))}")
print("="*70)
print("PIPELINES:")
for pid, pipeline in brain.pipelines.items():
    layers_str = ' → '.join(pool.name for pool in pipeline.pools)
    total_p = sum(pool.get_perceptron_count(brain.perceptrons) for pool in pipeline.pools)
    total_r = sum(len(pool.residual) for pool in pipeline.pools)
    print(f"  {pid} ({pipeline.num_layers}L, decay={pipeline.credit_decay}): {layers_str}")
    print(f"    perceptrons: {total_p} | residual: {total_r} | authority: {pipeline.get_total_authority():.0%}")
print(f"  Fallback chain: Pipeline → Markov → Hardcoded (graceful degradation)")
print("="*70)
print("THREADS:")
print(f"  IOThread: game state I/O (20ms)")
print(f"  SaveWorkerThread: background file saves (queue=3)")
print(f"  EventRecorderThread: AI event recording (flush={brain.EVENT_RECORDER_FLUSH_INTERVAL}s)")
print(f"  Priority: PartyMenu → Dialogue → StartMenu → Bag → Battle → Prep → Overworld")
print("="*70)
print("DIALOGUE:")
print(f"  TextFlag: ADDR_DIALOGUE 0x0202004F (0=no text, 1=text active)")
print(f"  Pure text → A/B skip (no utility tracking)")
print(f"  Dialogue choice → mm≤{brain.DIALOGUE_CHOICE_MM_MAX} with valid cursor (real decision)")
print(f"  Battle text → skip during animations (game ignores input)")
print("="*70)
print("START MENU:")
print(f"  Transitions: {'LOADED' if brain.start_menu_loaded else 'NOT FOUND'} | "
      f"Markov: {START_MENU_MARKOV_THRESHOLD:.2f}")
if brain.start_menu_loaded:
    print(f"  Frames: {len(brain.start_menu_transitions)} | "
          f"Sessions: {brain.start_menu_metadata.get('sessions_recorded', 0)}")
print(f"  Replaces hardcoded prep navigation when available")
print("="*70)
print("TYPE CHART:")
tc_status = brain.get_type_chart_status()
print(f"  Track A (empirical): {tc_status['move_clusters']}mc {tc_status['species_clusters']}sc "
      f"{tc_status['effectiveness_entries']}eff | runs: {tc_status['clustering_runs']}")
print(f"  Track B (ground truth): {'LOADED' if tc_status['track_b_loaded'] else 'NOT FOUND'}")
print(f"  Clustering: every {brain.CLUSTERING_BATTLE_INTERVAL} battles | "
      f"threshold: {brain.CLUSTERING_SIMILARITY_THRESHOLD}")
print("="*70)
print("REVENGE MODULE:")
rs = brain.get_revenge_status()
if rs['targets'] > 0:
    print(f"  Targets: {rs['targets']} ({rs['by_status']})")
    tid, target = brain.get_active_revenge_target()
    if target:
        print(f"  Active: {tid} → target Lv{target['target_avg_level']:.0f} "
              f"(losses: {target['losses_here']}, status: {target['status']})")
else:
    print(f"  No revenge targets")
print("="*70)
print("PREPARATION:")
tl = brain.get_timeline_status()
if tl['loaded']:
    print(f"  Timeline: {tl['events']}ev {tl['segments']}seg {tl['prep_points']}prep")
else:
    print(f"  Timeline: NOT LOADED")
mbs = brain.get_map_battle_stats_summary()
print(f"  Map stats: {mbs['maps_with_data']}maps {mbs['total_battles']}battles")
print("="*70)
print("BAG:")
print(f"  Transitions: {'LOADED' if brain.bag_loaded else 'NOT FOUND'} | Markov: {BAG_MARKOV_THRESHOLD:.2f}")
print(f"  Items: {len(brain.item_knowledge)} tracked")
print("="*70)
print("BATTLE:")
print(f"  Transitions: {'LOADED' if brain.battle_loaded else 'NOT FOUND'} | "
      f"Markov: {BATTLE_MARKOV_THRESHOLD_LOW:.2f}-{BATTLE_MARKOV_THRESHOLD_HIGH:.2f}")
print(f"  Move scoring: pipeline → direct → Track B → Track A clusters → avg damage")
print("="*70)
print("KNOWLEDGE:")
print(f"  Roster: {len(brain.roster)} | Moves: {len(brain.move_knowledge)} | Enemy: {len(brain.enemy_move_knowledge)}")
print("="*70)
print("CACHE | NAV:")
print(f"  Maps: {len(cache_manager.caches)} | Active: {cache_manager.active_map_id}")
ns = brain.get_nav_targets_status()
if ns['loaded']: print(f"  Nav: {ns['total']} targets")
print(f"  Graph: {len(graph_maps)}maps {graph_edges}edges")
print("="*70)
print("MEMORY:")
print(f"  tracemalloc: active | gc: milestones")
obs_stats = brain.get_activation_observation_stats()
print(f"  Activation obs: {obs_stats['_total']['observations']} (~{obs_stats['_total']['estimated_mb']:.2f}MB)")
print("="*70)

try:
    while True:
        active_cache = cache_manager.get_active()
        current_version = active_cache.get_version()
        if current_version == last_processed_version:
            time.sleep(0.005); continue

        (context_state, palette_state, tile_state, dead, raw_position,
         battle_data, party_data, game_state_raw, menu_data, bag_data,
         text_flag) = active_cache.get_state()
        last_processed_version = current_version

        if np.sum(np.abs(context_state)) < 0.001:
            time.sleep(0.01); continue

        raw_x, raw_y = raw_position
        in_battle = context_state[3]
        current_map = int(context_state[2])
        current_dir = int(context_state[5])

        brain.update_battle_data(battle_data)
        brain.update_party_data(party_data)
        brain.update_menu_data(menu_data)
        brain.update_bag_data(bag_data)
        brain.game_state_raw = game_state_raw

        brain.text_flag = text_flag
        brain.update_dialogue_state(context_state)

        brain.update_party_menu_state(context_state)
        brain.update_bag_thread_state(context_state)
        brain.update_start_menu_state(context_state)
        brain.check_item_observation()

        currently_in_battle = in_battle > 0.5

        if not currently_in_battle and game_state_raw == 0:
            brain.increment_map_steps(current_map)

        # === BATTLE START/END ===
        if currently_in_battle and not brain.in_battle_last_frame:
            brain.current_battle_id += 1
            brain.battle_frame_count = 0
            brain.battle_action_history.clear()
            brain.on_battle_start_with_data()

            bd = brain.battle_data
            bt_str = "TR" if (bd.get('battle_type',0) & 8) != 0 else "WD"
            sp = f" {bd['player_species']}v{bd['enemy_species']}" if bd['player_species'] > 0 else ""
            hp = f" HP:{bd['player_hp']}/{bd['player_max_hp']}" if bd['player_hp'] > 0 else ""
            print(f"\n  ⚔️ START #{brain.current_battle_id} {bt_str} Map{current_map}({raw_x},{raw_y}){sp}{hp}")

            if brain.has_battle_data() and bd['enemy_species'] > 0:
                ranked = brain.get_best_move_for_enemy(bd['enemy_species'])
                if ranked:
                    print(f"     📖 {', '.join(f'm{m}(s{s} {sc:.1f})' for m,s,sc in ranked[:4])}")

            # Pipeline authority at battle start
            bp_auth = brain.battle_pipeline.get_total_authority()
            if bp_auth > 0:
                print(f"     🔗 Battle pipeline: {bp_auth:.0%} authority")

            mbs_cur = brain.map_battle_stats.get(current_map)
            if mbs_cur and mbs_cur['battles_fought'] > 0:
                print(f"     📊 Map: {mbs_cur['battles_fought']}bat avg{mbs_cur['avg_hp_cost']:.0%}")

            # Revenge check
            rs = brain.get_revenge_status()
            if rs['active']:
                tid, target = brain.get_active_revenge_target()
                if target and target['map_id'] == current_map:
                    print(f"     ⚔️🔥 REVENGE ZONE: {tid} (target Lv{target['target_avg_level']:.0f})")

            if brain.is_nav_active(): brain.abort_navigation("battle")

        elif currently_in_battle:
            if brain.has_battle_data() and brain.detect_turn_resolved():
                brain.on_battle_turn_end()
                bd = brain.battle_data
                mi = f" m{brain.last_move_used}(s{brain.last_move_slot})" if brain.last_move_used > 0 else ""
                em = brain.detect_enemy_move()
                ei = f" em{em}" if em > 0 else ""
                ri = " 🏃" if brain.should_run() else ""
                print(f"  ⚔️ T{brain.turn_count} {bd['player_hp']}/{bd['player_max_hp']} "
                      f"E:{bd['enemy_hp']}/{bd.get('enemy_max_hp','?')}{mi}{ei}{ri}")

        elif not currently_in_battle and brain.in_battle_last_frame:
            outcome = brain.on_battle_end_with_data()
            battle_outcomes[outcome] = battle_outcomes.get(outcome, 0) + 1
            emoji = {'win':'🏆','loss':'💀','run':'🏃','catch':'🎊','unknown':'❓'}.get(outcome,'❓')
            bmr = brain.battle_markov_action_count / max(1, brain.battle_action_count)
            print(f"\n  ⚔️ END #{brain.current_battle_id} {brain.turn_count}t {emoji}{outcome.upper()} "
                  f"Mk:{bmr:.0%} Cur:{brain.battle_cursor_action_count}")

            mbs_cur = brain.map_battle_stats.get(current_map)
            if mbs_cur:
                print(f"     📊 Map{current_map}: {mbs_cur['battles_fought']}bat "
                      f"avg{mbs_cur['avg_hp_cost']:.0%} rate{mbs_cur['encounter_rate']:.4f}")

            # Revenge outcome logging
            if outcome == 'loss':
                rs = brain.get_revenge_status()
                if rs['active']:
                    tid, target = brain.get_active_revenge_target()
                    if target:
                        print(f"     ⚔️🔥 REVENGE: losses={target['losses_here']} "
                              f"target Lv{target['target_avg_level']:.0f} status={target['status']}")

            brain.battle_frame_count = 0

            brain.push_event('battle_end', {
                'battle_id': brain.current_battle_id,
                'outcome': outcome,
                'turns': brain.turn_count,
                'enemy_species': brain.prev_battle_data.get('enemy_species', -1),
                'enemy_level': brain.prev_battle_data.get('enemy_level', -1),
                'is_trainer': brain.is_trainer_battle(),
                'player_hp_start': brain.battle_start_hp,
                'player_hp_end': brain.prev_battle_data.get('player_hp', -1),
                'party_snapshot': [{'hp': s.get('hp',0), 'max_hp': s.get('max_hp',0)}
                                   for s in brain.party_data.get('slots', [])],
            })

            if brain.battles_since_last_clustering >= brain.CLUSTERING_BATTLE_INTERVAL:
                brain.run_type_clustering()
                if brain.type_clusters_dirty:
                    save_worker.submit_job({'type': 'type_clusters', 'brain': brain})

            save_worker.submit_dirty_knowledge(brain)

        brain.in_battle_last_frame = currently_in_battle

        # === MAP CHANGE ===
        if not currently_in_battle and current_map != cache_manager.active_map_id:
            prev_map = cache_manager.active_map_id
            cache_manager.switch_map(current_map)
            active_cache = cache_manager.get_active()
            print(f"  📦 Map{current_map} ({len(active_cache.get_taught_frames())}f)")

            brain.push_event('map_transition', {
                'from_map': prev_map,
                'to_map': current_map,
                'position': (raw_x, raw_y),
            })

            if brain.nav_map_chain and brain.is_nav_active() and not brain.is_nav_paused():
                if not brain.advance_map_chain(current_map, (raw_x, raw_y)):
                    brain.abort_navigation("chain broken")

        brain.update_position(raw_x, raw_y)
        derived = compute_derived_features(context_state, prev_context_state)
        learning_state = build_learning_state_overworld(derived, palette_state, tile_state, in_battle)
        brain.log_state(learning_state, context_state)
        brain.confirm_action_executed(context_state, prev_context_state)

        if brain.should_send_new_action():
            action = anticipatory_action(
                brain, learning_state, context_state,
                exploration_weight=exploration_weight,
                raw_position=raw_position,
                forced_explore_prob=forced_explore_prob,
                taught_frames=cache_manager.get_active_taught_frames(),
                map_density=cache_manager.get_map_density(),
                palette_state=palette_state
            )
            if action is not None:
                active_cache.set_pending_action(action.action)
                brain.last_action = action.action
                brain.set_pending_action(action.action)

                if not currently_in_battle and not brain.is_dialogue_skip_state():
                    brain.update_menu_trap_tracking(context_state, action.action, raw_position=raw_position)
            else:
                active_cache.set_pending_action("NONE")
        else:
            if brain.pending_action:
                active_cache.set_pending_action(brain.pending_action)

        # === PERIODIC BACKGROUND SAVES (every 200 steps) ===
        if brain.timestep % 200 == 0 and brain.timestep > 0:
            save_worker.submit_exploration(brain, cache_manager)
            save_worker.submit_dirty_knowledge(brain)

        # === LOGGING (every 100 steps) ===
        if brain.timestep % 100 == 0:
            memory = brain.get_current_map_memory(current_map)
            coverage = brain.get_exploration_coverage(current_map)
            density = cache_manager.get_map_density()
            gs_str = {0:"OW",1:"MENU",14:"BAG"}.get(game_state_raw, f"GS{game_state_raw}")
            dir_name = brain.DIRECTION_NAMES.get(current_dir, '?')
            ta = brain.markov_action_count + brain.curiosity_action_count
            mr = brain.markov_action_count / max(1, ta)
            tf_str = " TXT" if brain.text_flag == 1 else ""

            print(f"\n{'='*70}")
            print(f"Step {brain.timestep} | Map{current_map} ({raw_x},{raw_y}) {dir_name} | gs={gs_str}{tf_str}")
            print(f"  Mode: {'BOTH⚡' if brain.should_use_both_mode() else brain.control_mode} | Stag: {brain.state_stagnation_count}")

            # Dialogue status
            ds = brain.get_dialogue_status()
            if ds['active']:
                dtype = 'CHOICE' if ds.get('is_choice') else ('BATTLE_TXT' if ds.get('is_battle_text') else 'PURE_TXT')
                print(f"  💬 DIALOGUE: {dtype} ({ds.get('frames_in_current',0)}f)")
            if ds['total_skip_actions'] > 0 or ds['total_choice_actions'] > 0:
                print(f"  💬 Dialogue totals: {ds['total_skip_actions']}skip {ds['total_choice_actions']}choice {ds['total_frames']}f")

            # Pipeline status (NEW)
            p_summary = brain.get_pipeline_summary()
            if p_summary != 'all empty':
                print(f"  🔗 Pipelines: {p_summary}")

            # Revenge status (NEW)
            rs = brain.get_revenge_status()
            if rs['targets'] > 0:
                tid, target = brain.get_active_revenge_target()
                if target:
                    print(f"  ⚔️🔥 Revenge: {tid} → Lv{target['target_avg_level']:.0f} "
                          f"({target['status']}, {target['losses_here']}L)")
                else:
                    print(f"  ⚔️ Revenge: {rs['targets']} targets ({rs['by_status']})")

            # Threads
            if brain.is_party_menu_active():
                print(f"  📋 PARTY: {brain.party_menu_context} ({brain.timestep-brain.party_menu_entered_at}f)")
            if brain.is_start_menu_active():
                sms = brain.get_start_menu_status()
                tgt = sms.get('target_mc', -1)
                print(f"  📋 START MENU: {sms['context']} mc→{tgt} ({sms['frames']}f {sms['actions']}act)")
            if brain.is_bag_thread_active():
                bf = brain.timestep - brain.bag_thread_entered_at
                ci = brain.get_item_at_cursor()
                print(f"  🎒 BAG: {brain.bag_thread_context} ({bf}f {brain.bag_thread_action_count}act)"
                      f"{f' item={ci}' if ci>0 else ''}")
            ps = brain.get_preparation_status()
            if ps['active']:
                sm_nav = " (StartMenu nav)" if ps.get('start_menu_nav') else ""
                print(f"  🎯 PREP: {ps['phase']}→{ps['target']} ({ps['frames']}f){sm_nav} | {ps['reason']}")
            elif brain.prep_total_count > 0:
                print(f"  🎯 Prep idle ({brain.prep_total_count}att {brain.prep_success_count}suc)")

            # Battle
            if currently_in_battle and brain.has_battle_data():
                bd = brain.battle_data
                bt = "TR" if (bd.get('battle_type',0)&8)!=0 else "WD"
                cn = {0:'FIGHT',1:'BAG',2:'PKMN',3:'RUN'}.get(bd['battle_cursor'],'?')
                cr = brain.battle_cursor_action_count / max(1, brain.battle_action_count)
                print(f"\n  ⚔️ #{brain.current_battle_id} {bt} t{brain.turn_count} cursor={cn}")
                if bd['player_species']>0:
                    print(f"     👤 sp{bd['player_species']} {bd['player_hp']}/{bd['player_max_hp']}")
                if bd['enemy_species']>0:
                    print(f"     👾 sp{bd['enemy_species']} {bd['enemy_hp']}/{bd.get('enemy_max_hp','?')}")
                if brain.should_run(): print(f"     🏃 RUN nodmg={brain.battle_no_damage_turns}")
                print(f"     Cur:{brain.battle_cursor_action_count}({cr:.0%}) Mk:{brain.battle_markov_action_count}")
                bp_auth = brain.battle_pipeline.get_total_authority()
                if bp_auth > 0:
                    print(f"     🔗 Pipeline: {bp_auth:.0%}")
            elif not currently_in_battle:
                if brain.is_nav_active():
                    xms = brain.get_cross_map_status()
                    if xms['active']:
                        ch = '→'.join(str(m) for m in xms['chain'])
                        print(f"\n  🧭🌍 {'PAUSED: '+xms['paused_reason'] if brain.is_nav_paused() else ch}")
                    elif brain.nav_target:
                        print(f"\n  🧭 →({brain.nav_target[0]},{brain.nav_target[1]}) "
                              f"{brain.nav_path_index}/{len(brain.nav_path)} {brain.nav_steps_taken}s")
                else:
                    ns = brain.get_nav_targets_status()
                    tgt_str = f" tgt:{ns['remaining']}/{ns['total']}" if ns['loaded'] else ""
                    print(f"\n  🧭 Idle kn={brain.known_area_counter}/{brain.KNOWN_AREA_TRIGGER}{tgt_str}")

            # Decisions
            smr = brain.start_menu_markov_actions / max(1, brain.start_menu_total_actions)
            print(f"\n  🧠 OW:{brain.markov_action_count}Mk({mr:.0%}) {brain.curiosity_action_count}cur | "
                  f"Bat:{brain.battle_action_count} Bag:{brain.bag_thread_total_actions} "
                  f"SM:{brain.start_menu_total_actions}({smr:.0%}Mk) | {density['tier']}")

            if any(v>0 for v in battle_outcomes.values()):
                print(f"  🏆 W:{battle_outcomes['win']} L:{battle_outcomes['loss']} "
                      f"R:{battle_outcomes['run']} C:{battle_outcomes['catch']}")

            # Type chart status
            tc_st = brain.get_type_chart_status()
            if tc_st['clustering_runs'] > 0 or tc_st['track_b_loaded']:
                print(f"\n  🧬 Types: {tc_st['move_clusters']}mc {tc_st['species_clusters']}sc "
                      f"{tc_st['effectiveness_entries']}eff | runs:{tc_st['clustering_runs']} "
                      f"next:{brain.CLUSTERING_BATTLE_INTERVAL - tc_st['battles_since_clustering']}bat"
                      f"{' +TrackB' if tc_st['track_b_loaded'] else ''}")

            # Timeline + autonomous
            cn = brain.get_nearest_nav_order(raw_x, raw_y, current_map)
            if cn >= 0:
                if brain.event_timeline_loaded:
                    up = brain.get_upcoming_events(cn, 5)
                    hc = brain.get_estimated_hp_cost_ahead(cn, 5)
                    pp = brain.get_preparation_point(cn)
                    lh = brain.get_lowest_hp_ratio()
                    ub = [e for e in up if e.get('type')=='battle']
                    print(f"\n  📅 Nav#{cn} | {len(up)}ev {len(ub)}bat | cost:{hc:.0%} HP:{lh:.0%}")
                    if pp:
                        need = "⚠️PREP" if lh < pp.get('party_hp_threshold',1.0) else "✅"
                        print(f"     Prep: {pp.get('reason','?')} | {need}")

                ac, acf = brain.get_autonomous_hp_estimate(raw_x, raw_y, current_map)
                if acf > 0:
                    lh = brain.get_lowest_hp_ratio()
                    print(f"  🔮 Auto: cost{ac:.0%} conf{acf:.0%} HP:{lh:.0%} "
                          f"{'⚠️HEAL' if lh < ac else '✅'}")

            # Map stats
            cms = brain.map_battle_stats.get(current_map)
            if cms and cms['battles_fought'] > 0:
                print(f"\n  📊 Map{current_map}: {cms['battles_fought']}bat {cms['avg_hp_cost']:.0%}cost "
                      f"Lv{cms['avg_enemy_level']:.0f} rate{cms['encounter_rate']:.4f}")

            # Chain stats + activations
            cst = brain.get_chain_stats()
            if cst:
                cst_parts = [f"{c}:{s['actions']}a+{s['entities']}e" for c, s in cst.items()]
                print(f"\n  🧬 Chains: {' | '.join(cst_parts)}")
            act_counts = {}
            for p in brain.perceptrons:
                act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
            if len(act_counts) > 1:
                act_parts = [f"{k}:{v}" for k, v in sorted(act_counts.items(), key=lambda x: x[1], reverse=True)]
                print(f"  🧬 Act: {' '.join(act_parts)}")

            # Items
            known_items = [(iid,ik) for iid,ik in brain.item_knowledge.items() if ik.get('category','unknown')!='unknown']
            if known_items:
                print(f"\n  🎒 {len(known_items)} items categorized")

            # Party
            if brain.party_data.get('count', 0) > 0:
                print(f"\n  👥 Party({brain.party_data['count']}):")
                for i, s in enumerate(brain.party_data.get('slots', [])):
                    hp, mhp = s.get('hp',0), s.get('max_hp',0)
                    r = f"{hp/mhp:.0%}" if mhp>0 else "?"
                    st = f" ⚠️{s['status']}" if s.get('status',0)!=0 else ""
                    print(f"     [{i}] Lv{s.get('level',0)} {hp}/{mhp}({r}){st}")

            # Exploration
            vc = len(memory['visited_tiles'])
            ts = brain.get_tile_interaction_stats(current_map)
            print(f"\n  📊 V:{vc} Cov:{coverage:.0%} Probe:{ts['probed']} Exh:{ts['exhausted']}")

            utils = sorted([(a.action, a.utility) for a in brain.actions()], key=lambda x: x[1], reverse=True)
            print(f"  ⚡ {' '.join(f'{k}:{v:.2f}' for k,v in utils)}")
            print(f"  🧩 {len(brain.perceptrons)}p ({len(brain.actions())}a {len(brain.entities())}e)")

            if brain.state_stagnation_count > 10:
                print(f"  ⚠️ Stag:{brain.state_stagnation_count}/{brain.STATE_STAGNATION_THRESHOLD}")
            if brain.detected_pattern:
                print(f"  🔄 {'-'.join(str(a) for a in brain.detected_pattern)} x{brain.pattern_repeat_count}")

        # === MILESTONES (every 500 steps) ===
        if brain.timestep % 500 == 0 and brain.timestep > 0:
            tv = sum(len(m['visited_tiles']) for m in brain.exploration_memory.values())
            tt = sum(len(m.get('transitions',[])) for m in brain.exploration_memory.values())
            ta = brain.markov_action_count + brain.curiosity_action_count
            mr = brain.markov_action_count / max(1, ta)
            bmr = brain.battle_markov_action_count / max(1, brain.battle_action_count)
            bgmr = brain.bag_thread_markov_actions / max(1, brain.bag_thread_total_actions)
            smr = brain.start_menu_markov_actions / max(1, brain.start_menu_total_actions)
            mg = brain.build_map_graph()
            mbs_s = brain.get_map_battle_stats_summary()

            print(f"\n{'#'*70}")
            print(f"# MILESTONE {brain.timestep}")
            if bootstrapped_from_taught:
                print(f"# Bootstrapped from taught model ({taught_timestep} human steps)")
            print(f"# Maps:{len(brain.exploration_memory)} Tiles:{tv} Trans:{tt}")
            gs_n = {0:"OW",1:"MENU",14:"BAG"}.get(brain.game_state_raw, "?")
            tf_n = " TXT" if brain.text_flag == 1 else ""
            print(f"# gs={gs_n}{tf_n} party={'ON' if brain.party_menu_active else 'off'} "
                  f"bag={'ON' if brain.bag_thread_active else 'off'} "
                  f"start_menu={'ON' if brain.start_menu_active else 'off'} "
                  f"prep={'ON('+brain.prep_phase+')' if brain.prep_active else 'off'}")

            # Dialogue stats
            ds = brain.get_dialogue_status()
            print(f"# dialogue={'ACTIVE('+('choice' if ds.get('is_choice') else 'skip')+')' if ds['active'] else 'off'} "
                  f"skip:{ds['total_skip_actions']} choice:{ds['total_choice_actions']} frames:{ds['total_frames']}")

            print(f"#")
            print(f"# DECISIONS:")
            print(f"#   OW:{brain.markov_action_count}Mk({mr:.1%}) {brain.curiosity_action_count}cur")
            print(f"#   Bat:{brain.battle_action_count} Mk:{brain.battle_markov_action_count}({bmr:.1%})")
            print(f"#   Bag:{brain.bag_thread_total_actions} Mk:{brain.bag_thread_markov_actions}({bgmr:.1%})")
            print(f"#   SM:{brain.start_menu_total_actions} Mk:{brain.start_menu_markov_actions}({smr:.1%})")
            print(f"#   Dialogue: {ds['total_skip_actions']}skip {ds['total_choice_actions']}choice")
            print(f"#   Prep:{brain.prep_total_count}att {brain.prep_success_count}suc")
            print(f"#")
            print(f"# BATTLES: {brain.current_battle_id}")
            print(f"#   W:{battle_outcomes['win']} L:{battle_outcomes['loss']} "
                  f"R:{battle_outcomes['run']} C:{battle_outcomes['catch']}")
            print(f"#")
            print(f"# KNOWLEDGE:")
            print(f"#   Roster:{len(brain.roster)} Moves:{len(brain.move_knowledge)} "
                  f"Items:{len(brain.item_knowledge)} Enemy:{len(brain.enemy_move_knowledge)}")
            print(f"#   MapStats: {mbs_s['maps_with_data']}maps {mbs_s['total_battles']}bat")
            print(f"#")

            # Type chart
            tc_st = brain.get_type_chart_status()
            print(f"# TYPE CHART:")
            print(f"#   Track A: {tc_st['move_clusters']}mc {tc_st['species_clusters']}sc "
                  f"{tc_st['effectiveness_entries']}eff | runs:{tc_st['clustering_runs']}")
            print(f"#   Moves clustered: {tc_st['moves_clustered']} | Species: {tc_st['species_clustered']}")
            print(f"#   Track B: {'LOADED' if tc_st['track_b_loaded'] else 'not available'}")
            print(f"#   Next clustering in: {brain.CLUSTERING_BATTLE_INTERVAL - tc_st['battles_since_clustering']} battles")

            # Pipelines (NEW)
            print(f"#")
            print(f"# PIPELINES:")
            for pid, pipeline in brain.pipelines.items():
                p_status = pipeline.get_status(brain.perceptrons)
                total_p = sum(l['perceptrons'] for l in p_status['layers'])
                total_r = sum(l['residual_size'] for l in p_status['layers'])
                print(f"#   {pid}: {total_p}p {total_r}r auth={p_status['total_authority']:.0%} "
                      f"active={'YES' if p_status['active'] else 'no'}")
                for layer in p_status['layers']:
                    if layer['perceptrons'] > 0 or layer['residual_size'] > 0:
                        print(f"#     {layer['name']}: {layer['perceptrons']}p "
                              f"auth={layer['authority']:.0%} "
                              f"spawn={layer['spawn_count']} "
                              f"residual={layer['residual_size']}")

            # Revenge (NEW)
            print(f"#")
            print(f"# REVENGE MODULE:")
            rs = brain.get_revenge_status()
            if rs['targets'] > 0:
                print(f"#   Targets: {rs['targets']} ({rs['by_status']})")
                for tid, target in brain.revenge_targets.items():
                    print(f"#   {tid}: {target['enemy_type']} Lv{target['enemy_avg_level']:.0f} "
                          f"→target Lv{target['target_avg_level']:.0f} "
                          f"losses={target['losses_here']} status={target['status']}")
            else:
                print(f"#   No revenge targets")

            # Chain stats
            cst = brain.get_chain_stats()
            print(f"#")
            print(f"# CHAINS:")
            for c, s in cst.items():
                cap = brain.ENTITY_CAPACITY.get(c, '?')
                sc = brain.entity_spawn_counts.get(c, 0)
                mc = brain.entity_merge_counts.get(c, 0)
                print(f"#   {c}: {s['actions']}a {s['entities']}e (cap:{cap} spawn:{sc} merge:{mc})")

            # Activations
            act_counts = {}
            act_changes = 0
            for p in brain.perceptrons:
                act_counts[p.active_activation] = act_counts.get(p.active_activation, 0) + 1
                act_changes += p.activation_change_count
            print(f"# ACTIVATIONS: {act_changes} total changes")
            for k, v in sorted(act_counts.items(), key=lambda x: x[1], reverse=True):
                print(f"#   {k}: {v} perceptrons")

            # Memory
            print(f"#")
            print(f"# MEMORY:")
            current_mem, peak_mem = tracemalloc.get_traced_memory()
            print(f"#   RSS current: {current_mem / (1024*1024):.1f}MB  peak: {peak_mem / (1024*1024):.1f}MB")
            obs_stats = brain.get_activation_observation_stats()
            print(f"#   Activation obs: {obs_stats['_total']['observations']} (~{obs_stats['_total']['estimated_mb']:.2f}MB)")
            file_sizes = brain.get_knowledge_file_sizes()
            total_kb = file_sizes.get('_total', 0) / 1024
            print(f"#   Knowledge files: {total_kb:.1f}KB total")

            # Save worker
            sw_stats = save_worker.get_stats()
            print(f"#   SaveWorker: {sw_stats['saves_completed']}saves {sw_stats['saves_dropped']}dropped "
                  f"{sw_stats['total_save_time']:.1f}s q:{sw_stats['queue_size']}")

            # Event recorder
            er_stats = event_recorder.get_stats()
            print(f"#   EventRecorder: {er_stats['total_events']}ev "
                  f"({er_stats['battles']}bat {er_stats['bags']}bag {er_stats['maps']}map "
                  f"{er_stats['levelups']}lvl) q:{er_stats['queue_size']}")

            print(f"#")
            if brain.event_timeline_loaded:
                tl = brain.get_timeline_status()
                print(f"# Timeline: {tl['events']}ev {tl['segments']}seg {tl['prep_points']}prep")
            ac, acf = brain.get_autonomous_hp_estimate(raw_x, raw_y, current_map)
            if acf > 0: print(f"# Auto: cost{ac:.0%} conf{acf:.0%}")
            print(f"# Nav:{'active' if brain.is_nav_active() else 'idle'} Graph:{len(mg)}maps")
            ns = brain.get_nav_targets_status()
            if ns['loaded']: print(f"# Targets:{ns['remaining']}/{ns['total']}")
            print(f"# Blend: t{brain.blend_tier} #{brain.blend_count}")
            print(f"{'#'*70}")

            # Background save
            save_worker.submit_all_knowledge(
                brain,
                filepath=BASE_PATH / "model_checkpoint.json",
                timestep=brain.timestep,
                cache_manager=cache_manager
            )
            print(f"# Save queued (bg)")

            gc.collect()

        # === WAIT + LEARN ===
        for _ in range(10):
            time.sleep(0.005)
            if active_cache.get_version() > last_processed_version: break

        (next_ctx, next_pal, next_til, dead, next_raw_pos,
         next_bd, next_pd, next_gs, next_md, next_bgd,
         next_tf) = active_cache.get_state()
        last_processed_version = active_cache.get_version()

        next_derived = compute_derived_features(next_ctx, context_state)
        next_ls = build_learning_state_overworld(next_derived, next_pal, next_til, next_ctx[3])

        if brain.is_dialogue_skip_state():
            saved_stagnation = brain.state_stagnation_count
            brain.learn(learning_state, next_ls, context_state, next_ctx, dead=dead,
                        raw_position=raw_position, next_raw_position=next_raw_pos)
            brain.state_stagnation_count = saved_stagnation
        else:
            brain.learn(learning_state, next_ls, context_state, next_ctx, dead=dead,
                        raw_position=raw_position, next_raw_position=next_raw_pos)

        prev_context_state = context_state.copy()
        prev_raw_position = raw_position
        brain.timestep += 1

except KeyboardInterrupt:
    print("\n\n🛑 Stopping...")

    # === STOP ALL BACKGROUND THREADS ===
    io_thread.stop()
    io_thread.join(timeout=2)

    event_recorder.stop()
    event_recorder.join(timeout=3)

    save_worker.stop()
    save_worker.join(timeout=5)

    # === FINAL SYNCHRONOUS SAVE PASS ===
    print("  💾 Final synchronous save...")
    cache_manager.save_exploration_memory()
    brain.save_model_checkpoint(BASE_PATH / "model_checkpoint.json")
    brain.save_roster()
    brain.save_move_knowledge()
    brain.save_item_knowledge()
    brain.save_type_clusters()
    brain.save_residual_file()

    # === FINAL STATS ===
    cst = brain.get_chain_stats()
    chain_parts = [f"{c}:{s['actions']}a+{s['entities']}e" for c, s in cst.items()]
    print(f"  Chains: {' | '.join(chain_parts)}")
    print(f"  Pipelines: {brain.get_pipeline_summary()}")
    act_changes = sum(p.activation_change_count for p in brain.perceptrons)
    print(f"  Activation changes: {act_changes}")
    if bootstrapped_from_taught:
        print(f"  Bootstrapped from taught model ({taught_timestep} human steps)")
    print(f"  Bag:{brain.bag_thread_total_actions}act SM:{brain.start_menu_total_actions}act "
          f"Prep:{brain.prep_total_count}att/{brain.prep_success_count}suc")

    # Dialogue final stats
    ds = brain.get_dialogue_status()
    print(f"  Dialogue: {ds['total_skip_actions']}skip {ds['total_choice_actions']}choice {ds['total_frames']}f")

    # Revenge final stats
    rs = brain.get_revenge_status()
    if rs['targets'] > 0:
        print(f"  Revenge: {rs['targets']} targets ({rs['by_status']})")

    # Pipeline final stats
    for pid, pipeline in brain.pipelines.items():
        total_p = sum(pool.get_perceptron_count(brain.perceptrons) for pool in pipeline.pools)
        total_r = sum(len(pool.residual) for pool in pipeline.pools)
        if total_p > 0 or total_r > 0:
            print(f"  Pipeline {pid}: {total_p}p {total_r}r auth={pipeline.get_total_authority():.0%}")

    print(f"  MapStats: {brain.get_map_battle_stats_summary()['maps_with_data']}maps")
    print(f"  W:{battle_outcomes['win']} L:{battle_outcomes['loss']} R:{battle_outcomes['run']} C:{battle_outcomes['catch']}")

    # Type chart
    tc_st = brain.get_type_chart_status()
    print(f"  TypeChart: {tc_st['clustering_runs']}runs {tc_st['move_clusters']}mc {tc_st['species_clusters']}sc"
          f"{' +TrackB' if tc_st['track_b_loaded'] else ''}")

    # Save worker
    sw_stats = save_worker.get_stats()
    print(f"  SaveWorker: {sw_stats['saves_completed']}saves {sw_stats['saves_dropped']}dropped {sw_stats['total_save_time']:.1f}s")

    # Event recorder
    er_stats = event_recorder.get_stats()
    print(f"  EventRecorder: {er_stats['total_events']}ev ({er_stats['battles']}bat {er_stats['maps']}map)")

    # Memory
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    print(f"  Memory: current={current_mem/(1024*1024):.1f}MB peak={peak_mem/(1024*1024):.1f}MB")
    tracemalloc.stop()

    print("✅ Done.")